# AIM:
create evaluation workflow. Taking the manually extracted Ci impacts (validation set) and compare it with the CI impacts (llm_geolocations.ipynb) extrracted by the first LLM 1. 
As a first step the evaluation should be done only for the direct CI impacts - CI type, damage and geolocation

Issue:
* What is needed an approach that recognizes when an direct impact case is not detected by the model

Idea: 
* Split the original texts passed to the model on the exact chunks as again
* Then chunkwise check if the CI impacts from the validation set correspond in number and their textual similarity to the CI impacts infered by the LLM 1 and Entity Linking 

### Direct CI impacts: LLM 1 vs domain-expertise 

In [40]:
### Direct CI impacts: LLM 1 vs domain-expertise 
import os
import sys
from pathlib import Path
import io
import gc
import time
import warnings
import subprocess
import importlib
import glob

from unidecode import unidecode
import langdetect
from fuzzywuzzy import fuzz
import torch
from huggingface_hub import login
import numpy as np
import spacy
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from matplotlib import pyplot as plt


sys.path.append('./')
from src.settings import settings as s
import src.document_cleaning as dc
import src.translation_model as tm
import src.utils as u
import src.datahandler as dh
import src.postprocess as pp

# login to HF
# NOTE raises exception when env.variable does not exist (compared to os.envrion.get and its shortcut os.getenv)
os.getenv("HUGGINGFACE_TOKEN")

#  automatic linebreaks and multi-line cells.
pd.set_option("display.colheader_justify", "left")
pd.set_option('display.max_colwidth', 5000)


step = "step2"


/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/.venv/lib/python3.12/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


Running on TUB cluster
Running on TUB Cluster


In [41]:
#  Suppress future warnings from PyTorch
warnings.filterwarnings("ignore", category=FutureWarning)


#  Define data dir where tags.csv and domain-expertise derived tag lists are found 
# VALID_DATA_FILENAME = s.VALID_DATA_FILENAME
VALID_DATA_FILENAME = "table_ci_impacts_sm_hpc.csv"
PATH_VALID_DATA = s.PATH_VALID_DATA
PATH_EVAL_RESULT = s.PATH_EVAL_RESULT

# s.LLM_DATA_FILENAME = "llm1_geollm_step2_Koks 2022.csv" #f"df_responses_{step}_ner_geollm.csv"
# LLM_DATA_FILEPATH = Path("interim_results" / s.LLM_DATA_FILENAME)
# LLM_DATA_FILEPATH = Path(s.PATH_LLM_DATA / s.LLM_DATA_FILENAME)
SIMILARITY_LLM_FILENAME = s.SIMILARITY_LLM_FILENAME

df_valid_org = pd.read_csv(
    "../" + str(PATH_VALID_DATA / VALID_DATA_FILENAME),
    # usecols=["publication_id", "ci1_type", "ci1_damage", "ci1_location", "sentence_reference"],
)
print(len(df_valid_org))
## pre-process: 
# remove undone entries
df_valid_org = df_valid_org[~df_valid_org.astype(str).apply(lambda x: x.str.contains("xx")).any(axis=1)]
# remove further location info (e.g. that entry is a town, Landkreis, Bavaria etc.)
# df_valid_org["ci1_location"] = df_valid_org["ci1_location"].replace(r"\s*\(.*\)", "", regex=True).str.strip()
df_valid_org = df_valid_org.dropna(subset=["publication_id"], how="all") # drop rows where citation info is missing
print(len(df_valid_org))
print(df_valid_org.publication_id.unique())

print(f"Collecting LLM responses from {step}")
# ## prediction data
# df_pred = pd.read_csv(
#     LLM_DATA_FILEPATH,
#    # usecols=["citation_id", "chunk_id", "infrastructure_type", "damage", "location", "chunk_text"]
# )
df_pred = pd.DataFrame()
from ast import literal_eval
# for i, file in enumerate(glob.glob(os.path.join("../interim_results", f'*{step}*.csv'))):
# for i, file in enumerate(glob.glob(os.path.join("../interim_results_beforetool", f'*{step}*.csv'))):
for i, file in enumerate(glob.glob(os.path.join("../interim_results_debug", f'*chain*step2*.csv'))):
#for i, file in enumerate(glob.glob(os.path.join(s.PATH_DATA, "llm_outputs/responses_single_docs", f'*{step}*.csv'))):
    print(i, file)
    df_pred = pd.concat([df_pred, pd.read_csv(file)], ignore_index=True)

# Explode lists
#NOTE: Make ensure that your column is of list type to be able to use pandas' explode(). Here is a working solution:
# df_pred["location"] = df_pred["location"].apply(literal_eval) #convert to list type
#df_pred = df_pred.explode("location")

# ## explode lists to string
# df_pred_pp["location"] = df_pred_pp["location"].apply(literal_eval) #convert to list type
# df_pred_pp["location"] = df_pred_pp["location"].explode()

## keep only rows which passed verification
print(df_pred.shape)
df_pred = df_pred[df_pred["approved_location_of_the_infrastructure"]==True]
print(df_pred.shape)


301
280
['ABC 2024' 'Artemis 2015' 'Brown 2010' 'Containerlift 2024' 'Deidda 2025'
 'EFE 2024' 'Euronews 2024' 'European Investment Bank 2025' 'Ferlita 2023'
 'Gilbody Dickerson 2024' 'Karakatsani 2023' 'Kaur 2025' 'Nour 2011'
 'Chamra 2006' 'Mitsakis 2014' 'Pescaroli 2017' 'Hladny 2004' 'Fink 2009'
 'Keller 2014' 'Khazai 2013' 'Koks 2022' "Lloyds' List 2024" 'PWC 2015'
 'Rozendaal 2021' 'Skoulding 2023' 'The Guardian 2018' 'The Vibes 2022'
 'Treanor 2015' 'Wildhagen 2013' 'Wilson 2024']
0 ../interim_results_debug/llm_geollm_chain_step2_Lloyd's List 2024.csv
1 ../interim_results_debug/llm_geollm_chain_step2_Kaur 2025.csv
2 ../interim_results_debug/llm_geollm_chain_step2_European Investment Bank 2025.csv
3 ../interim_results_debug/llm_geollm_chain_step2_Karakatsani 2023.csv
4 ../interim_results_debug/llm_geollm_chain_step2_Koks 2022.csv
5 ../interim_results_debug/llm_geollm_chain_step2_Skoulding 2023.csv
6 ../interim_results_debug/llm_geollm_chain_step2_Artemis 2015.csv
7 ../interim_res

In [42]:
df_pred["citation_id"].unique()

array(["Lloyd's List 2024", 'Kaur 2025', 'European Investment Bank 2025',
       'Karakatsani 2023', 'Koks 2022', 'Skoulding 2023', 'EFE 2024',
       'Gilbody Dickerson 2024', 'Fink 2004', 'ABC 2024', 'Ferlita 2023',
       'Kettle 2020', 'AFP 2022', 'Euronews 2024', 'Khazai 2013',
       'Rozendaal 2021', 'Containerlift 2024', 'Wilson 2024',
       'Wildhagen 2013'], dtype=object)

### cleanup entries in prediction set

In [43]:
## remove entry in df_pred when no "infrastructure_type_org" exists, then it is likely a hallucinated case
# df_pred = df_pred.dropna(subset=["infrastructure_type_org"], how="all")


#### Remove records which have a faulty record (FAC or other non-CI entity) in CI_entity column ->
-> due that respective LLM response is often faulty (hallucinated)


In [44]:

## remove entry in df_pred when a "non-CI" record occurs in the "ci_entity" column 
# NOTE this should solves the issue from fix-commit "9fb76a6" - where some ci_entires contained LOCs or non-CI (e.g. EFE 2024: "the Monastery of La Cartuja", "the Advanced Command Post")
 
print(len(df_pred))

ci_patterns = pd.read_json("../ner_patterns.jsonl/patterns", lines=True)

# treat removal only on records which have something in "ci_entity" column
tt = df_pred
tt = tt[tt["ci_entity"].notna()]
# get where actual Ci entities are present in "ci_entity" column
tt["ci_entity_grouped"] = None
tt = pp.group_ci_types(tt, col_type="ci_entity", col_grouped="ci_entity_grouped", ci_patterns=ci_patterns)
## get cases of non-CI 
non_ci_records = tt[tt["ci_entity_grouped"].isna()]
print("NOn ci_records", non_ci_records)
# write back - keep only records which are in "ci_entity" either np.nan or a CI type
print(f"Remove {len(non_ci_records)} from {len(df_pred)} records which are not CI")
df_pred = df_pred.drop(index=non_ci_records.index)

print(len(df_pred))

1067


KeyError: 'ci_entity'

#### AS FUNC: Evaluate on same documents that were passed to LLM



In [45]:
print("Use only citations which are in both")


df_valid = df_valid_org[df_valid_org["publication_id"].isin(df_pred.citation_id.unique())]
# df_valid_org[df_valid_org["publication_id"].isin(citation_list)]
print("Valid citations:")
print(set(sorted(df_valid.publication_id)))

df_pred = df_pred[df_pred["citation_id"].isin(df_valid.publication_id.unique())]
print("Predicted citations:")
print(set(sorted(df_pred.citation_id)))

Use only citations which are in both
Valid citations:
{'Ferlita 2023', 'Kaur 2025', 'Wildhagen 2013', 'European Investment Bank 2025', 'EFE 2024', 'Wilson 2024', 'Skoulding 2023', 'Rozendaal 2021', 'Khazai 2013', 'Koks 2022', 'ABC 2024', 'Karakatsani 2023', 'Containerlift 2024', 'Gilbody Dickerson 2024', 'Euronews 2024'}
Predicted citations:
{'Ferlita 2023', 'Kaur 2025', 'Wildhagen 2013', 'European Investment Bank 2025', 'EFE 2024', 'Wilson 2024', 'Skoulding 2023', 'Rozendaal 2021', 'Khazai 2013', 'Koks 2022', 'ABC 2024', 'Karakatsani 2023', 'Containerlift 2024', 'Gilbody Dickerson 2024', 'Euronews 2024'}


In [46]:
print(df_valid.info())
print(df_pred.info())

<class 'pandas.core.frame.DataFrame'>
Index: 210 entries, 0 to 288
Data columns (total 28 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   event_id                    210 non-null    object 
 1   event_time                  210 non-null    object 
 2   publication_id              210 non-null    object 
 3   sentence_reference          210 non-null    object 
 4   Unnamed: 4                  27 non-null     object 
 5   ci1_type                    198 non-null    object 
 6   ci_damage_numeric           70 non-null     object 
 7   ci23_damage_numeric_merged  37 non-null     object 
 8   Unnamed: 8                  42 non-null     object 
 9   ci1_damage                  188 non-null    object 
 10  ci23_dam_test               30 non-null     object 
 11  ci1_location                190 non-null    object 
 12  ci1_location_spec           82 non-null     object 
 13  ci23_type                   59 non-null 

In [47]:
print(df_valid.info())
print(df_pred.info())

<class 'pandas.core.frame.DataFrame'>
Index: 210 entries, 0 to 288
Data columns (total 28 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   event_id                    210 non-null    object 
 1   event_time                  210 non-null    object 
 2   publication_id              210 non-null    object 
 3   sentence_reference          210 non-null    object 
 4   Unnamed: 4                  27 non-null     object 
 5   ci1_type                    198 non-null    object 
 6   ci_damage_numeric           70 non-null     object 
 7   ci23_damage_numeric_merged  37 non-null     object 
 8   Unnamed: 8                  42 non-null     object 
 9   ci1_damage                  188 non-null    object 
 10  ci23_dam_test               30 non-null     object 
 11  ci1_location                190 non-null    object 
 12  ci1_location_spec           82 non-null     object 
 13  ci23_type                   59 non-null 

## MV to postprocess.fuc() drop dublicated predictions + upd (encod-utf-8)saving_llm_reuslts in loop (rm fix saving) + pp of NAN strings in LLm response


#### MV to postprocess: cases with NANs 

In [48]:

def convert_nan(series: pd.Series) -> pd.Series:
    """ convert representations of "NAN" to np.nan """
    # TODO use regex instead of ["NAN", "NaN", "nan"] by setting all possible representations of nan (e.g. "Nan") to lowercase 
    series = series.replace(["NAN", "NaN", "nan"], np.nan)

    return series


print(df_pred.info())
df_pred["infrastructure_type"] = convert_nan(df_pred["infrastructure_type"])
df_pred["damage"] = convert_nan(df_pred["damage"])
df_pred["location"] = convert_nan(df_pred["location"])

print(df_pred.info(), len(df_pred))



<class 'pandas.core.frame.DataFrame'>
Index: 768 entries, 13 to 1693
Data columns (total 13 columns):
 #   Column                                         Non-Null Count  Dtype  
---  ------                                         --------------  -----  
 0   citation_id                                    768 non-null    object 
 1   chunk_id                                       768 non-null    object 
 2   infrastructure_type                            768 non-null    object 
 3   infrastructure_group                           768 non-null    object 
 4   location                                       768 non-null    object 
 5   location_type                                  768 non-null    object 
 6   approved_location_of_the_infrastructure        768 non-null    object 
 7   damage                                         768 non-null    object 
 8   damage_value                                   692 non-null    object 
 9   coord_potential_locations                      0 non-null

/tmp/ipykernel_294437/2820037641.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_pred["infrastructure_type"] = convert_nan(df_pred["infrastructure_type"])
/tmp/ipykernel_294437/2820037641.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_pred["damage"] = convert_nan(df_pred["damage"])
/tmp/ipykernel_294437/2820037641.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the 

#### clean-up multiple locs/Cis in one entry (MV FUNC. to pp )

In [49]:
def split_text_into_multiple_rows(df: pd.DataFrame, column: str, split_at = " and ") -> pd.DataFrame:
    """ split text at splitting_point into multiple rows """
    # split CIs and LOCs with "and" into multiple rows
    df[column] = df[column].str.split(rf"{split_at}")   
    # NOTE: Removes info from CI - drops info if CI is singular o plural (e.g, road and railway infrastrcutre --> "road", "railway infrastructure")
    df = df.explode(column=column)
    df = df.drop_duplicates().reset_index(drop=True)
    return df

# disentangle rows which contain multiple locations or CIs
df_pred = split_text_into_multiple_rows(df_pred, "location",  split_at = " or ")
df_valid = split_text_into_multiple_rows(df_valid, "ci1_location",  split_at = " or ")
df_pred = split_text_into_multiple_rows(df_pred, "location",  split_at = ", ")  # Germany, Neterlands, Belgium
df_valid = split_text_into_multiple_rows(df_valid, "ci1_location",  split_at = ", ")
df_pred = split_text_into_multiple_rows(df_pred, "location",  split_at = " to ") 
df_valid = split_text_into_multiple_rows(df_valid, "ci1_location",  split_at = " to ")

# multiple location splitting - "and", "to"
df_pred = split_text_into_multiple_rows(df_pred, "location",  split_at = " and ")
df_valid = split_text_into_multiple_rows(df_valid, "ci1_location",  split_at = " and ")
df_pred = split_text_into_multiple_rows(df_pred, "infrastructure_type",  split_at = " and ").reset_index(drop=True)
df_valid = split_text_into_multiple_rows(df_valid, "ci1_type",  split_at = " and ")


# rm trailing and leading whitespaces
df_pred = df_pred.replace(r"^ +| +$", r"", regex=True)
df_valid = df_valid.replace(r"^ +| +$", r"", regex=True)



/tmp/ipykernel_294437/476747079.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[column] = df[column].str.split(rf"{split_at}")
/tmp/ipykernel_294437/476747079.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[column] = df[column].str.split(rf"{split_at}")


#### group CI into subgroups (MAKE AS FUNC)

make CI_groups after clean-up LOC and CI-columns

In [50]:
ci_patterns = pd.read_json("../ner_patterns.jsonl/patterns", lines=True)


## group Ci types into subgroups,
if not "infrastructure_group" in df_pred.columns or df_pred["infrastructure_group"].isna().any():
    #print("Remove all potential brackets for plural forms in infrastructure types [(s)]")
    #df_pred["infrastructure_type"] = df_pred["infrastructure_type"].str.replace(r"\(s\)", "", regex=True).str.strip()
    df_pred = pp.group_ci_types(df_pred, "infrastructure_type", "infrastructure_group", ci_patterns)

if not "ci1_group" in df_valid.columns or df_valid["ci1_group"].isna().any():
    df_valid["ci1_group"] = None
    df_valid = pp.group_ci_types(df_valid, "ci1_type", "ci1_group", ci_patterns)


print(df_pred.infrastructure_group.isna().sum())  # mostly cases which are not CI (theater, stadion..)
print(df_valid.ci1_group.isna().sum())

## keep only records which are actually about CI (e.g., not theatre, stadion ..)
df_valid.dropna(subset=["ci1_group"], inplace=True)
    # df_pred.dropna(subset=["infrastructure_group"], inplace=True)

print(df_pred.infrastructure_group.value_counts())
# df_pred.infrastructure_group.unique()
print(df_valid.ci1_group.value_counts())
# df_pred.infrastructure_group.unique()


0

/home/a-buch/Documents/TUB_DWN/_PROJECTS/CI-impacts-information-retrieval/notebooks/../src/postprocess.py:39: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = df[col_type].str.contains(pattern, regex=True, na=False)
/home/a-buch/Documents/TUB_DWN/_PROJECTS/CI-impacts-information-retrieval/notebooks/../src/postprocess.py:39: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = df[col_type].str.contains(pattern, regex=True, na=False)
/home/a-buch/Documents/TUB_DWN/_PROJECTS/CI-impacts-information-retrieval/notebooks/../src/postprocess.py:39: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = df[col_type].str.contains(pattern, regex=True, na=False)
/home/a-buch/Documents/TUB_DWN/_PROJECTS/CI-impacts-information-retrieval


30
infrastructure_group
roads                           30
rail                            21
bridges                         18
airports                        16
electricity_supply              15
motorways                       14
transport_others                13
water_supply                    10
healthcare_hospitals_clinics    10
waste_others                     9
rail_service                     9
it_telecommunication             8
gas_distribution                 6
healthcare_others                6
ports                            5
education_school                 4
waterprotection                  3
wastewater                       3
electricity_others               3
electricity_distribution         2
gas_supply                       2
transport_supply                 2
education_kita                   1
nursing                          1
education_others                 1
water_others                     1
tunnels                          1
Name: count, dtype: int64
ci1_

In [51]:

print("Workaround for changing ci_group for drinking water ")

df_valid["ci1_group"] = df_valid["ci1_group"].replace(["drinking_water"], "water_supply")
df_pred["infrastructure_group"] = df_pred["infrastructure_group"].replace(["drinking_water"], "water_supply")
df_pred.infrastructure_group.value_counts()

Workaround for changing ci_group for drinking water 


infrastructure_group
roads                           30
rail                            21
bridges                         18
airports                        16
electricity_supply              15
motorways                       14
transport_others                13
water_supply                    10
healthcare_hospitals_clinics    10
waste_others                     9
rail_service                     9
it_telecommunication             8
gas_distribution                 6
healthcare_others                6
ports                            5
education_school                 4
waterprotection                  3
wastewater                       3
electricity_others               3
electricity_distribution         2
gas_supply                       2
transport_supply                 2
education_kita                   1
nursing                          1
education_others                 1
water_others                     1
tunnels                          1
Name: count, dtype: int64

In [52]:
print("Cases of CI which could not be grouped")

print(df_pred[df_pred.infrastructure_group.isna()].shape[0])
print(df_valid[df_valid.ci1_group.isna()].shape[0])

Cases of CI which could not be grouped
0
0


#### drop cases in valid and pred where Ci or LOC is empty


In [53]:
print("Removing all records which have erroneous CI or missing LOC entry")

df_pred = df_pred[~df_pred.infrastructure_group.isna()]
df_valid = df_valid[~df_valid.ci1_group.isna()]

df_pred = df_pred[~df_pred.location.isna()]
df_valid = df_valid[~df_valid.ci1_location.isna()]


Removing all records which have erroneous CI or missing LOC entry


In [54]:
## set "affected" to NAN in damage columns (pred, valid)
## TODO check if affected is in general decreasing recall or precision score for "dam" class if yes then set to NAN otherwise keep unchanged


In [55]:
print(len(df_pred))
unique_ci_geo_pairs = df_pred.drop_duplicates()
print("number of duplicates to remove:", len(df_pred) - len(unique_ci_geo_pairs))

df_pred = df_pred.drop_duplicates( )# .reset_index(drop=True, inplace=True)
print(len(df_pred))


214
number of duplicates to remove: 0
214


### remove records which are not about Europe 


In [56]:
df_pred = df_pred[~df_pred["location"].isin(["New York", "New Jersey", "New Orleans", "Fukushima"])]

# TODO replace this with a search based on Europe after getting coords

### remove all records which are on country-level or "Europe"
<as during LLm extraction WF, the geollama model only shows for guiding LLM 1 (step 2) only sub-country locations>

In [57]:
import geonamescache

def get_countries():
    geolocs_cache = geonamescache.GeonamesCache()
    countries = geolocs_cache.get_countries()
    ci_geo_countries = [*u.gen_dict_extract(countries, 'name')] 
    # add further country names with abbrev. or "the" , incl. als regions which have the same name as their country (eg. Luxembourg- Provinz in Belgium)
    ci_geo_countries = ci_geo_countries + ["the Netherlands", "Netherlands", "UK", "US", "U.S.", "USA"]
    return ci_geo_countries


# remove all records which are on country-level
# NOTE do it as during STEP2 in LLM-WF only sub-country potential_locaitons where shown
ci_geo_countries = get_countries()

print(f"Removing {len(df_pred[df_pred['location'].isin(ci_geo_countries)])} records which are on country-level in prediction set")
print(f"Removing {len(df_valid[df_valid['ci1_location'].isin(ci_geo_countries)])} records which are on country-level in validation set")
print("NOTE: Done as during STEP2 in LLM-WF only sub-country potential_locations were shown")

df_pred = df_pred[~df_pred["location"].isin(ci_geo_countries)]
df_valid = df_valid[~df_valid["ci1_location"].isin(ci_geo_countries)]

## remove all records which mention Europe
df_pred = df_pred[~df_pred["location"].str.contains(r"Europe.*|European.*", case=False, na=False)]
df_valid = df_valid[~df_valid["ci1_location"].str.contains(r"Europe.*|European.*", case=False, na=False)]



# # df_pred[col].apply(lambda x: unidecode(x) if isinstance(x, str) else x)

Removing 17 records which are on country-level in prediction set
Removing 31 records which are on country-level in validation set
NOTE: Done as during STEP2 in LLM-WF only sub-country potential_locations were shown


#### FIXME country-wise removal can be loss of info for certain CI sectors or monetary impacts

In [58]:
# df_pred[df_pred["location"].isin(ci_geo_countries)].drop(["chunk_text","coord_potential_locations"], axis=1)
# df_valid[df_valid["ci1_location"].isin(ci_geo_countries)].drop(["sentence_reference"], axis=1)

## FIXME country-wise removal can be loss of info for certain CI sectors or monetary impacts: 
#        pred: it/telecommunication, waste_*, wastewater; Valid: also gas_supply, electricity infrastructure	

### drop dublicated cases which differ only in Tier 2 or Tier 3 impacts
> e.g. valid ABC 2024: - identical c1_type, ci1_damage, ci1_loc (but diff. ci2_damages -which are not used in this eval) 


In [59]:
print(f"Dropping {df_valid.duplicated().sum()} duplicates in valid data")
df_valid = df_valid.drop_duplicates()

print(f"Dropping {df_pred[['citation_id', 'chunk_id', 'infrastructure_type','damage', 'location', 'chunk_text']].duplicated().sum()} duplicates in pred data")
df_pred = df_pred[df_pred[['citation_id', 'chunk_id', 'infrastructure_type', 'damage', "damage_value", 'location', 'chunk_text']].duplicated() == False]


Dropping 0 duplicates in valid data
Dropping 2 duplicates in pred data


#### AS FUNC: clean-up locations

In [60]:

# print(df_pred.location.unique())
# print(df_valid.ci1_location.unique())

## TODO clean-up locs --> MV this postprocessing to LLm extraction before passing to LLm Step2
# "( "  eg "Sinzig (in North Rhine-Westphalia)""
df_pred["location"] = df_pred["location"].str.split(r"\(", regex=True).str[0].str.strip()
df_valid["ci1_location"] = df_valid["ci1_location"].str.split(r"\(", regex=True).str[0].str.strip()

# rm text after comma, e.g. "Ahr valley, Germany" --> "Ahr valley"
df_pred["location"] = df_pred["location"].str.split(", ").str[0].str.strip()  
df_valid["ci1_location"] = df_valid["ci1_location"].str.split(", ").str[0].str.strip()  
# remove all remaining brackets and commas
df_pred["location"] = df_pred["location"].replace(r"[\(\),]", "", regex=True)
df_valid["ci1_location"] = df_valid["ci1_location"].replace(r"[\(\),]", "", regex=True)

## handling loc with "railway tracks between "
df_pred["location"] = df_pred["location"].str.split("between").str[-1].str.strip()
df_valid["ci1_location"] = df_valid["ci1_location"].str.split("between").str[-1].str.strip()
## handling loc with "railway tracks in"  , set this after cleaning up "( " and ", " as it otherwise would take the later location
df_pred["location"] = df_pred["location"].str.split("in ").str[-1].str.strip()
df_valid["ci1_location"] = df_valid["ci1_location"].str.split("in ").str[-1].str.strip()

#  try to remove "the"
#  already before passing to GeoLLM in extraction-WF
df_pred["location"] = df_pred["location"].replace("the ", " ", regex=True).str.strip()   # regex= True needed to identify string part (e.g "the ") 
df_valid["ci1_location"] = df_valid["ci1_location"].replace("the ", " ", regex=True).str.strip()
## handling loc with "passing "
df_pred["location"] = df_pred["location"].replace("passing ", " ", regex=True).str.strip()   
df_valid["ci1_location"] = df_valid["ci1_location"].replace("passing ", " ", regex=True).str.strip()


print("\nAfter cleanup: drop records which are not about CI anymore")
ci_patterns = pd.read_json("../ner_patterns.jsonl/patterns", lines=True)

df_pred = pp.group_ci_types(df_pred, "infrastructure_type", "infrastructure_group", ci_patterns)
## keep only records which are actually about CI (e.g., not remainings from cleanup)
df_pred.dropna(subset=["infrastructure_group"], inplace=True)


print(df_pred.location.unique())
print(df_valid.ci1_location.unique())


After cleanup: drop records which are not about CI anymore


['Humboldt County' 'Valencia' 'National highway' 'Ahr valley'
 'A1 motorway' 'Spa-Pepinster' 'Spa' 'Pepinster' 'A76' 'Maastricht'
 'Liége' 'North Rhine-Westphalia' 'Rhineland-Palatinate' 'Chaudfontaine'
 'Altenahr' 'Mayschoss' 'Sinzig' 'Bad Münstereifel' 'city centre'
 'near Heimersheim' 'Heimersheim' 'Altenburg' 'Eschweiler' 'Trier'
 'Bad Neuenahr-Ahrweiler' 'Ahr' 'Erft' 'Ahr/Erft rivers' 'Palermo'
 'Catania' 'Sicily' 'Milan' 'Castellón de la Plana' 'Valencian Community'
 'Andalusia' 'province of Valencia' 'near Malaga' 'Malaga airport'
 'Malaga Airport' 'Guerrero Strachan Avenue' 'Velázquez Avenue' 'Malaga'
 'Cártama' 'Catania airport' 'highway 113' 'Polizzi Generosa'
 'Danube-Elbe catchment area' '89 districts' 'German highways'
 'federal roads' 'Deggendorf' 'along  Elbe' 'Stendal district'
 'district of Stendal' 'German federal roads' '89 German districts'
 'Traunstein' 'Rosenheim' 'Erzgebirgskreis' 'Erbacher underpass'
 'Fischbeck' 'Walloon rail network' 'Wallonia' 'Barcelona' 'ea

/home/a-buch/Documents/TUB_DWN/_PROJECTS/CI-impacts-information-retrieval/notebooks/../src/postprocess.py:39: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = df[col_type].str.contains(pattern, regex=True, na=False)
/home/a-buch/Documents/TUB_DWN/_PROJECTS/CI-impacts-information-retrieval/notebooks/../src/postprocess.py:39: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = df[col_type].str.contains(pattern, regex=True, na=False)
/home/a-buch/Documents/TUB_DWN/_PROJECTS/CI-impacts-information-retrieval/notebooks/../src/postprocess.py:39: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = df[col_type].str.contains(pattern, regex=True, na=False)
/home/a-buch/Documents/TUB_DWN/_PROJECTS/CI-impacts-information-retrieval

### handling of certain LOC in potential_coords and loc columns: Málaga


In [61]:
df_pred["location"] = df_pred["location"].str.replace(r"Málaga", "Malaga", regex=True)
df_pred["coord_potential_locations"] = df_pred["coord_potential_locations"].str.replace(r"Málaga", "Malaga", regex=True)#.

df_valid["ci1_location"] = df_valid["ci1_location"].str.replace(r"Málaga", "Malaga", regex=True)


AttributeError: Can only use .str accessor with string values!

### locations_2_coordinates() matching, MAKE AS FUNC

In [ ]:
def get_bbox(points):
    x_coordinates, y_coordinates = zip(*points)
    return [(min(x_coordinates), min(y_coordinates)), (max(x_coordinates), max(y_coordinates))]

# TODO mv to utils.py or geolocalization.py

In [ ]:
# extract dict from geollm-string
df_pred["coord_potential_locations"] = df_pred["coord_potential_locations"].apply(lambda x: eval(str(x))) 


## init geolocalized location info (name of loc in OSM and its coordinates or bbox of potential location)
df_pred["location_osm"] = None
df_pred["coords"] = None
df_pred["coords_coarse"] = None


# reindex to avoid issues in iloc
df_pred.reset_index(drop=True, inplace=True)

## get most likely coordinates for locations
counter_errors = 0
for entry in range(len(df_pred)):

    loc = df_pred["location"].iloc[entry]
    loc_potential_coords = df_pred["coord_potential_locations"].iloc[entry]
    try:
        # go to next entity when it is NAN
        if loc == np.nan or loc == "nan" or loc == "NAN" or loc == "NaN" or loc == "":
            continue
        # write coordinates and OSM name of location back
        df_pred.at[entry,  "coords"] = loc_potential_coords[loc][0:2]
        df_pred.at[entry,  "location_osm"] = loc

    except Exception as e:
        try:
            for loc_potential in loc_potential_coords.keys(): 

                ## WORKAROUND handling "Ahr river valley" <-> "Ahrtal"
                if loc_potential == "Ahrtal":  # geollama response
                    df_pred.at[entry, "location"] = df_pred.at[entry,"location"].replace("Ahr River valley", "Ahrtal")
                    df_pred.at[entry, "location"] = df_pred.at[entry,"location"].replace("Ahr valley", "Ahrtal")
                    loc = df_pred["location"].iloc[entry]

                smlrty = fuzz.partial_ratio(loc, loc_potential)
                if smlrty > 90: 
                    print("\nUsing partial ratio for matching:", loc, "<->", loc_potential, f"{smlrty}")
                    # NOTE: handles "Malaga airport", "the Ahr valley", "Erft region"
                    # FIXME needs imrovement in the future, to get more concrete spat. info (if it is region, a river etc.)
                    
                    # write coordinates and OSM name of location back
                    df_pred.at[entry,  "coords"] = loc_potential_coords[loc_potential][0:2]
                    df_pred.at[entry,  "location_osm"] = loc_potential
                else: pass
       
        except Exception as e:
            print(f"\nEntry {entry}: {df_pred['location'].iloc[entry]}")
            print(f"No geolocalization possible for row {entry}: {e} \n{df_pred['coord_potential_locations'].iloc[entry]}")
            # e.g. "flood region", "A76 in both directions"
            print("Creating BBox of potential location based on locations mentioned in respective chunk text")
            coords_list = [[float(v[0]), float(v[1])] for v in loc_potential_coords.values()]
            # write approximated coordinates back
            df_pred.at[entry,  "coords_coarse"] =  get_bbox(coords_list)
            df_pred.at[entry,  "location_osm"] = None
            
            counter_errors = counter_errors + 1


print(f"{counter_errors} Cases where only coarse loc could be extracted ( based on loc in entire chunk):")
## drop cases where no geolocalization could be done 


# # TODO when loc= "A76 in both directions" --> make new column with "eigenname" new column "potentially_location_in" with list of geollm returns and bbox based on these geollm_locs
# # TODO measure location new based on centroid of loc_red or centroid of "potentially_in"


Using partial ratio for matching: Vltava River watershed <-> Vltava River 100

Using partial ratio for matching: Orlík water reservoir <-> Orlík 100

Using partial ratio for matching: Lipno I reservoir <-> Lipno I 100

Using partial ratio for matching: Orlík reservoir <-> Orlík 100

Using partial ratio for matching: Orlík dam <-> Orlík 100

Using partial ratio for matching: Erft region <-> Erft 100

Using partial ratio for matching: Erft region <-> Erft 100

Using partial ratio for matching: Paiporta ravine <-> Paiporta 100

Using partial ratio for matching: Malaga airport <-> Malaga 100
0 Cases where only coarse loc could be extracted ( based on loc in entire chunk):


In [ ]:
len(df_pred)

193

In [ ]:
# df_pred.loc[df_pred["citation_id"]== "Krausmann 2014"] # Krausmann -> large hallucinations when chunk-text is title or contact info (i.e when not about CI /impacts)

## MV to postprocess.fuc() drop dublicated predictions + upd (encod-utf-8)saving_llm_reuslts in loop (rm fix saving) + pp of NAN strings in LLm response


#### MAKE AS FUNC: group CI into subgroups

In [11]:
ci_patterns = pd.read_json("./ner_patterns.jsonl/patterns", lines=True)


## group Ci types into subgroups,
if not "infrastructure_group" in df_pred.columns or df_pred["infrastructure_group"].isna().any():
    #print("Remove all potential brackets for plural forms in infrastructure types [(s)]")
    #df_pred["infrastructure_type"] = df_pred["infrastructure_type"].str.replace(r"\(s\)", "", regex=True).str.strip()
    df_pred = pp.group_ci_types(df_pred, "infrastructure_type", "infrastructure_group", ci_patterns)
    ## keep only records which are actually about CI (e.g., not theatre, stadion ..)
    df_pred.dropna(subset=["infrastructure_group"], inplace=True)

if not "ci1_group" in df_valid.columns or df_valid["ci1_group"].isna().any():
    df_valid["ci1_group"] = None
    df_valid = pp.group_ci_types(df_valid, "ci1_type", "ci1_group", ci_patterns)
    ## keep only records which are actually about CI (e.g., not theatre, stadion ..)
    df_valid.dropna(subset=["ci1_group"], inplace=True)


print(df_pred.infrastructure_group.isna().sum())  # mostly cases which are not CI (theater, stadion..)
print(df_pred.infrastructure_group.value_counts())
# df_pred.infrastructure_group.unique()


print(df_valid.ci1_group.isna().sum())
print(df_valid.ci1_group.value_counts())
# df_pred.infrastructure_group.unique()


/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/src/postprocess.py:50: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return ci_entity.str.contains(regex_pattern, regex=True, na=False)
/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/src/postprocess.py:50: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return ci_entity.str.contains(regex_pattern, regex=True, na=False)
/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/src/postprocess.py:50: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return ci_entity.str.contains(regex_pattern, regex=True, na=False)
/beegfs/home/users/a/a-buch/_PROJECTS/CI-impacts-information-retrieval/src/postprocess.py:50: UserWarning: This 

0
infrastructure_group
road_others                     72
waterprotection                 28
bridges                         28
rail                            25
water_supply                    12
transport_others                 9
motorways                        9
healthcare_hospitals_clinics     9
airports                         8
wastewater                       7
it_telecommunication             6
waste_others                     5
electricity_supply               5
electricity_others               5
transport_supply                 4
education_school                 4
electricity_distribution         4
water_others                     3
ports                            3
gas_supply                       3
metro                            2
gas_distribution                 2
healthcare_others                2
power_plants                     2
education_kita                   1
Name: count, dtype: int64
0
ci1_group
road_others                      78
rail                        

In [12]:

print("Workaround for changing ci_group for drinking water ")

df_valid["ci1_group"] = df_valid["ci1_group"].replace(["drinking_water"], "water_supply")
df_pred["infrastructure_group"] = df_pred["infrastructure_group"].replace(["drinking_water"], "water_supply")
df_pred.infrastructure_group.value_counts()

Workaround for changing ci_group for drinking water 


infrastructure_group
road_others                     72
waterprotection                 28
bridges                         28
rail                            25
water_supply                    12
transport_others                 9
motorways                        9
healthcare_hospitals_clinics     9
airports                         8
wastewater                       7
it_telecommunication             6
waste_others                     5
electricity_supply               5
electricity_others               5
transport_supply                 4
education_school                 4
electricity_distribution         4
water_others                     3
ports                            3
gas_supply                       3
metro                            2
gas_distribution                 2
healthcare_others                2
power_plants                     2
education_kita                   1
Name: count, dtype: int64

In [13]:
print("Cases of CI which could not be grouped")

print(df_pred[df_pred.infrastructure_group.isna()].shape[0])
print(df_valid[df_valid.ci1_group.isna()].shape[0])

Cases of CI which could not be grouped
0
0


#### MV to postprocess: cases with NANs 

In [14]:

def convert_nan(series: pd.Series) -> pd.Series:
    """ convert representations of "NAN" to np.nan """
    # TODO use regex instead of ["NAN", "NaN", "nan"] by setting all possible representations of nan (e.g. "Nan") to lowercase 
    series = series.replace(["NAN", "NaN", "nan"], np.nan)

    return series


print(df_pred.info())
df_pred["infrastructure_type"] = convert_nan(df_pred["infrastructure_type"])
df_pred["damage"] = convert_nan(df_pred["damage"])
df_pred["location"] = convert_nan(df_pred["location"])

print(df_pred.info(), len(df_pred))



<class 'pandas.DataFrame'>
Index: 258 entries, 4 to 361
Data columns (total 18 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   citation_id                258 non-null    object
 1   chunk_id                   258 non-null    object
 2   infrastructure_type        258 non-null    str   
 3   infrastructure_group       258 non-null    str   
 4   damage                     251 non-null    object
 5   damage_value               80 non-null     object
 6   location                   231 non-null    object
 7   ci_entity                  80 non-null     object
 8   geo_entity                 80 non-null     object
 9   coord_potential_locations  258 non-null    object
 10  chunk_text                 258 non-null    object
 11  infrastructure_type_org    258 non-null    str   
 12  damage_org                 253 non-null    str   
 13  damage_value_org           154 non-null    object
 14  locations_org             

#### drop cases in valid and pred where Ci or LOC is empty


In [15]:
print("Removing all records which have erroneous CI or missing LOC entry")

df_pred = df_pred[~df_pred.infrastructure_group.isna()]
df_valid = df_valid[~df_valid.ci1_group.isna()]

df_pred = df_pred[~df_pred.location.isna()]
df_valid = df_valid[~df_valid.ci1_location.isna()]


Removing all records which have erroneous CI or missing LOC entry


In [16]:
## set "affected" to NAN in damage columns (pred, valid)
## TODO check if affected is in general decreasing recall or precision score for "dam" class if yes then set to NAN otherwise keep unchanged


In [17]:
print(len(df_pred))
unique_ci_geo_pairs = df_pred.drop_duplicates()
print("number of duplicates to remove:", len(df_pred) - len(unique_ci_geo_pairs))

df_pred = df_pred.drop_duplicates( )# .reset_index(drop=True, inplace=True)
print(len(df_pred))


223
number of duplicates to remove: 1
222


In [18]:
df_valid[df_valid.publication_id=="Nour 2011"].drop(["sentence_reference", "ci23_damage_numeric_merged", "ci23_dam_test"], axis=1)


,event_id,event_time,publication_id,Unnamed: 4,ci1_type,ci_damage_numeric,Unnamed: 8,ci1_damage,ci1_location,ci1_location_spec,...,societal_impact_who,societal_impact_damage,s_impact_location,economic_impact,e_impact_location,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,ci1_group
94,Romania - multiple floods,2005 - 2010,Nour 2011,NaN,schools,NaN,NaN,disturbed,Maramureş County,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,education_school
96,Romania - multiple floods,2005 - 2010,Nour 2011,NaN,bridges,NaN,NaN,damaged,Maramureş County,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,bridges
97,Romania - multiple floods,2005 - 2010,Nour 2011,NaN,village roads,NaN,NaN,damaged,Maramureş County,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,road_others
98,Romania - multiple floods,2005 2010,Nour 2011,NaN,national and county roads,NaN,NaN,disturbed,Maramureş County,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,road_others
99,Romania - multiple floods,2005 - 2010,Nour 2011,NaN,water supply,water supply networks,NaN,damaged,Maramureş County,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,water_supply
101,Romania - multiple floods,2005 - 2010,Nour 2011,NaN,electric networks,NaN,NaN,damaged,Maramureş County,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,electricity_distribution
104,Romania - multiple floods,2005 - 2010,Nour 2011,NaN,transport network,NaN,NaN,NaN,Maramureş County,NaN,...,NaN,NaN,NaN,21 million dollars,21 million dollars (around 65%) of the totla 32 million dollars damages,NaN,NaN,NaN,NaN,transport_others
105,Romania - multiple floods,2005 - 2010,Nour 2011,NaN,national roads,6.7 km,NaN,damaged,Maramureş County,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,road_others
106,Romania - multiple floods,2005 - 2010,Nour 2011,NaN,county roads,26.03 km,NaN,damaged,Maramureş County,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,road_others
107,Romania - multiple floods,2005 - 2010,Nour 2011,NaN,village roads,442 km,NaN,damaged,Maramureş County,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,road_others


In [19]:
df_pred[[
    "citation_id","chunk_id", "infrastructure_type", "infrastructure_group", "damage", "damage_value", "location", "ci_entity",	"geo_entity", "infrastructure_type_org",	"damage_org","damage_value_org","locations_org","coords","coords_coarse"
    ]][4:50]  
# check for duplicates


,citation_id,chunk_id,infrastructure_type,infrastructure_group,damage,damage_value,location,ci_entity,geo_entity,infrastructure_type_org,damage_org,damage_value_org,locations_org,coords,coords_coarse
4,Chamra 2006,5.0,underground railway,rail,NaN,NaN,Prague,an underground railway,Prague,underground railway,NaN,NaN,Prague,"(50.0874654, 14.4212535)",None
5,Chamra 2006,6.0,Metro station,metro,NaN,NaN,Kobylisy,NaN,NaN,NaN,NaN,NaN,NaN,"(50.0874654, 14.4212535)",None
6,Chamra 2006,6.0,Metro station,metro,NaN,NaN,Prague,NaN,NaN,Metro station,NaN,NaN,NaN,"(50.1294426, 14.4636945)",None
7,Chamra 2006,12.0,water infrastructure,water_others,NaN,NaN,Prague,NaN,NaN,water infrastructure,NaN,NaN,NaN,"(50.0874654, 14.4212535)",None
8,Chamra 2006,15.0,reservoirs,renewable_hydro,exhaustion of retention capacity,NaN,afflicted territory,the reservoirs,the Vltava River,reservoirs,exhaustion of retention capacity,NaN,the afflicted territory,"(0.0, 0.0)",None
9,Chamra 2006,15.0,reservoirs,renewable_hydro,exhaustion of retention capacity,NaN,Vltava River watershed,NaN,NaN,reservoirs,exhaustion of retention capacity,NaN,the Vltava River watershed,"(49.3347601, 14.2973161)",None
10,Chamra 2006,16.0,water reservoir,renewable_hydro,uncontrolled water runoff,NaN,Orlík,NaN,NaN,water reservoir,uncontrolled water runoff,NaN,Orlík water reservoir,"(49.8067619, 13.3835292)",None
11,Chamra 2006,20.0,Metro transportation system,transport_others,disabled,NaN,Prague,NaN,NaN,Metro transportation system,disabled,NaN,Prague Metro,"(50.0874654, 14.4212535)",None
12,Chamra 2006,22.0,railway infrastructure,rail,flooded,NaN,Prague,NaN,NaN,railway infrastructure,flooded,NaN,"Prague Metro, Line A","(50.0874654, 14.4212535)",None
13,Rozendaal 2021,1.0,railway network,rail,disrupted,between thirty and fifty million euros,Wallonia,NaN,NaN,railway network,disrupted,between thirty and fifty million euros,NaN,None,None


##### Loc_2_coords for validation set
By using DuckDb 

In [ ]:
import requests
import json
from geopy.geocoders import Nominatim
import duckdb
import pandas as pd
import geopandas as gpd 
from glob import glob

## install osmium and spaita lextension once
duckdb.sql("""
    INSTALL spatial FROM core; 
    INSTALL osmium FROM community;
    LOAD spatial;
    LOAD osmium;
  """)
duckdb.close()  # NOTE always close the pointer! 



In [ ]:
# !uv remove duckdb
# !uv add duckdb=="1.5.1"
# !uv lock
# !uv sync
# # !duckdb  --version

In [ ]:
# df_valid.at[entry,  "coords"] = df_geoms[df_geoms["geometry"].notnull() & df_geoms["tags"]["name"] == loc]["geometry"].iloc[0]

FILEPATH_OSM_PBF =  'rheinland-pfalz-260531.osm.pbf' # "../europe-latest.osm.pbf"

geom_type_1 = 'relation'
geom_type_2 = 'area'
name = 'name'
ref_type = 'boundary'
ref_value = 'administrative'


# dummy
loc_name = "Ahrweiler"

# ## need to write to geojson to get transverted coordinates [epsg: 4326], with .df() only int-based coords
# duckdb.sql(f"""
#     LOAD osmium; LOAD spatial;
#     COPY(
#         SELECT id, tags['ref'] AS ref, json_value(tags, '$[0]') AS loc_type, tags, geometry, '{FILEPATH_OSM_PBF}' AS source
#         FROM '{FILEPATH_OSM_PBF}'
#         WHERE kind IN ('{geom_type_1}', '{geom_type_2}')
#             AND (tags['{name}'] = '{loc_name}' OR tags['alt_name'] = '{loc_name}' OR tags['alt_name:br'] = '{loc_name}' OR tags['short_name'] = '{loc_name}')
#             AND tags['{ref_type}'] = '{ref_value}'
#     ) TO  'locations_geocoded' (FORMAT GDAL, DRIVER GeoJSON, OVERWRITE_OR_IGNORE, PARTITION_BY (loc_type), FILENAME_PATTERN '{loc_name}')
#     ;
# """) 
# duckdb.close() 

In [ ]:
# df_geoms# [df_geoms["geometry"].notnull() & df_geoms["tags"]["name"] == loc]["geometry"].iloc[0]


In [ ]:
# #############   ------------- EXTRACT geoms for locations in valid data -------------   ##############

# # FILEPATH_OSM_PBF = "europe-latest.osm.pbf" 
# import re

# FILEPATH_OSM_PBF = 'rheinland-pfalz-260531.osm.pbf'


# ## init geolocalized location info (name of loc in OSM and its coordinates or bbox of potential location)
# df_valid["location_osm"] = None
# df_valid["coords"] = None
# df_valid["coords_coarse"] = None


# ## cleaning for localization
# df_valid["c1_location"] = df_valid["ci1_location"].str.replace(r"Ahr valley", "Ahrtal", regex=True)



# # reindex to avoid issues in iloc
# df_valid.reset_index(drop=True, inplace=True)

# ## get most likely coordinates for locations
# counter_errors = 0
# for entry in range(len(df_valid)):

#     loc_name = df_valid["ci1_location"].iloc[entry]
    
#     # loc_potential_coords = df_valid["coord_potential_locations"].iloc[entry]
    
#     # go to next entity when it is NAN
#     if loc_name == np.nan or loc_name == "nan" or loc_name == "NAN" or loc_name == "NaN" or loc_name == "":
#         continue



# for doc in df_valid.publication_id.unique():
    

#     # FILEPATH_OSM_PBF = "europe-latest.osm.pbf" 
#     FILEPATH_OSM_PBF = 'rheinland-pfalz-260531.osm.pbf'

#     ## TESTING 
#     if doc not in  ["Koks 2022"]:# , "Khazai 2013"]:
#         continue

#     print(f"\nDocument: {doc}")

#     df_single_doc = df_valid[df_valid["publication_id"] == doc]
    
#     ## load location names (based on OSM), when existent, and get their geoms
#     for loc_name in df_single_doc["ci1_location"].dropna().unique():  
#         print(loc_name)
#     # for loc_name, loc_type in zip(df_valid["location"], df_valid["addresstype"]):


#         try:
#             geom_type_1 = 'relation'
#             geom_type_2 = 'area'
#             name = "name"
#             ref_type = 'boundary'
#             ref_value = "administrative" 
#             # loc_value = xxx  # "Bad Münstereifel", "Eschweiler", "Ahrweiler district"

#             # if loc_type in ["city", "town", "village", "region", "state", "county"]:
#             #     geom_type = 'relation'         
            
#             # extract Ci locations from PBF file
#             # NOTE: explicit LOAD osmium and spatial for GDAL format
#             r = duckdb.sql(f"""
#                 LOAD osmium; LOAD spatial;
#                 COPY(
#                     SELECT id, tags['ref'] AS ref, json_value(tags, '$[0]') AS loc_type, tags, geometry, '{FILEPATH_OSM_PBF}' AS source
#                     FROM '{FILEPATH_OSM_PBF}'
#                     WHERE kind IN ('{geom_type_1}', '{geom_type_2}')
#                         AND (tags['{name}'] = '{loc_name}' OR tags['alt_name'] = '{loc_name}' OR tags['alt_name:br'] = '{loc_name}' OR tags['name:en'] = '{loc_name}' OR tags['short_name'] = '{loc_name}')
#                         AND tags['{ref_type}'] = '{ref_value}'
#                 ) TO  'locations_geocoded' (FORMAT GDAL, DRIVER GeoJSON, LAYER_NAME '{loc_name}_{ref_value_2}',OVERWRITE_OR_IGNORE, PARTITION_BY (loc_type), FILENAME_PATTERN '{loc_name}')
#                 ;
#             """)# .df()
#             duckdb.close()  
    

#         except Exception as e:

#             geom_type_1 = 'relation'
#             geom_type_2 = 'area'
#             name = "name"
#             ref_type = 'boundary'
#             ref_value = "administrative"
            
#             ## try without district prefix/postfix, e.g. "Ahrweiler district" --> "Ahrweiler"
#             if "district" in loc_name:
#                 loc_name = loc_name.replace("district", "").strip()

#                 print(f"Trying again with modified location name: {loc_name}")
#                 try:
#                     r = duckdb.sql(f"""
#                         LOAD osmium; LOAD spatial;
#                         COPY(
#                             SELECT id, tags['ref'] AS ref, json_value(tags, '$[0]') AS loc_type, tags, geometry, '{FILEPATH_OSM_PBF}' AS source
#                             FROM '{FILEPATH_OSM_PBF}'
#                             WHERE kind IN ('{geom_type_1}', '{geom_type_2}')
#                                 AND (tags['{name}'] = '{loc_name}' OR tags['alt_name'] = '{loc_name}' OR tags['alt_name:br'] = '{loc_name}' OR tags['name:en'] = '{loc_name}' OR tags['short_name'] = '{loc_name}')
#                                 AND tags['{ref_type}'] = '{ref_value}'
#                         ) TO  'locations_geocoded' (FORMAT GDAL, DRIVER GeoJSON, LAYER_NAME '{loc_name}_{ref_value_2}', OVERWRITE_OR_IGNORE, PARTITION_BY (loc_type), FILENAME_PATTERN '{loc_name}')
#                         ;
#                     """)
#                     duckdb.close() 

#                 except Exception as e:
#                     print(f"Error occurred while processing location **{loc_name}**: {e}")
#                     duckdb.close() 
#                     continue
#             pass

#         try: 
#             # highways
#             # if ref_type == 'highway' or ref_type == 'river':    
#             geom_type_1 = 'way'
#             geom_type_2 = 'line'
#             name = "ref"
#             ref_type = 'highway'
#             ref_value = "motorway"
        
#             duckdb.sql(f"""
#                 COPY(
#                     LOAD osmium; LOAD spatial;
#                     SELECT id, tags['ref'] AS ref, json_value(tags, '$[0]') AS loc_type, tags, geometry, '{FILEPATH_OSM_PBF}' AS source
#                     FROM '{FILEPATH_OSM_PBF}'
#                     WHERE kind IN ('{geom_type_1}', '{geom_type_2}')
#                         AND (tags['{name}'] = '{loc_name}' OR tags['alt_name'] = '{loc_name}' OR tags['alt_name:br'] = '{loc_name}' OR tags['name:en'] = '{loc_name}' OR tags['short_name'] = '{loc_name}')
#                         AND tags['{ref_type}'] = '{ref_value}'
#                 ) TO  'locations_geocoded' (FORMAT GDAL, DRIVER GeoJSON, LAYER_NAME '{loc_name}_{ref_value_2}', OVERWRITE_OR_IGNORE, PARTITION_BY (loc_type), FILENAME_PATTERN '{loc_name}')
#                 ;
#             """)
#             duckdb.close()


#         except Exception as e:

#             geom_type_1 = 'way'
#             geom_type_2 = 'line'
#             name = "ref"
#             ref_type = 'highway'
#             ref_value = "motorway"
            
#             ## try if highway name misses whitespace, A76 -> A 76
#             if bool(re.search(r'\d', loc_name)): # does str contain digit
#                 # alternative:  (?<=\D)(?=\d) - matches a position between a non-digit (\D) and a digit (\d)
#                 loc_name = re.sub("[A-Za-z]+", lambda e: " " + e[0] + " ", loc_name)
                
#                 print(f"Trying again with modified location name: {loc_name}")
#                 try:
#                     r = duckdb.sql(f"""
#                         LOAD osmium; LOAD spatial;
#                         COPY(
#                             SELECT id, tags['ref'] AS ref, json_value(tags, '$[0]') AS loc_type, tags, geometry, '{FILEPATH_OSM_PBF}' AS source
#                             FROM '{FILEPATH_OSM_PBF}'
#                             WHERE kind IN ('{geom_type_1}', '{geom_type_2}')
#                                 AND (tags['{name}'] = '{loc_name}' OR tags['alt_name'] = '{loc_name}' OR tags['alt_name:br'] = '{loc_name}' OR tags['name:en'] = '{loc_name}' OR tags['short_name'] = '{loc_name}')
#                                 AND tags['{ref_type}'] = '{ref_value}'
#                         ) TO  'locations_geocoded' (FORMAT GDAL, DRIVER GeoJSON, LAYER_NAME '{loc_name}_{ref_value_2}', OVERWRITE_OR_IGNORE, PARTITION_BY (loc_type), FILENAME_PATTERN '{loc_name}')
#                         ;
#                     """)
#                     duckdb.close() 

#                 except Exception as e:
#                     print(f"Error occurred while processing location **{loc_name}**: {e}")
#                     duckdb.close() 
#                     continue
#             pass

# ###########################
#         except Exception as e:
#             pass

#         try:
#             # natural
#             geom_type_1 = 'way'
#             geom_type_2 = 'relation'
#             name = "name"
#             ref_type_1 = 'natural'
#             ref_value_1 = "valley"
#             ref_type_2 = "waterway"
#             ref_value_2 = "river"
        
#             duckdb.sql(f"""
#                 LOAD osmium; LOAD spatial;
#                 COPY(
#                     SELECT id, tags['ref'] AS ref, json_value(tags, '$[0]') AS loc_type, tags, geometry, '{FILEPATH_OSM_PBF}' AS source
#                     FROM '{FILEPATH_OSM_PBF}'
#                     WHERE kind IN ('{geom_type_1}', '{geom_type_2}')
#                         AND (tags['{name}'] = '{loc_name}' OR tags['alt_name'] = '{loc_name}' OR tags['alt_name:br'] = '{loc_name}' OR tags['name:en'] = '{loc_name}' OR tags['short_name'] = '{loc_name}')
#                         AND (tags['{ref_type_1}'] = '{ref_value_1}' 
#                         OR tags['{ref_type_2}'] = '{ref_value_2}')

#                 ) TO  'locations_geocoded' (FORMAT GDAL, DRIVER GeoJSON, LAYER_NAME '{loc_name}_{ref_value_2}', OVERWRITE_OR_IGNORE, PARTITION_BY (loc_type), FILENAME_PATTERN '{loc_name}')
#                 ;
#             """) # TODO query later when addresstype is given based on addresstype
#             duckdb.close()

#         except Exception as e:
#             print(f"Error occurred while processing location **{loc_name}**: {e}")
#             duckdb.close() 
#             continue

#     # ) TO  'output.csv' (FORMAT CSV, HEADER, DELIMITER ',')

#         time.sleep(5)  # to avoid overloading the system, adjust as needed

# # TODO fix followings: not found
# # Ahr valley
# # Trying again with modified location name:  A 1
# # Mayschoss
# # Erft region
# # Eschweiler
# # river sAhr and Erft <- need "and", "between x and y" splits/handling


In [ ]:
# duckdb.close()  # NOTE always close the pointer! 



# geom_type_1 = 'relation'
# geom_type_2 = 'area'
# name = "name"
# ref_type = 'boundary'
# ref_value = "administrative" 

# loc_name = "Sinzig"

# # loc_value = xxx  # "Bad Münstereifel", "Eschweiler", "Ahrweiler district"

#             # if loc_type in ["city", "town", "village", "region", "state", "county"]:
#             #     geom_type = 'relation'         
            
#             # extract Ci locations from PBF file
#             # NOTE: explicit LOAD osmium and spatial for GDAL format

# r = duckdb.sql(f"""
#     LOAD osmium; LOAD spatial;
#     COPY(
#         SELECT id, tags['ref'] AS ref, json_value(tags, '$[0]') AS loc_type, tags, geometry, '{FILEPATH_OSM_PBF}' AS source
#         FROM '{FILEPATH_OSM_PBF}'
#         WHERE kind IN ('{geom_type_1}', '{geom_type_2}')
#             AND tags['{name}'] = '{loc_name}'
#             AND tags['{ref_type}'] = '{ref_value}'
#        ) TO  'locations_geocoded_2.geojson' WITH (DRIVER 'GeoJSON', FORMAT gdal, LAYER_NAME 'test_layer_name', OVERWRITE_OR_IGNORE, PARTITION_BY (loc_type), FILENAME_PATTERN '{loc_name}', SRS 'EPSG:4326')
#     ;
# """)

# # AND (tags['{name}'] = '{loc_name}' OR tags['alt_name'] = '{loc_name}' OR tags['alt_name:br'] = '{loc_name}' OR tags['name:en'] = '{loc_name}' OR tags['short_name'] = '{loc_name}')
# # TO  'locations_geocoded' (DRIVER JSON, FORMAT GDAL, OVERWRITE_OR_IGNORE, PARTITION_BY (loc_type), FILENAME_PATTERN '{loc_name}')

#### write geoms back to df_valid


In [ ]:
# ## catch localized CI locations
# geolocs = glob("./locations_geocoded/**/*geojson")
# print(geolocs [:4])

# df_locs = gpd.GeoDataFrame({
#     "citation_id": [],
# }, geometry=[], crs="EPSG:4326")

# # for doc_citation in df_pred_geolocalized.citation_id.unique():

# for geoloc in geolocs:
#     # geoloc = geoloc.replace("json ", "")
#     df_geoloc = gpd.read_file(geoloc, driver="GeoJSON")#, geometry="geometry", crs="EPSG:4326")

#     # df_geoloc = gpd.GeoDataFrame(df_geoloc, geometry="geometry", crs="EPSG:4326")  # ensure correct geometry and CRS
#     # enrich file with meta data
#     # df_geoloc["citation_id"] = doc_citation
#     # df_geoloc["location_name"] = ref


#     df_locs = pd.concat([df_locs, df_geoloc], ignore_index=True)



In [ ]:
## add to original valid df

# df_locs[df_locs["geometry"].notnull() & df_locs["tags"]["name"] == loc]["geometry"].iloc[0]

# df_valid.at[entry,  "coords"] = df_locs[df_locs["geometry"].notnull() & df_locs["tags"]["name"] == loc]["geometry"].iloc[0]


In [ ]:
## catch localized CI locations
geolocs = glob("./locations_geocoded/**/*geojson")
print(geolocs [:4])

df_locs = gpd.GeoDataFrame({
    "citation_id": [],
}, geometry=[], crs="EPSG:4326")

# for doc_citation in df_pred_geolocalized.citation_id.unique():

for geoloc in geolocs:
    # geoloc = geoloc.replace("json ", "")
    df_geoloc = gpd.read_file(geoloc, driver="GeoJSON")#, geometry="geometry", crs="EPSG:4326")

    # df_geoloc = gpd.GeoDataFrame(df_geoloc, geometry="geometry", crs="EPSG:4326")  # ensure correct geometry and CRS
    # enrich file with meta data
    # df_geoloc["citation_id"] = doc_citation
    # df_geoloc["location_name"] = ref


    df_locs = pd.concat([df_locs, df_geoloc], ignore_index=True)



['./locations_geocoded/loc_type=__HIVE_DEFAULT_PARTITION__/A 30.json geojson', './locations_geocoded/loc_type=__HIVE_DEFAULT_PARTITION__/Ahrtal0.json geojson', './locations_geocoded/loc_type=__HIVE_DEFAULT_PARTITION__/Altenahr0.json geojson', './locations_geocoded/loc_type=__HIVE_DEFAULT_PARTITION__/Ahrweiler0.json geojson']


/home/a-buch/Documents/TUB_DWN/_PROJECTS/CI-impacts-information-retrieval/.venv/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarning: driver GeoJSON does not support open option DRIVER
  return ogr_read(
/home/a-buch/Documents/TUB_DWN/_PROJECTS/CI-impacts-information-retrieval/.venv/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarning: driver GeoJSON does not support open option DRIVER
  return ogr_read(
/home/a-buch/Documents/TUB_DWN/_PROJECTS/CI-impacts-information-retrieval/.venv/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarning: driver GeoJSON does not support open option DRIVER
  return ogr_read(
/home/a-buch/Documents/TUB_DWN/_PROJECTS/CI-impacts-information-retrieval/.venv/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarning: driver GeoJSON does not support open option DRIVER
  return ogr_read(
/home/a-buch/Documents/TUB_DWN/_PROJECTS/CI-impacts-information-retrieval/.venv/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarn

## Select records which have text references

In [62]:
print(len(df_pred), len(df_valid))
df_pred = df_pred[~df_pred["chunk_text"].isna()].reset_index(drop=True)
df_valid = df_valid[~df_valid["sentence_reference"].isna()].reset_index(drop=True)
print(len(df_pred), len(df_valid))


197 146
197 146


## Translation of validation sentences

In [63]:

for entry in df_valid.itertuples():
    
    src_language = langdetect.detect(str(entry.sentence_reference))
    
    if src_language != "en":
        supported_languages = ["fr", "de", "es", "it", "itc", "nl"]
        if src_language not in supported_languages:
            print(f"Unsupported source language: {src_language}. Continue with original version of the sentence in validation set ")
            continue 

        print(f"\n ######## -------- Translating {entry.publication_id}: {src_language} --> en -------- ######## \n")

        # # clean up before applying translator
        # gc.collect()
        # torch.cuda.empty_cache()  # mainly after training needed, small effect when LLM applied only for inference
        # torch.no_grad()
        
        # overwrite original sentence(s) with translated versions
        translated_sentence = tm.translate_2_english(src_language, str(entry.sentence_reference))
        df_valid.loc[df_valid.index[df_valid["sentence_reference"] == entry.sentence_reference], "sentence_reference"] = translated_sentence


In [64]:
df_valid.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 146 entries, 0 to 145
Data columns (total 29 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   event_id                    146 non-null    object 
 1   event_time                  146 non-null    object 
 2   publication_id              146 non-null    object 
 3   sentence_reference          146 non-null    object 
 4   Unnamed: 4                  16 non-null     object 
 5   ci1_type                    146 non-null    object 
 6   ci_damage_numeric           61 non-null     object 
 7   ci23_damage_numeric_merged  12 non-null     object 
 8   Unnamed: 8                  38 non-null     object 
 9   ci1_damage                  144 non-null    object 
 10  ci23_dam_test               19 non-null     object 
 11  ci1_location                146 non-null    object 
 12  ci1_location_spec           67 non-null     object 
 13  ci23_type                   46 non-

In [65]:
# # unicode to ascii representation
# print("Apply unicode on CI and damages, but not on Locations (with ä, ü and other special chars) as it removes them potentially from the DFs" )
# try:
#     for col in ["infrastructure_group", "infrastructure_type", "damage", "location", "ci_entity", "geo_entity"]:
#         df_pred[col] = df_pred[col].apply(lambda x: unidecode(x) if isinstance(x, str) else x) # handle potential np.nan
# except KeyError as e:
#     for col in ["infrastructure_group", "infrastructure_type", "damage", "location"]:
#         df_pred[col] = df_pred[col].apply(lambda x: unidecode(x) if isinstance(x, str) else x) # handle potential np.nan

# for col in ["ci1_group", "ci1_type", "ci1_damage", "ci1_location"]:
#     df_valid[col] = df_valid[col].apply(lambda x: unidecode(x) if isinstance(x, str) else x) # handle potential np.nan


## add unique identifiers
helps in calculating FPs and FNs


In [66]:
# df_pred[["citation_id",	"chunk_id",	"infrastructure_type",	"infrastructure_group",	"damage",	"location"	]]

df_pred.loc[:,"id_pred"] = df_pred.reset_index().index
df_valid.loc[:,"id_valid"] = df_valid.reset_index().index

## Document-wise evaluation

Measures simply if the predicted CI-LOC case also occurs in the validation set\
It does not measure the frequency - just if the CI-LOC case exists in the validation set. In this way, the approach is similar to a spatial evaluation which also just captures the occurrence and location, not the frequency with which the impact (ie CI-LOC case) was reported in the document 


In [84]:
## WORKAORUND - to check how spat. eval could perform eventually (or when variations of location names are aligned)
# regex=True needed to recognize relevant part of longer string (here "Malaga ")
df_pred["location"] = df_pred["location"].str.replace(r".*airport", " ", regex=True).str.strip() # malaga airport, catania airport
df_pred["location"] = df_pred["location"].str.replace(r".*Airport", " ", regex=True).str.strip()

df_pred["location"] = df_pred["location"].str.replace(r"Port of Valencia", "Valencia", regex=True)
df_pred["location"] = df_pred["location"].str.replace(r"Maramureş Depression", "Maramureş County", regex=True).str.strip()
df_pred["location"] = df_pred["location"].str.replace(r"Maramureş County", "Maramureş", regex=True).str.strip()
df_pred["location"] = df_pred["location"].str.replace(r"Maramureş County County", "Maramureş", regex=True).str.strip()
df_pred["location"] = df_pred["location"].str.replace(r"Mar amureş\b", "Maramureş", regex=True).str.strip()
df_pred["location"] = df_pred["location"].str.replace(r"city of", "", regex=True).str.strip()
df_pred["location"] = df_pred["location"].str.replace(r"town of", "", regex=True).str.strip()
df_pred["location"] = df_pred["location"].str.replace(r"city", "", regex=True).str.strip()
df_pred["location"] = df_pred["location"].str.replace(r" port", "", regex=True).str.strip()
df_pred["location"] = df_pred["location"].str.replace(r" Port", "", regex=True).str.strip()
df_pred["location"] = df_pred["location"].str.replace(r" basin", "", regex=True).str.strip()
df_pred["location"] = df_pred["location"].str.replace(r"  ", " ", regex=True).str.strip()


df_valid["ci1_location"] = df_valid["ci1_location"].str.replace(r".*airport", " ", regex=True).str.strip() # malaga airport, catania airport
df_valid["ci1_location"] = df_valid["ci1_location"].str.replace(r".*Airport", " ", regex=True).str.strip()
df_valid["ci1_location"] = df_valid["ci1_location"].str.replace(r"Port of Valencia", "Valencia port", regex=True)

df_valid["ci1_location"] = df_valid["ci1_location"].str.replace(r"Maramureş Depression", "Maramureş", regex=True).str.strip()
df_valid["ci1_location"] = df_valid["ci1_location"].str.replace(r"Maramureş County", "Maramureş", regex=True).str.strip()

df_valid["ci1_location"] = df_valid["ci1_location"].str.replace(r"city of", "", regex=True).str.strip()
df_valid["ci1_location"] = df_valid["ci1_location"].str.replace(r"town of", "", regex=True).str.strip()
df_valid["ci1_location"] = df_valid["ci1_location"].str.replace(r"city", "", regex=True).str.strip()
df_valid["ci1_location"] = df_valid["ci1_location"].str.replace(r" port", "", regex=True).str.strip()
df_valid["ci1_location"] = df_valid["ci1_location"].str.replace(r" Port", "", regex=True).str.strip()
df_valid["ci1_location"] = df_valid["ci1_location"].str.replace(r" basin", "", regex=True).str.strip()
df_valid["ci1_location"] = df_valid["ci1_location"].str.replace(r"  ", " ", regex=True).str.strip()


## remove "in ", "near ", "close to", "passing " from location names
df_pred["location"] = df_pred["location"].str.replace(r"^(in |near |close to |parts of |direction of |passing |along )", " ", regex=True).str.strip()
df_valid["ci1_location"] = df_valid["ci1_location"].str.replace(r"^(in |near |close to |parts of |direction of |passing |along )", " ", regex=True).str.strip()



## drop "district"
df_pred["location"] = df_pred["location"].str.replace(r"district", "", regex=True).str.strip()
df_valid["ci1_location"] = df_valid["ci1_location"].str.replace(r"district", "", regex=True).str.strip()

##  "Rheinland-Pfalz" --> "Rhineland-palatinate"
df_pred["location"] = df_pred["location"].str.replace(r"Rheinland-Pfalz", "Rhineland-Palatinate", regex=True).str.strip()
df_valid["ci1_location"] = df_valid["ci1_location"].str.replace(r"Rheinland-Pfalz", "Rhineland-Palatinate", regex=True).str.strip()
df_pred["location"] = df_pred["location"].str.replace(r"RhinelandPalatinate", "Rhineland-Palatinate", regex=True).str.strip()
df_valid["ci1_location"] = df_valid["ci1_location"].str.replace(r"RhinelandPalatinate", "Rhineland-Palatinate", regex=True).str.strip()


print(len(df_pred))
print(len(df_valid))

## drop numbers of highways in LOC
def drop_numbers_in_location(df_pred, column):
    print("remove rows which contain a number in a certain column, eg. B518")
    df_pred = df_pred[~df_pred[column].str.contains(r'.*[0-9].*', na=True)]
    print(len(df_pred))
    return df_pred

df_pred = drop_numbers_in_location(df_pred, "location")
df_valid = drop_numbers_in_location(df_valid, "ci1_location")


187
143
remove rows which contain a number in a certain column, eg. B518
187
remove rows which contain a number in a certain column, eg. B518
143


In [85]:
# ## TEST rm records where no geolocalization was possible (incl. coarse coords)
# print(f"Would remove {len(df_pred[df_pred['coords'].isna()])} from {len(df_pred)} records where no geolocalization was possible (incl. coarse coords)")

## --> currently incl. non-geolocalized entries in the EVAL process ->  leads better performance scores
df_pred_geolocalized = df_pred#[~df_pred["coords"].isna()]

# df_pred[df_pred['coords'].isna()][
#     ["citation_id", "chunk_id", "infrastructure_type", "infrastructure_group", "location", "damage", "coord_potential_locations"]
#     ][:60]

In [86]:
# df_pred[df_pred["coords"].isna()][["citation_id","chunk_id","infrastructure_type","infrastructure_group","damage","damage_value","location", "coord_potential_locations"]][:60]


#- drop cases from newer valid docs
df_pred_geolocalized = df_pred_geolocalized[~df_pred_geolocalized.citation_id.isin(["Chamra 2006", "Hladny 2004", "Pescaroli 2017"])]
df_valid = df_valid[~df_valid.publication_id.isin(["Chamra 2006", "Hladny 2004", "Pescaroli 2017"])]

In [87]:
fns_list = []
fps_list = []   
tps_list = []



for publication in df_valid.publication_id.unique():

    print(f"\n\nDocument: {publication}")
    docs_valid = df_valid[df_valid["publication_id"] == publication]
    docs_pred = df_pred_geolocalized[df_pred_geolocalized["citation_id"] == publication]

    # if publication in ['Wildhagen 2013', "EFE 2024" ]:  # get all non-english texts
    #     pass
    # else:
    #     continue  # english text , pass to ext doc

    print(f"Document: {publication}")

    # get all valid and predicted CI-LOC pairs for each publication
    docs_valid_pairs = docs_valid[["ci1_group", "ci1_location"]].drop_duplicates().values.tolist()
    docs_pred_pairs = docs_pred[["infrastructure_group", "location"]].drop_duplicates().values.tolist()
    print(docs_valid_pairs)
    print(docs_pred_pairs)
    
    # calc TPs, FPs, FNs
    tps = len([t for t in docs_pred_pairs if t in docs_valid_pairs]) 
    fps = len([t for t in docs_pred_pairs if t not in docs_valid_pairs]) 
    fns = len([t for t in docs_valid_pairs if t not in docs_pred_pairs]) # geo-llama.trsting_on_news2024.ipynb
    print(f"TPs: {tps}, FPs: {fps}, FNs: {fns}")
    print(f"  FPs: { [t for t in docs_pred_pairs if t not in docs_valid_pairs]}")
    print(f"  FNs: { [t for t in docs_valid_pairs if t not in docs_pred_pairs]}")

    print(f"---------- For each Document - Evaluation statistics (document-wise)-----------")
    recall_score = u.calc_recall(tps_no=tps, fns_no=fns) 
    precision_score = u.calc_precision(tps_no=tps, fps_no=fps)
    try:
        f1_score = u.calc_f1(precision=precision_score, recall=recall_score)
    except ZeroDivisionError:
        f1_score = 0.0
    print(f"Precision: {precision_score}, Recall: {recall_score}, F1-score: {f1_score}")
    tps_list.append(tps)
    fps_list.append(fps)
    fns_list.append(fns)
    
 
 
    #### ---------- check performance for quantified impacts  ---------- ####
    
    # # docs_v_dam1 = docs_valid["ci_damage_numeric"].drop_duplicates().values.tolist()
    # # docs_p_dam1 = docs_pred["damage_value"].drop_duplicates().values.tolist()
    # docs_v_dam1 = docs_valid["ci23_damage_numeric_merged"].drop_duplicates().values.tolist()
    # docs_p_dam1 = docs_pred["damage_value"].drop_duplicates().values.tolist()    
    # print(docs_v_dam1)
    # print(docs_p_dam1)
    # # print(docs_v_dam2)
    # # print(docs_p_dam2)    

    # # calc TPs, FPs, FNs
    # tps = len([t for t in docs_p_dam1 if t in docs_v_dam1]) 
    # fps = len([t for t in docs_p_dam1 if t not in docs_v_dam1]) 
    # fns = len([t for t in docs_v_dam1 if t not in docs_p_dam1]) # geo-llama.trsting_on_news2024.ipynb
    # print(f"TPs: {tps}, FPs: {fps}, FNs: {fns}")
    # tps_list.append(tps)
    # fps_list.append(fps)
    # fns_list.append(fns)
    # print(f"  TPs: { [t for t in docs_p_dam1 if t in docs_v_dam1]}")
    # print(f"  FPs: { [t for t in docs_p_dam1 if t not in docs_v_dam1]}")
    # print(f"  FNs: { [t for t in docs_v_dam1 if t not in docs_p_dam1]}")


recall_score = u.calc_recall(tps_no=sum(tps_list), fns_no=sum(fns_list)) 
precision_score = u.calc_precision(tps_no=sum(tps_list), fps_no=sum(fps_list))
try:
    f1_score = u.calc_f1(precision=precision_score, recall=recall_score)
except ZeroDivisionError:
    f1_score = 0.0

print(f"\n\n ---------- Evaluation statistics (document-wise)-----------")
print(f"Precision: {precision_score}, Recall: {recall_score}, F1-score: {f1_score}")

# only gelocolized:
# Precision: 0.275, Recall: 0.3728813559322034,  F1-score: 0.3165467625899281
# Precision: 0.3, Recall: 0.3305084745762712, F1-score: 0.31451612903225806
# Precision: 0.4230769230769231, Recall: 0.3618421052631579, F1-score: 0.3900709219858156
# Precision: 0.38636363636363635, Recall: 0.288135593220339, F1-score: 0.3300970873786408 # noner , Koks+2: TPs: 16, FPs: 12, FNs: 34
# Precision: 0.45161290322580644, Recall: 0.3783783783783784, F1-score: 0.411764705882353  # with ner, Koks+2 (doc-wise)
#  Precision: 0.6923076923076923, Recall: 0.627906976744186, F1-score: 0.6585365853658537, # with ner, Koks+2 (non-geolocalized, chunk-wise) tps 27  fps: 12  fns: 16

# Precision: 0.5612244897959183, Recall: 0.3741496598639456, F1-score: 0.4489795918367347

## chain of prompts (no-geoloco)
# Precision: 0.5294117647058824, Recall: 0.6, F1-score: 0.5625 (only koks, chain-of-prompts-non-geoloco, doc-wise)
# Precision: 0.47619047619047616,  Recall: 0.6060606060606061,  F1-score: 0.5333333333333333 (only koks, chain-of-prompts-non-geolo, chunk-wise)
# Precision: 0.43434343434343436, Recall: 0.36134453781512604, F1-score: 0.3944954128440367 (nearly all diocs, step 2, verified=true, chain-prompt-no-geoloc, doc-wise)
# Precision: 0.45714285714285713, Recall: 0.5333333333333333, F1-score: 0.4923076923076923 (only koks, step 2, verified=true, chain-prompt-no-geoloc, doc-wise)



# incl non gelocalized:
# Precision: 0.2660098522167488, Recall: 0.4576271186440678, F1-score: 0.33644859813084116
# Precision: 0.27647058823529413, Recall: 0.3983050847457627, F1-score: 0.3263888888888889
# Precision: 0.3941176470588235, Recall: 0.4407894736842105, F1-score: 0.4161490683229813
# Precision: 0.46153846153846156, Recall: 0.4864864864864865, F1-score: 0.47368421052631576 (ner: koks+2papers etc) doc-wise
# Precision: 0.5625, Recall: 0.627906976744186, F1-score: 0.5934065934065934 (ner, koks+2, chunk-wise) tps 27  fps: 21  fns: 16


# koks step2
# Precision: 0.45714285714285713, Recall: 0.5333333333333333, F1-score: 0.4923076923076923




Document: ABC 2024
Document: ABC 2024
[['airports', 'Malaga'], ['roads', 'Malaga'], ['education_school', 'Cártama'], ['roads', 'Lope de Vega Avenue'], ['roads', 'Julio Cortázar Avenue'], ['roads', 'Churriana intersection'], ['roads', 'Avenida Herrera Oria-Virgen de las Flores'], ['roads', 'Pasillo del Matadero Puente del Carmen'], ['roads', 'Pasillo Santa Isabel - Puente de la Aurora'], ['roads', 'Avenida Lope de Vega - Atabal']]
[['airports', ''], ['transport_supply', ''], ['roads', 'Guerrero Strachan Avenue'], ['roads', 'Velázquez Avenue'], ['roads', 'Malaga'], ['airports', 'Malaga'], ['education_school', 'Cártama']]
TPs: 3, FPs: 4, FNs: 7
  FPs: [['airports', ''], ['transport_supply', ''], ['roads', 'Guerrero Strachan Avenue'], ['roads', 'Velázquez Avenue']]
  FNs: [['roads', 'Lope de Vega Avenue'], ['roads', 'Julio Cortázar Avenue'], ['roads', 'Churriana intersection'], ['roads', 'Avenida Herrera Oria-Virgen de las Flores'], ['roads', 'Pasillo del Matadero Puente del Carmen'], ['

In [71]:
df_pred[df_pred.citation_id == "Khazai 2013"][["infrastructure_type", "location", "chunk_text"]]

# STEP 1 (with NER tool)
# Precision: 0.32, Recall: 0.4129032258064516, F1-score: 0.36056338028169016 (incl chamra ..)
# Precision: 0.3765432098765432, Recall: 0.41496598639455784, F1-score: 0.39482200647249194 (excl chamra ..)

# STEP 1 (without NER tool)
# Precision: 0.46616541353383456, Recall: 0.4217687074829932, F1-score: 0.44285714285714284 (excl chamra ..)


# STEP 2 (with NER)
# Precision: 0.35526315789473684, Recall: 0.34838709677419355, F1-score: 0.3517915309446254 (incl chamra ..)
# Precision: 0.46551724137931033, Recall: 0.3698630136986301, F1-score: 0.4122137404580153  (excl chamra ..)

# STEP 2 (without NER tool)
# Precision: 0.5267857142857143, Recall: 0.4041095890410959, F1-score: 0.45736434108527135 (excl chamra ..)


,infrastructure_type,location,chunk_text
144,Water network,Danube-Elbe catchment area,"Water network affected by at least 5 years of floods) At many lakes in the Danube and Elbe catchment area new record values (water level, drainage) Type of flood: Spreads river floods with profusions and danger of dike failure with surface flooding of the hinterland Preconditions and meteorological causes Snow cover in the alpine high mountains until May. Very wet May, therefore widespread oversaturation of the soil (largest extent in 50 years), greatly reduced water absorption capacity of the soil.\nStable large-scale weather conditions (TM) brought constantly humid air from south-east Europe to north and from north-east direction to central Europe. Heavy rain areas are reinforced in the congested mountains and Alps: Long-lasting heavy precipitations at the middle mountains and the Alpine edge.\nSources Own analyses, Deutscher Wetterdienst (DWD), wetterfahrt-fruehwarz.de, Hochfluchtzentrale.de.\nBasic information Impacts of fatalities and evacuation(as at 07."
145,German highways,89 s,June 2013) 7 fatalities in Germany (24 fatalities in all affected states) at least 52500 persons affected by evacuations on Danube and Elbe Infrastructure interruptions (31.05. to 04.06.2013) Traffic obstructions on German highways and federal roads due to floods in 89 districts. Summates at least 4866 h of traffic obstructions in the national transport network.
146,German highways,89 s,June 2013) 7 fatalities in Germany (24 fatalities in all affected states) at least 52500 persons affected by evacuations on Danube and Elbe Infrastructure interruptions (31.05. to 04.06.2013) Traffic obstructions on German highways and federal roads due to floods in 89 districts. Summates at least 4866 h of traffic obstructions in the national transport network.
147,national transport network,German highways,June 2013) 7 fatalities in Germany (24 fatalities in all affected states) at least 52500 persons affected by evacuations on Danube and Elbe Infrastructure interruptions (31.05. to 04.06.2013) Traffic obstructions on German highways and federal roads due to floods in 89 districts. Summates at least 4866 h of traffic obstructions in the national transport network.
148,national transport network,federal roads,June 2013) 7 fatalities in Germany (24 fatalities in all affected states) at least 52500 persons affected by evacuations on Danube and Elbe Infrastructure interruptions (31.05. to 04.06.2013) Traffic obstructions on German highways and federal roads due to floods in 89 districts. Summates at least 4866 h of traffic obstructions in the national transport network.
149,dikes,Deggendorf,"Long-lasting, strong rainfall, combined with extremely unfavourable conditions, has led to a large-scale flood event across the catchment areas. The event exceeds the August flood in 2002 and the previous record summer flood in July 1954 (subject to uncertainties in the raw data of water level and discharge, see Table 1). The catchment areas of Danube and Elbe are particularly affected (see Figure 1).\nOn the Danube, the flood rift has crossed the German-Austrian border. The highest water level of 12.75 m in Passau is a new historical record. In addition to Passau, the district of Deggendorf is particularly affected, where dikes have not resisted the high water levels and the permanent load."
150,dike breaks,Elbe,"The impact of this flood on people, traffic and the economy is great. Numerous dike breaks, for example in Bavaria and along the Elbe, resulted in large-scale flooding in the region. In many places, thousands of people had to leave their homes, houses and places due to evacuation measures. In the Salzlandkreis in Saxony-Anhalt, a voluntary helper died while filling the sandbags and a woman during an evacuation. In Baden-Württemberg, three people, including a fireman, have been killed since the start of the flood. Up to now, at least 24 victims and 5 missing persons have been killed 

## Understand evaluation method Ni Li for lists 

In [285]:

from statistics import mean


# taken from Ni Li 2026 to assess performance onf qualitative fields [lists]
def sequence(v, w):
    null_penality = 0.5
    """Compare sequences. Returns Jaccard distance between sets of elements in sequences.
    Note: ordering is not taken into consideration."""
    if v == None and w == None:
        return 0
    if v == None and w != None or v != None and w == None:
        print("! Using null penality as one list is empty")
        return null_penalty
    v, w = set(v), set(w)
    return 1.0 - len(v.intersection(w)) / len(v.union(w))

# my modification based on list:
def calc_simi_list(v, w):
    null_penality = 0.5
    if v == None and w == None:
        return 0
    if v == None and w != None or v != None and w == None:
        return null_penalty
    identical = [element for element in v if element in w]
    return np.round(1 -  (len(identical)/ np.abs((len(docs_pred_citype) + len(docs_valid_citype)))), 3)


In [286]:


def calc_similarity(gold_instance: list, sys_list=list): 
    #gold_instance: dict, sys_list: list) -> list[float]:
    int_cat: dict[str, int] = {
                "Num_Min": 1,
                "Num_Max": 1,
                "Start_Date_Day": 0.125,
                "Start_Date_Month": 0.125,
                "Start_Date_Year": 0.125,
                "End_Date_Day": 0.125,
                "End_Date_Month": 0.125,
                "End_Date_Year": 0.125,

                "ci1_group": 1,
                "location": 1,
                "damage": 1,
    }

    score_list: float = []
    # for si in sys_list:
    scores = []
    # for k in gold_instance.keys():
        # # NOTE: k = column from valid set
        # if k in self.int_cat:
        #     # Only include gold_instance[k] from numerical categories
        #     # For monetary categories, gold_instance[k] is a list
        #     # For numerical caterogies, it is always an int (or can be cast to an int)
        #     try:
        #         if isinstance(int(gold_instance[k]), int):
        #             r = self.comp.integer(gold_instance[k], si[k])
        #     except:
        #         pass
        # elif k in str_cat:
        # if k in [{"ci1_group": 1}]:
        #     r = comp.string(gold_instance[k], si[k])
        # elif k in list_cat:
    #if k in [{"ci1_group": 1}]:
        #   r = sequence(gold_instance[k], si[k])
    r = sequence(gold_instance, sys_list)
    try:
        scores.append(1 - (r * int_cat))
        del r
    except Exception:
        # if k != "Event_ID":
        # print(f"Unsupported column name: {k} will be ignored during matching.")
        print(f"Unsupported column name:will be ignored during matching.")

    score_list.append(mean(scores))

    # index of mean score corresponds to sys_list item
    return score_list


gold_list = ['airports', 'bridges', 'education_kita', 'education_others', 'education_school', 'electricity_others', 'electricity_supply', 'gas_distribution',      'gas_supply',          'healthcare_hospitals_clinics', 'healthcare_others', 'it_telecommunication', 'motorways', 'ports', 'rail', 'rail_service', 'roads', 'transport_others', 'transport_supply', 'tunnels', 'waste_others',        'wastewater', 'water_others', 'water_supply', 'waterprotection']
sys_list =  ['airports', 'aviation', 'bridges',       'education_kita', 'education_school', 'electricity_others', 'electricity_supply', 'gas_distribution', 'healthcare_hospitals_clinics', 'healthcare_others', 'it_telecommunication',                    'motorways', 'nursing', 'ports', 'rail', 'rail_service', 'roads', 'waste_others',           'wastewater', 'water_supply', 'waterprotection']# , "", "", "", "", ]
print(len(gold_list))
print(len(sys_list))
threshold  = 0.6

gold, sys, similarity, gold_matched, sys_matched = [], [], [], [], []
similarity_matrix = [calc_similarity(gold_list, sys_list)]

# similarity_matrix = [calc_similarity(si, sys_list) for si in gold_list]
best_matches = [
    (gi, si, similarity_matrix[gi][si])
    for gi in range(len(similarity_matrix))
    for si in range(len(similarity_matrix[gi]))
    if similarity_matrix[gi][si] > threshold
]
best_matches.sort(key=lambda x: x[2], reverse=True)

# find the best matches in the similarity matrix
for gi, si, sim in best_matches:
    if gi not in gold_matched and si not in sys_matched:
        gold.append(gold_list[gi])
        sys.append(sys_list[si])
        gold_matched.append(gi)
        sys_matched.append(si)
        similarity.append(sim)

# pad remaining unmatched specific instances
for gi in range(len(gold_list)):
    if gi not in gold_matched:
        gold.append(gold_list[gi])
        sys.append(self.create_pad(gold_list[gi]))

for si in range(len(sys_list)):
    if si not in sys_matched:
        sys.append(sys_list[si])
        gold.append(self.create_pad(sys_list[si]))

assert len(gold) == len(sys), AssertionError(
    f"Something went wrong! number of specific instances in gold: {len(gold)}; in sys: {len(sys)}"
)

for ds in [gold, sys]:
    counter = 0
    for si in ds:
        si["Event_ID"] = f"{si['Event_ID']}-{counter}"
        counter += 1

print(gold, sys)



25
21
Unsupported column name:will be ignored during matching.


StatisticsError: mean requires at least one data point

In [287]:

gold_list = ['airports', 'bridges', 'education_kita', 'education_others', 'education_school', 'electricity_others', 'electricity_supply', 'gas_distribution',      'gas_supply',          'healthcare_hospitals_clinics', 'healthcare_others', 'it_telecommunication', 'motorways', 'ports', 'rail', 'rail_service', 'roads', 'transport_others', 'transport_supply', 'tunnels', 'waste_others',        'wastewater', 'water_others', 'water_supply', 'waterprotection']
sys_list =  ['airports', 'aviation', 'bridges',       'education_kita', 'education_school', 'electricity_others', 'electricity_supply', 'gas_distribution', 'healthcare_hospitals_clinics', 'healthcare_others', 'it_telecommunication',                    'motorways', 'nursing', 'ports', 'rail', 'rail_service', 'roads', 'waste_others',           'wastewater', 'water_supply', 'waterprotection']# , "", "", "", "", ]

r = sequence(gold_list, sys_list)
print(r)
print(len(gold_list))
print(gold_list)
print(len(sys_list))
print(sys_list)

## padd lists, ie align length of both list, fill with Nones
len_gold =  len(gold_list)
len_sys =  len(sys_list)
if len_gold > len_sys:
    n = len_gold - len_sys
    fill = [""] * n
    sys_list = sys_list + fill 
elif len_gold < len_sys:
    n = len_sys - len_gold
    fill = [""] * n
    gold_list = gold_list + fill 

print(sys_list)

print("\npadded")
r = sequence(gold_list, sys_list)
print(r)
print(len(gold_list))
print(gold_list)
print(len(sys_list))
print(sys_list)

print("number of common elements in Koks -ci_group")
print(len(common_elements(sys_list, gold_list)))



# try:
#     scores.append(1 - (r * int_cat))
#     del r
# except Exception:
#     # if k != "Event_ID":
#     # print(f"Unsupported column name: {k} will be ignored during matching.")
#     print(f"Unsupported column name:will be ignored during matching.")

#         score_list.append(mean(score

0.2962962962962963
25
['airports', 'bridges', 'education_kita', 'education_others', 'education_school', 'electricity_others', 'electricity_supply', 'gas_distribution', 'gas_supply', 'healthcare_hospitals_clinics', 'healthcare_others', 'it_telecommunication', 'motorways', 'ports', 'rail', 'rail_service', 'roads', 'transport_others', 'transport_supply', 'tunnels', 'waste_others', 'wastewater', 'water_others', 'water_supply', 'waterprotection']
21
['airports', 'aviation', 'bridges', 'education_kita', 'education_school', 'electricity_others', 'electricity_supply', 'gas_distribution', 'healthcare_hospitals_clinics', 'healthcare_others', 'it_telecommunication', 'motorways', 'nursing', 'ports', 'rail', 'rail_service', 'roads', 'waste_others', 'wastewater', 'water_supply', 'waterprotection']
['airports', 'aviation', 'bridges', 'education_kita', 'education_school', 'electricity_others', 'electricity_supply', 'gas_distribution', 'healthcare_hospitals_clinics', 'healthcare_others', 'it_telecommun

## Benachmark against Ni Li - per doc and Ci sector 

score for boolean or str : 
if identical = 0
if not identical = 1

score for lists
= 1 - |{schnittmenge von a und r}|  / |{Vereiningunsmenge von a un d r}

-->  no. of common elements ["identicals"] / entire number of elements which occur at least in a or r or in both

```
Schnittmenge (\(\cap \)): Elemente, die in beide Mengen gehören.
Vereinigung (\(\cup \)): Elemente, die in mindestens einer der beiden Mengen liegen (Zeichen: \(A \cup B\), sieht aus wie ein normales „u“).
```

In [343]:
# def common_elements(list1, list2):
#     return list(set(list1) & set(list2)) 
#     #return [element for element in list1 if element in list2]

In [344]:
print(sorted(df_pred_geolocalized.infrastructure_group.unique()))
print(sorted(df_valid.ci1_group.unique()))



['airports', 'bridges', 'education_kita', 'education_others', 'education_school', 'electricity_others', 'electricity_supply', 'gas_distribution', 'gas_supply', 'healthcare_hospitals_clinics', 'healthcare_others', 'it_telecommunication', 'motorways', 'ports', 'rail', 'rail_service', 'roads', 'transport_others', 'transport_supply', 'tunnels', 'waste_others', 'wastewater', 'water_others', 'water_supply', 'waterprotection']
['airports', 'aviation', 'bridges', 'education_kita', 'education_school', 'electricity_others', 'electricity_supply', 'gas_distribution', 'healthcare_hospitals_clinics', 'healthcare_others', 'it_telecommunication', 'motorways', 'nursing', 'ports', 'rail', 'rail_service', 'roads', 'waste_others', 'wastewater', 'water_supply', 'waterprotection']


In [345]:

ci_sectors = {
    "airports" : "Transportation",
    "aviation" : "Transportation",
    "roads" : "Transportation",
    "bridges" : "Transportation",
    "motorways" : "Transportation",
    "rail" : "Transportation",
    "rail_service" : "Transportation",
    "ports" : "Transportation",
    "port_operation": "Transportation",
    "transport_others": "Transportation",
    "transport_supply": "Transportation",
    "tunnels" : "Transportation",
    "waterways_ships": "Transportation",

    "water_others" : "Water",
    "water_supply" : "Water",
    "waterprotection" : "Water",

    "electricity_others" : "Energy",
    "electricity_supply" : "Energy",
    "electricity_distribution": "Energy",
    "gas_distribution" : "Energy",
    "gas_supply" : "Energy",
    "renewable_wind": "Energy",
    "renewable_hydro": "Energy",

    "wastewater" : "Waste",
    "waste_others" : "Waste",

    "it_telecommunication" : "IT / communication", #Not specified",

    "healthcare_others" : "Social", # Not specified",
    "healthcare_hospitals_clinics" : "Social",
    "nursing" : "Social",
    "education_kita" : "Social",
    "education_others" : "Social",
    "education_school" : "Social",
}


docs_valid = df_valid.copy(deep=True)
docs_pred = df_pred_geolocalized.copy(deep=True)

## create ci sector columns
docs_valid["infrastructure_sector"] = docs_valid["ci1_group"].map(ci_sectors)  # non-matched entries are set to NAN
docs_pred["infrastructure_sector"] = docs_pred["infrastructure_group"].map(ci_sectors)  # non-matched entries are set to NAN


print(docs_valid["infrastructure_sector"].value_counts())
print(docs_pred["infrastructure_sector"].value_counts())


infrastructure_sector
Transportation        96
Social                15
Water                 14
Energy                 9
Waste                  7
IT / communication     2
Name: count, dtype: int64
infrastructure_sector
Transportation        114
Energy                 24
Social                 18
Water                  12
Waste                  12
IT / communication      7
Name: count, dtype: int64


In [346]:
from itertools import chain

list_entity_valid = ["ci1_group", "ci1_location", "ci1_damage"]
list_entity_pred = ["infrastructure_group", "location", "damage"]

# calc share of loc=1 and loc=0 (to benchmark against Ni li)

# # get alls cases of same doc 
# docs_pred = docs_pred[docs_pred["citation_id"] == "Koks 2022"]
# docs_valid = docs_valid[docs_valid["publication_id"] == "Koks 2022"]


# The remaining entries in the gold data or the LLM output are matched to empty entries padded with “NULL” values. 
# This results in two lists of entries of equal length, which is the desired format for evaluation
# Both cases result in the highest possible error rate – a NULL penalty score of “1”. To investigate this further

print(f"Using all Documents")
df_performamce_doc_ci = pd.DataFrame()


for doc_id in docs_valid["publication_id"].unique(): 

    # get alls cases of same doc 
    docs_pred_doc = docs_pred[docs_pred["citation_id"] == doc_id]
    docs_valid_doc = docs_valid[docs_valid["publication_id"] == doc_id]

    # get perofamcne for each ci sector within a doc
    for ci_sector in docs_valid_doc["infrastructure_sector"].unique():

        # for each ci secotr get perofmance score
        docs_pred_ci = docs_pred_doc[docs_pred_doc["infrastructure_sector"] == ci_sector]
        docs_valid_ci = docs_valid_doc[docs_valid_doc["infrastructure_sector"] == ci_sector]

        for pred_ci_loc_dam, valid_ci_loc_dam in zip(list_entity_pred, list_entity_valid):

            docs_valid_citype = docs_valid_ci[valid_ci_loc_dam].values.astype(str).tolist()
            docs_pred_citype = docs_pred_ci[pred_ci_loc_dam].values.astype(str).tolist()#.drop_duplicates().values.tolist()
            docs_valid_citype = sorted(docs_valid_citype)
            docs_pred_citype = sorted(docs_pred_citype)

            len_docs_valid_citype_nonan = len(docs_valid_citype)
            len_docs_pred_citype_nonan = len(docs_pred_citype)

            # docs_valid_citype = sorted(list(chain.from_iterable(docs_valid_citype)))
            # docs_pred_citype = sorted(list(chain.from_iterable(docs_pred_citype)))
            # if docs_valid_citype or docs_pred_citype:
            print(doc_id, "-", ci_sector, "-", pred_ci_loc_dam)
            print(docs_valid_citype)
            print(docs_pred_citype)

            #

            ## pad lists, 
            # ie align length of both list, fill with ""
            len_gold =  len(docs_valid_citype)
            len_sys =  len(docs_pred_citype)
            if len_gold > len_sys:
                n = len_gold - len_sys
                fill = [""] * n
                docs_pred_citype = docs_pred_citype + fill 
            elif len_gold < len_sys:
                n = len_sys - len_gold
                fill = [""] * n
                docs_valid_citype = docs_valid_citype + fill 


            # calc similarities
            score = sequence(docs_pred_citype, docs_valid_citype)  # hanldes also when one of the lists is empty == 1, if bot list empty == 0
            score_list = calc_simi_list(docs_pred_citype, docs_valid_citype)

            # if len(docs_pred_citype)!=0 and len(docs_valid_citype)!=0:
            #     identical = common_elements(docs_pred_citype, docs_valid_citype) # TP
            # elif len(docs_pred_citype)==0 and len(docs_valid_citype)==0:
            #     score = 0.0
            #     # print(f"no CI {ci_sector} [field: {valid_ci_loc_dam}] in: {doc_id}")
            # else:
            #     score = sequence(docs_pred_citype, docs_valid_citype)

            # print("share according to Ni Li - list", np.round(1 -  (len(identical)/ np.abs((len(docs_pred_citype) + len(docs_valid_citype)))), 3))
            # print("share according to Ni Li - set ", np.round(1 - (len(set(identical))/  np.abs((len(set(docs_pred_citype)) + len(set(docs_valid_citype))))), 3))
            print("share according to Ni Li - set [sequence()] ",  np.round(score, 3)) # np.round(1 - (len(set(identical))/  np.abs((len(set(docs_pred_citype)) + len(set(docs_valid_citype))))), 3))
            print("share according to Ni Li - set [my_calc_list()] ",  np.round(score_list, 3)) # np.round(1 - (len(set(identical))/  np.abs((len(set(docs_pred_citype)) + len(set(docs_valid_citype))))), 3))

            df_performamce_doc_ci = pd.concat([df_performamce_doc_ci, pd.DataFrame(
                {
                    "citation_id": [doc_id],
                    "ci_sector": [ci_sector],
                    "number_cases_pred": [len_docs_pred_citype_nonan],
                    "number_cases_valid": [len_docs_valid_citype_nonan],
                    f"{pred_ci_loc_dam}": [score],
                    f"{pred_ci_loc_dam}_list": [score_list],
                }
            ) ])
            #(37/ (docs_pred_citype + docs_valid_citype))

            del docs_valid_citype
            del docs_pred_citype




Using all Documents
ABC 2024 - Transportation - infrastructure_group
['airports', 'airports', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads']
['airports', 'airports', 'airports', 'airports', 'roads', 'roads', 'roads', 'transport_supply']
share according to Ni Li - set [sequence()]  0.5
share according to Ni Li - set [my_calc_list()]  0.682
ABC 2024 - Transportation - location
['Avenida Herrera Oria-Virgen de las Flores', 'Avenida Lope de Vega - Atabal', 'Churriana intersection', 'Julio Cortázar Avenue', 'Lope de Vega Avenue', 'Lope de Vega Avenue', 'Malaga', 'Malaga', 'Malaga', 'Pasillo Santa Isabel - Puente de la Aurora', 'Pasillo del Matadero Puente del Carmen']
['', '', '', '', 'Guerrero Strachan Avenue', 'Malaga', 'Malaga', 'Velázquez Avenue']
share according to Ni Li - set [sequence()]  0.909
share according to Ni Li - set [my_calc_list()]  0.909
ABC 2024 - Transportation - damage
['affected', 'affected', 'affected', 'affected', 'affected', 'affect

In [364]:
# df_performance_avg = df_performamce_doc_ci[df_performamce_doc_ci["ci_sector"] == ci_sector]
# df_performance_avg

## Example output
# 	citation_id	ci_sector	number_cases_pred	number_cases_valid	infrastructure_group	infrastructure_group_list	location	location_list	damage	damage_list
# 0	Koks 2022	Social	15	6	0.333333	0.533	NaN	NaN	NaN	NaN
# 0	Koks 2022	Social	15	6	NaN	NaN	0.5	0.533	NaN	NaN
# 0	Koks 2022	Social	15	6	NaN	NaN	NaN	NaN	0.8	0.867
t = df_performamce_doc_ci[df_performamce_doc_ci["ci_sector"] == "Waste"]
t.groupby("citation_id").first()["number_cases_valid"]

citation_id
Koks 2022    7
Name: number_cases_valid, dtype: int64

In [367]:
## avg performance 
print("Performance scores for each sector averaged across all documents")

for ci_sector in df_performamce_doc_ci["ci_sector"].unique():
    df_performance_avg = df_performamce_doc_ci[df_performamce_doc_ci["ci_sector"] == ci_sector]

    df_performance_avg = {
        "CI sector": ci_sector,
        "CI group": np.round(df_performance_avg["infrastructure_group"].mean(), 3),
        "location": np.round(df_performance_avg["location"].mean(), 3),
        "damage": np.round(df_performance_avg["damage"].mean(), 3),
        "number_cases_pred": sum(df_performance_avg.groupby("citation_id").first()["number_cases_valid"]),
        "number_cases_valid": sum(df_performance_avg.groupby("citation_id").first()["number_cases_pred"]),
    }
    print(df_performance_avg)


print("\nPerformance scores for all sectors in total, averaged across all documents")
df_performance_avg = {
    "CI sector": "All sectors",
    "CI group": np.round(df_performamce_doc_ci["infrastructure_group"].mean(), 3),
    "location": np.round(df_performamce_doc_ci["location"].mean(), 3),
    "damage": np.round(df_performamce_doc_ci["damage"].mean(), 3),
    "number_cases_pred_avg": sum(df_performamce_doc_ci.groupby("citation_id").first()["number_cases_pred"]),
    "number_cases_valid_avg": sum(df_performamce_doc_ci.groupby("citation_id").first()["number_cases_valid"]),
}
print(df_performance_avg)

Performance scores for each sector averaged across all documents
{'CI sector': 'Transportation', 'CI group': np.float64(0.601), 'location': np.float64(0.739), 'damage': np.float64(0.76), 'number_cases_pred': 96, 'number_cases_valid': 111}
{'CI sector': 'Social', 'CI group': np.float64(0.583), 'location': np.float64(0.625), 'damage': np.float64(0.95), 'number_cases_pred': 15, 'number_cases_valid': 18}
{'CI sector': 'Water', 'CI group': np.float64(0.633), 'location': np.float64(0.683), 'damage': np.float64(0.733), 'number_cases_pred': 14, 'number_cases_valid': 12}
{'CI sector': 'Energy', 'CI group': np.float64(0.4), 'location': np.float64(0.393), 'damage': np.float64(0.708), 'number_cases_pred': 9, 'number_cases_valid': 18}
{'CI sector': 'Waste', 'CI group': np.float64(0.333), 'location': np.float64(0.625), 'damage': np.float64(0.667), 'number_cases_pred': 7, 'number_cases_valid': 12}
{'CI sector': 'IT / communication', 'CI group': np.float64(0.5), 'location': np.float64(0.667), 'damage'

In [ ]:
## new eval based on NI LI code (only sequence())


## koks
# Performance scores for each sector averaged across all documents
# {'CI sector': 'Transportation', 'CI group': np.float64(0.4), 'location': np.float64(0.714), 'damage': np.float64(0.333), 'number_cases_pred': 84, 'number_cases_valid': 27}
# {'CI sector': 'Energy', 'CI group': np.float64(0.6), 'location': np.float64(0.571), 'damage': np.float64(0.833), 'number_cases_pred': 42, 'number_cases_valid': 12}
# {'CI sector': 'Waste', 'CI group': np.float64(0.333), 'location': np.float64(0.625), 'damage': np.float64(0.667), 'number_cases_pred': 36, 'number_cases_valid': 21}
# {'CI sector': 'Water', 'CI group': np.float64(0.5), 'location': np.float64(0.667), 'damage': np.float64(0.667), 'number_cases_pred': 18, 'number_cases_valid': 12}
# {'CI sector': 'IT / communication', 'CI group': np.float64(0.5), 'location': np.float64(0.667), 'damage': np.float64(0.6), 'number_cases_pred': 21, 'number_cases_valid': 6}
# {'CI sector': 'Social', 'CI group': np.float64(0.333), 'location': np.float64(0.5), 'damage': np.float64(0.8), 'number_cases_pred': 45, 'number_cases_valid': 18}

# Performance scores for all sectors in total, averaged across all documents
# {'CI sector': 'All sectors', 'CI group': np.float64(0.444), 'location': np.float64(0.624), 'damage': np.float64(0.65), 'number_cases_pred': 246, 'number_cases_valid': 96}


## All docs
# Performance scores for each sector averaged across all documents
# {'CI sector': 'Transportation', 'CI group': np.float64(0.601), 'location': np.float64(0.739), 'damage': np.float64(0.76), 'number_cases_pred': 333, 'number_cases_valid': 288}
# {'CI sector': 'Social', 'CI group': np.float64(0.583), 'location': np.float64(0.625), 'damage': np.float64(0.95), 'number_cases_pred': 54, 'number_cases_valid': 45}
# {'CI sector': 'Water', 'CI group': np.float64(0.633), 'location': np.float64(0.683), 'damage': np.float64(0.733), 'number_cases_pred': 36, 'number_cases_valid': 42}
# {'CI sector': 'Energy', 'CI group': np.float64(0.4), 'location': np.float64(0.393), 'damage': np.float64(0.708), 'number_cases_pred': 54, 'number_cases_valid': 27}
# {'CI sector': 'Waste', 'CI group': np.float64(0.333), 'location': np.float64(0.625), 'damage': np.float64(0.667), 'number_cases_pred': 36, 'number_cases_valid': 21}
# {'CI sector': 'IT / communication', 'CI group': np.float64(0.5), 'location': np.float64(0.667), 'damage': np.float64(0.6), 'number_cases_pred': 21, 'number_cases_valid': 6}

# Performance scores for all sectors in total, averaged across all documents
# {'CI sector': 'All sectors', 'CI group': np.float64(0.561), 'location': np.float64(0.654), 'damage': np.float64(0.766), 'number_cases_pred': 534, 'number_cases_valid': 429}

In [ ]:
## all docs
# Performance scores for each sector averaged across all documents
# Performance scores for each sector averaged across all documents
# {'CI sector': 'Transportation', 'number_cases_pred': 333, 'number_cases_valid': 282, 'CI group': np.float64(0.641), 'location': np.float64(0.603), 'damage': np.float64(0.76)}
# {'CI sector': 'Energy', 'number_cases_pred': 3, 'number_cases_valid': 6, 'CI group': np.float64(1.0), 'location': np.float64(1.0), 'damage': np.float64(1.0)}

# Performance scores for all sectors in total, averaged across all documents
# {'CI sector': 'All sectors', 'number_cases_pred': 336, 'number_cases_valid': 288, 'CI group': np.float64(0.671), 'location': np.float64(0.636), 'damage': np.float64(0.78)}


## for koks
# Performance scores for each sector averaged across all documents
# {'CI sector': 'Transportation', 'CI group': np.float64(0.27), 'location': np.float64(0.622), 'damage': np.float64(0.27), 'number_cases_pred': 84, 'number_cases_valid': 27}
# {'CI sector': 'Energy', 'CI group': np.float64(0.5), 'location': np.float64(0.444), 'damage': np.float64(0.5), 'number_cases_pred': 42, 'number_cases_valid': 12}
# {'CI sector': 'Waste', 'CI group': np.float64(0.368), 'location': np.float64(0.368), 'damage': np.float64(0.526), 'number_cases_pred': 36, 'number_cases_valid': 21}
# {'CI sector': 'Water', 'CI group': np.float64(0.4), 'location': np.float64(0.5), 'damage': np.float64(0.8), 'number_cases_pred': 18, 'number_cases_valid': 12}
# {'CI sector': 'IT / communication', 'CI group': np.float64(0.222), 'location': np.float64(0.222), 'damage': np.float64(0.667), 'number_cases_pred': 21, 'number_cases_valid': 6}
# {'CI sector': 'Social', 'CI group': np.float64(0.333), 'location': np.float64(0.333), 'damage': np.float64(0.81), 'number_cases_pred': 45, 'number_cases_valid': 18}

# Performance scores for all sectors in total, averaged across all documents
# {'CI sector': 'All sectors', 'CI group': np.float64(0.349), 'location': np.float64(0.415), 'damage': np.float64(0.595), 'number_cases_pred': 246, 'number_cases_valid': 96}

In [77]:
### ----------- ALL docs combined   (no calc of score per foc and then avg)  -----

## common valid docs, step2, verifi=true

# Using all Documents
# ['airports', 'airports', 'airports', 'airports', 'airports', 'airports', 'airports', 'aviation', 'bridges', 'bridges', 'bridges', 'bridges', 'bridges', 'bridges', 'education_kita', 'education_school', 'education_school', 'electricity_others', 'electricity_others', 'electricity_others', 'electricity_others', 'electricity_supply', 'electricity_supply', 'electricity_supply', 'gas_distribution', 'gas_distribution', 'healthcare_hospitals_clinics', 'healthcare_hospitals_clinics', 'healthcare_hospitals_clinics', 'healthcare_hospitals_clinics', 'healthcare_hospitals_clinics', 'healthcare_hospitals_clinics', 'healthcare_hospitals_clinics', 'healthcare_others', 'it_telecommunication', 'it_telecommunication', 'motorways', 'motorways', 'motorways', 'motorways', 'motorways', 'motorways', 'motorways', 'motorways', 'nursing', 'nursing', 'nursing', 'nursing', 'ports', 'ports', 'ports', 'rail', 'rail', 'rail', 'rail', 'rail', 'rail', 'rail', 'rail', 'rail', 'rail', 'rail', 'rail', 'rail', 'rail', 'rail_service', 'rail_service', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'waste_others', 'waste_others', 'waste_others', 'waste_others', 'wastewater', 'wastewater', 'wastewater', 'water_supply', 'water_supply', 'water_supply', 'water_supply', 'water_supply', 'water_supply', 'water_supply', 'water_supply', 'waterprotection', 'waterprotection', 'waterprotection', 'waterprotection', 'waterprotection', 'waterprotection']
# ['airports', 'airports', 'airports', 'airports', 'airports', 'airports', 'airports', 'airports', 'airports', 'airports', 'aviation', 'aviation', 'bridges', 'bridges', 'bridges', 'bridges', 'bridges', 'bridges', 'bridges', 'bridges', 'bridges', 'bridges', 'bridges', 'bridges', 'bridges', 'bridges', 'bridges', 'bridges', 'bridges', 'bridges', 'education_kita', 'education_others', 'education_others', 'education_school', 'education_school', 'education_school', 'electricity_others', 'electricity_others', 'electricity_others', 'electricity_others', 'electricity_supply', 'electricity_supply', 'electricity_supply', 'electricity_supply', 'electricity_supply', 'electricity_supply', 'electricity_supply', 'electricity_supply', 'electricity_supply', 'electricity_supply', 'electricity_supply', 'electricity_supply', 'electricity_supply', 'electricity_supply', 'electricity_supply', 'gas_distribution', 'gas_distribution', 'gas_supply', 'gas_supply', 'healthcare_hospitals_clinics', 'healthcare_hospitals_clinics', 'healthcare_hospitals_clinics', 'healthcare_hospitals_clinics', 'healthcare_hospitals_clinics', 'healthcare_hospitals_clinics', 'healthcare_hospitals_clinics', 'healthcare_hospitals_clinics', 'healthcare_hospitals_clinics', 'healthcare_hospitals_clinics', 'healthcare_hospitals_clinics', 'healthcare_others', 'healthcare_others', 'healthcare_others', 'inland_waterways', 'it_telecommunication', 'it_telecommunication', 'it_telecommunication', 'it_telecommunication', 'it_telecommunication', 'it_telecommunication', 'it_telecommunication', 'motorways', 'motorways', 'motorways', 'motorways', 'motorways', 'motorways', 'nursing', 'nursing', 'ports', 'ports', 'rail', 'rail', 'rail', 'rail', 'rail', 'rail', 'rail', 'rail', 'rail', 'rail', 'rail', 'rail', 'rail', 'rail', 'rail', 'rail', 'rail', 'rail_service', 'rail_service', 'rail_service', 'rail_service', 'rail_service', 'rail_service', 'renewable_hydro', 'renewable_hydro', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'transport_others', 'transport_others', 'transport_others', 'transport_others', 'transport_others', 'transport_others', 'transport_others', 'transport_others', 'transport_others', 'transport_others', 'transport_others', 'transport_others', 'transport_others', 'transport_others', 'transport_supply', 'transport_supply', 'waste_others', 'waste_others', 'waste_others', 'waste_others', 'waste_others', 'waste_others', 'waste_others', 'waste_others', 'waste_others', 'wastewater', 'wastewater', 'wastewater', 'wastewater', 'water_others', 'water_others', 'water_supply', 'water_supply', 'water_supply', 'water_supply', 'water_supply', 'water_supply', 'water_supply', 'water_supply', 'water_supply', 'waterprotection', 'waterprotection', 'waterprotection']
# Identical [score 0] 166
# FNs [score 1] 23
# FPs [score 1] 25
# share according to Ni Li - list 0.503
# share according to Ni Li -set  0.571
# ['', '', '', 'Ahr valley', 'Ahr valley', 'Ahr valley', 'Ahr valley', 'Ahr valley', 'Ahr valley', 'Ahrweiler', 'Algemes', 'Alicante', 'Altenahr', 'Alzira', 'Anhalt-Bitterfeld', 'Anhalt-Bitterfeld', 'Avenida Herrera Oria-Virgen de las Flores', 'Avenida Lope de Vega - Atabal', 'Bad Münstereifel', 'Barcelona', 'Bavaria', 'Buol', 'Castilla-La Mancha', 'Catania', 'Catania', 'Catania', 'Chemnitz', 'Chiva', 'Churriana intersection', 'Cártama', 'Elbe', 'Erft', 'Erft region', 'Erzgebirgskreis', 'Erzgebirgskreis', 'Erzgebirgskreis', 'Eschweiler', 'Fischbeck', 'Guadassuar', 'Jerichower Land', 'Julio Cortázar Avenue', 'La Alcudia', 'Leipzig', 'Liége', 'Lope de Vega Avenue', 'Lope de Vega Avenue', 'Maastricht', 'Madrid', 'Madrid', 'Magdeburg', 'Magdeburg', 'Makarska', 'Malaga', 'Malaga', 'Malaga', 'Malaga', 'Malaga', 'Malaga', 'Marušići', 'Mayschoss', 'Milan', 'Milan', 'North Rhine-Westphalia', 'North Rhine-Westphalia', 'North Rhine-Westphalia', 'Oberbayern', 'Omiš', 'Palermo', 'Palermo', 'Palermo', 'Partinico', 'Partinico', 'Pasillo Santa Isabel - Puente de la Aurora', 'Pasillo del Matadero Puente del Carmen', 'Picassent', 'Pisak', 'Polizzi Generosa', 'Polizzi Generosa', 'Raubling', 'Requena', 'Rheingau-Taunus-Kreis', 'Rheingau-Taunus-Kreis', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rosenheim', 'Rosenheim', 'Saale-Holzland', 'Saale-Holzland-Kreis', 'Saale-Holzland-Kreis', 'Sagunto', 'Salzlandkreis', 'Saxony-Anhalt', 'Sinzig', 'Stendal', 'Stendal', 'Stendal', 'Stendal', 'Sueca', 'Thuringia', 'Thuringia', 'Thuringia', 'Traunstein', 'Traunstein', 'Traunstein', 'Trier', 'Utiel', 'Valencia', 'Valencia', 'Valencia', 'Valencia', 'Valencia', 'Valencia', 'Valencia', 'Valencia', 'Valencia', 'Valencia', 'Valencia', 'Valencia area', 'Valencia area', 'Valencia area', 'Valencian Community', 'Vogtlandkreis', 'Vogtlandkreis', 'Wartburg', 'Wittenberg', 'Zwickau', 'all severely affected areas', 'eastern', 'flood region', 'province of Valencia', 'province of Valencia', 'rivers Ahr', 'sanctuary of Madonna del Ponte', 'southern Spain', 'surrounding areas', 'to Madrid']
# ['', '', '', '', '', '', 'Ahr', 'Ahr', 'Ahr', 'Ahr', 'Ahr', 'Ahr', 'Ahr', 'Ahr valley', 'Ahr valley', 'Ahr valley', 'Ahr valley', 'Ahr valley', 'Ahr valley', 'Ahr valley', 'Ahr valley', 'Ahr valley', 'Ahr valley', 'Alps', 'Altenahr', 'Altenahr', 'Altenahr', 'Altenahr', 'Altenburg', 'Altenburg', 'Altenburg', 'Andalusia', 'Bad Münstereifel', 'Bad Münstereifel', 'Bad Münstereifel', 'Bad Münstereifel', 'Bad Münstereifel', 'Bad Münstereifel', 'Barcelona', 'Barcelona', 'Belgian rail network', 'Castellón de la Plana', 'Catania', 'Catania', 'Catania', 'Catania', 'Catania', 'Catania', 'Catania', 'Central Mountains', 'Chaudfontaine', 'Chemnitz-Glauchau', 'Cártama', 'Cártama', 'Cártama', 'Danube', 'Deggendorf', 'Elbe', 'Elbe catchment area', 'Erft', 'Erft', 'Erft', 'Erft', 'Erft', 'Erft', 'Erft', 'Erft region', 'Erzgebirgskreis', 'Erzgebirgskreis', 'Eschweiler', 'Eschweiler', 'Eschweiler', 'Eschweiler', 'Eschweiler', 'Eschweiler', 'Eschweiler', 'Eschweiler', 'Eschweiler', 'Eschweiler', 'Eschweiler', 'Eschweiler', 'Fischbeck', 'German s', 'Guerrero Strachan Avenue', 'Heimersheim', 'Heimersheim', 'Humboldt County', 'Juan XXIII Avenue', 'Juan XXIII Avenue', 'Magdeburg', 'Magdeburg', 'Magdeburg', 'Malaga', 'Malaga', 'Malaga', 'Malaga', 'Malaga', 'Malaga', 'Malaga', 'Malaga', 'Malaga', 'Malaga', 'Malaga', 'Malaga province', 'Marušići', 'Mayschoss', 'Mayschoss', 'Mayschoss', 'Mayschoss', 'Milan', 'Milan', 'National highway', 'North Rhine-Westphalia', 'North Rhine-Westphalia', 'North Rhine-Westphalia', 'Palermo', 'Palermo', 'Palermo', 'Palermo', 'Pepinster', 'Pepinster', 'Pepinster', 'Pisak', 'Polizzi Generosa', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rome', 'Rosenheim', 'Rosenheim', 'Sinzig', 'Sinzig', 'Sinzig', 'Sinzig', 'Spa', 'Spa', 'Spa-Pepinster', 'Spa-Pepinster', 'Traunstein', 'Traunstein', 'Traunstein', 'Traunstein', 'Traunstein', 'Trier', 'Trier', 'Valencia', 'Valencia', 'Valencia', 'Valencia', 'Valencia', 'Valencia', 'Valencia', 'Valencia', 'Valencia', 'Valencia', 'Valencia', 'Valencia', 'Valencia', 'Valencian Community', 'Valencian Community', 'Valencian Community', 'Valencian Community', 'Velázquez Avenue', 'Wallonia', 'Walloon rail network', 'Zermatt', 'of Stendal', 'of Stendal', 'of Stendal', 'of Stendal', 'of Stendal', 'of Stendal', 'of Stendal', 'of Stendal', 'of Stendal', 'of Stendal', 'of Stendal', 'of Stendal', 'province of Valencia', 'sanctuary']
# Identical [score 0] 137
# FNs [score 1] 6
# FPs [score 1] 54
# share according to Ni Li - list 0.59
# share according to Ni Li -set  0.79
# ['Dam failure', 'Dam failure', 'Dam failure', 'affected', 'affected', 'affected', 'affected', 'affected', 'affected', 'affected', 'affected', 'affected', 'affected', 'affected', 'affected', 'affected', 'affected', 'affected', 'affected', 'affected', 'affected', 'affected', 'affected', 'affected', 'affected', 'affected', 'affected', 'affected', 'affected', 'affected', 'affected', 'affected', 'blocked', 'blocked', 'blocked', 'blocked', 'blocked', 'blocked', 'blocked', 'blocked', 'blocked', 'blocked', 'blocked', 'blocked', 'blocked', 'closed', 'closed', 'closed', 'closed', 'closed', 'closure', 'closure', 'closures', 'closures', 'closures', 'closures', 'closures', 'closures', 'closures', 'closures', 'closures', 'closures', 'closures', 'closures', 'closures', 'closures', 'closures', 'closures', 'closures', 'closures', 'contaminated', 'damaged', 'damaged', 'damaged', 'damaged', 'damaged', 'damaged', 'damaged', 'damaged', 'damaged', 'damaged', 'delays', 'derailed', 'derailed', 'destroyed', 'destroyed', 'destroyed', 'dike break', 'dike break', 'dike breaks', 'dike breaks', 'disrupted', 'disrupted', 'disrupted', 'disrupted', 'disrupted', 'disrupted', 'disrupted', 'disrupted', 'disrupted', 'disrupted', 'disrupted', 'disrupted', 'disrupted', 'evacuated', 'evacuated', 'evacuated', 'evacuated', 'evacuated', 'evacuated', 'evacuated', 'flooded', 'flooded', 'flooded', 'increase in patients', 'indirectly affected', 'interrupted', 'interrupted', 'interrupted', 'interrupted', 'largely destroyed', 'largely destroyed', 'largely destroyed', 'nan', 'nan', 'outages', 'outages', 'outages', 'outages', 'polluted', 'power outages', 'power outages', 're-established', 're-established', 'reopened', 'restored', 'severely damaged', 'severely damaged', 'severely damaged', 'severely damaged', 'severely damaged', 'severely damaged', 'severely damaged']
# ['affected', 'blocked', 'blocked', 'blocked', 'blocked', 'blocked', 'blocked', 'blocked', 'blocked', 'breached dike(s)', 'breached dike(s)', 'closed', 'closure', 'closure', 'closure', 'closure(s)', 'closure(s)', 'closure(s)', 'closure(s)', 'closure(s)', 'collapsed', 'dam failure(s)', 'damaged', 'damaged', 'damaged', 'damaged', 'damaged', 'damaged', 'damaged', 'damaged', 'damaged', 'damaged', 'damaged', 'damaged', 'delay(s)', 'derailed', 'derailed', 'derailed', 'derailed', 'derailed', 'destroyed', 'destroyed', 'destroyed', 'destroyed', 'destroyed', 'destroyed', 'destroyed', 'destroyed', 'destroyed', 'destroyed', 'destroyed', 'destroyed', 'destroyed', 'destroyed', 'destroyed', 'destroyed', 'destroyed', 'destroyed', 'destroyed', 'destroyed', 'destroyed', 'destroyed', 'dike break', 'dike break', 'disrupted', 'disrupted', 'disrupted', 'disrupted', 'disrupted', 'disrupted', 'disrupted', 'disrupted', 'disrupted', 'disrupted', 'disrupted high-speed trains', 'disrupted high-speed trains', 'disrupted rail service', 'disrupted rail services', 'disrupted rail services', 'fallen tree(s)', 'fallen trees', 'fallen trees', 'fire damage', 'flight delay(s)', 'flood damage', 'flood damage', 'flood damage', 'flooded', 'flooded', 'flooded', 'flooded', 'flooded', 'flooded', 'flooded', 'flooded', 'flooded', 'flooded', 'flooded', 'flooded', 'flooded', 'flooded', 'flooded', 'flooded', 'flooded', 'flooded', 'flooded', 'flooded', 'flooded', 'flooded', 'flooded', 'flooded', 'flooded', 'flooded', 'flooded', 'flooded', 'flooding', 'impassable', 'irreparably damaged', 'irreparably damaged', 'irreparably damaged', 'irreparably damaged', 'landslide', 'landslide', 'largely destroyed', 'largely destroyed', 'largely destroyed', 'largely destroyed', 'largely destroyed', 'largely destroyed', 'largely destroyed', 'long delays', 'nan', 'nan', 'nan', 'outages', 'outages', 'outages', 'outages', 'outages', 'outages', 'outages', 'outages', 'outages', 'outages', 'outages', 'partially destroyed', 'polluted', 'polluted', 'polluted', 'polluted', 'polluted', 'polluted', 're-established', 'rebuilt', 'road erosion', 'road erosion', 'seriously damaged', 'severely damaged', 'severely damaged', 'severely damaged', 'severely damaged', 'severely damaged', 'severely damaged', 'severely damaged', 'severely damaged', 'severely damaged', 'severely damaged', 'severely damaged', 'severely damaged', 'severely damaged', 'severely damaged', 'severely damaged', 'severely damaged', 'severely damaged', 'severely damaged', 'severely damaged', 'severely damaged', 'severely damaged', 'severely disrupted', 'traffic congestion', 'traffic congestion', 'traffic disruption', 'traffic disruption', 'traffic disruption(s)', 'traffic jam', 'traffic jam', 'traffic jam(s)', 'traffic jam(s)', 'train failure', 'train failure', 'train failures']
# Identical [score 0] 141
# FNs [score 1] 2
# FPs [score 1] 50
# share according to Ni Li - list 0.578
# share according to Ni Li -set  0.784


## only koks , step2, vierif=true
## Benchmark against Ni Li Wiki-impacts - Loc calculation
# Using all Documents
# ['bridges', 'bridges', 'bridges', 'bridges', 'bridges', 'bridges', 'education_kita', 'education_school', 'electricity_others', 'electricity_others', 'gas_distribution', 'gas_distribution', 'healthcare_hospitals_clinics', 'healthcare_hospitals_clinics', 'healthcare_hospitals_clinics', 'healthcare_others', 'it_telecommunication', 'it_telecommunication', 'rail', 'rail', 'roads', 'waste_others', 'waste_others', 'waste_others', 'waste_others', 'wastewater', 'wastewater', 'wastewater', 'water_supply', 'water_supply', 'water_supply', 'water_supply']
# ['bridges', 'bridges', 'bridges', 'bridges', 'bridges', 'bridges', 'bridges', 'bridges', 'bridges', 'bridges', 'bridges', 'bridges', 'bridges', 'bridges', 'bridges', 'bridges', 'bridges', 'bridges', 'education_kita', 'education_school', 'electricity_others', 'electricity_others', 'electricity_others', 'electricity_others', 'electricity_supply', 'electricity_supply', 'electricity_supply', 'electricity_supply', 'gas_distribution', 'gas_distribution', 'gas_supply', 'gas_supply', 'healthcare_hospitals_clinics', 'healthcare_hospitals_clinics', 'healthcare_hospitals_clinics', 'healthcare_hospitals_clinics', 'healthcare_hospitals_clinics', 'healthcare_hospitals_clinics', 'healthcare_hospitals_clinics', 'healthcare_hospitals_clinics', 'healthcare_hospitals_clinics', 'healthcare_hospitals_clinics', 'healthcare_others', 'healthcare_others', 'healthcare_others', 'it_telecommunication', 'it_telecommunication', 'it_telecommunication', 'it_telecommunication', 'it_telecommunication', 'it_telecommunication', 'it_telecommunication', 'motorways', 'rail', 'rail', 'rail', 'rail', 'rail', 'rail', 'rail', 'rail', 'roads', 'roads', 'roads', 'roads', 'roads', 'roads', 'waste_others', 'waste_others', 'waste_others', 'waste_others', 'waste_others', 'waste_others', 'waste_others', 'waste_others', 'waste_others', 'wastewater', 'wastewater', 'wastewater', 'wastewater', 'water_supply', 'water_supply', 'water_supply', 'water_supply', 'water_supply', 'water_supply', 'water_supply']
# Identical [score 0] 80
# FNs [score 1] 48
# FPs [score 1] 7
# share according to Ni Li - list 0.328
# share according to Ni Li -set  0.552
# ['Ahr valley', 'Ahr valley', 'Ahr valley', 'Ahr valley', 'Ahr valley', 'Ahr valley', 'Ahrweiler', 'Altenahr', 'Bad Münstereifel', 'Erft', 'Erft region', 'Eschweiler', 'Liége', 'Maastricht', 'Mayschoss', 'North Rhine-Westphalia', 'North Rhine-Westphalia', 'North Rhine-Westphalia', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Sinzig', 'Trier', 'all severely affected areas', 'flood region', 'rivers Ahr']
# ['Ahr', 'Ahr', 'Ahr', 'Ahr', 'Ahr', 'Ahr', 'Ahr', 'Ahr valley', 'Ahr valley', 'Ahr valley', 'Ahr valley', 'Ahr valley', 'Ahr valley', 'Ahr valley', 'Ahr valley', 'Ahr valley', 'Ahr valley', 'Altenahr', 'Altenahr', 'Altenahr', 'Altenahr', 'Altenburg', 'Altenburg', 'Altenburg', 'Bad Münstereifel', 'Bad Münstereifel', 'Bad Münstereifel', 'Bad Münstereifel', 'Bad Münstereifel', 'Bad Münstereifel', 'Chaudfontaine', 'Erft', 'Erft', 'Erft', 'Erft', 'Erft', 'Erft', 'Erft', 'Erft region', 'Eschweiler', 'Eschweiler', 'Eschweiler', 'Eschweiler', 'Eschweiler', 'Eschweiler', 'Eschweiler', 'Eschweiler', 'Eschweiler', 'Eschweiler', 'Eschweiler', 'Eschweiler', 'Heimersheim', 'Heimersheim', 'Mayschoss', 'Mayschoss', 'Mayschoss', 'Mayschoss', 'North Rhine-Westphalia', 'North Rhine-Westphalia', 'North Rhine-Westphalia', 'Pepinster', 'Pepinster', 'Pepinster', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Sinzig', 'Sinzig', 'Sinzig', 'Sinzig', 'Spa', 'Spa', 'Spa-Pepinster', 'Spa-Pepinster', 'Trier', 'Trier']
# Identical [score 0] 67
# FNs [score 1] 35
# FPs [score 1] 20
# share according to Ni Li - list 0.437
# share according to Ni Li -set  0.686
# ['affected', 'affected', 'affected', 'closed', 'closed', 'contaminated', 'damaged', 'damaged', 'damaged', 'damaged', 'damaged', 'damaged', 'destroyed', 'destroyed', 'destroyed', 'disrupted', 'largely destroyed', 'largely destroyed', 'largely destroyed', 'nan', 'nan', 'polluted', 're-established', 're-established', 'restored', 'severely damaged', 'severely damaged', 'severely damaged', 'severely damaged', 'severely damaged', 'severely damaged', 'severely damaged']
# ['collapsed', 'damaged', 'damaged', 'damaged', 'damaged', 'damaged', 'damaged', 'damaged', 'damaged', 'destroyed', 'destroyed', 'destroyed', 'destroyed', 'destroyed', 'destroyed', 'destroyed', 'destroyed', 'destroyed', 'destroyed', 'destroyed', 'destroyed', 'destroyed', 'destroyed', 'destroyed', 'destroyed', 'destroyed', 'destroyed', 'destroyed', 'destroyed', 'destroyed', 'destroyed', 'disrupted', 'disrupted', 'disrupted', 'disrupted', 'disrupted', 'flooded', 'flooded', 'flooded', 'flooded', 'flooded', 'flooded', 'flooded', 'flooded', 'flooded', 'irreparably damaged', 'irreparably damaged', 'irreparably damaged', 'irreparably damaged', 'largely destroyed', 'largely destroyed', 'largely destroyed', 'largely destroyed', 'largely destroyed', 'largely destroyed', 'largely destroyed', 'outages', 'outages', 'outages', 'polluted', 'polluted', 'polluted', 'polluted', 'polluted', 'polluted', 're-established', 'rebuilt', 'road erosion', 'road erosion', 'severely damaged', 'severely damaged', 'severely damaged', 'severely damaged', 'severely damaged', 'severely damaged', 'severely damaged', 'severely damaged', 'severely damaged', 'severely damaged', 'severely damaged', 'severely damaged', 'severely damaged', 'severely damaged', 'severely damaged', 'severely damaged', 'severely damaged', 'severely disrupted']
# Identical [score 0] 66
# FNs [score 1] 34
# FPs [score 1] 21
# share according to Ni Li - list 0.445
# share according to Ni Li -set  0.731

## spatial mapping

In [78]:
df_pred_geolocalized[["location", "location_osm", "coords", "coord_potential_locations", "chunk_text"]][:20  ]


KeyError: "['location_osm', 'coords'] not in index"

In [ ]:
# test duckdb

import requests
import json
from geopy.geocoders import Nominatim
import duckdb
import pandas as pd
import geopandas as gpd 
from glob import glob

## install osmium and spaita lextension once
duckdb.sql("""
    INSTALL spatial FROM core; 
    INSTALL osmium FROM community;
    LOAD spatial;
    LOAD osmium;
  """)
duckdb.close()  # NOTE always close the pointer! 

# # Set configurations on jupysql to directly output data to Pandas and to simplify the output that is printed to the notebook.
# %config SqlMagic.autopandas = True
# %config SqlMagic.feedback = False
# %config SqlMagic.displaycon = False


# # Import jupysql Jupyter extension to create SQL cells
# %load_ext sql


### SAFETY
# geom_type_1 = 'line'
# geom_type_2 = 'way'
# loc_name = 'A 3'
# loc_type = 'highway'
# FILEPATH_OSM_PBF = 'rheinland-pfalz-260531.osm.pbf'  # kenrle crashes as too much cache needed for "europe-latest.osm.pbf"

# duckdb.close() 
# r = duckdb.sql(f"""
#     LOAD osmium; LOAD spatial;
#     COPY(
#         SELECT id, tags['ref'] AS ref, json_value(tags, '$[0]') AS loc_type, tags, geometry, '{FILEPATH_OSM_PBF}' AS source
#         FROM '{FILEPATH_OSM_PBF}'
#         WHERE kind IN ('{geom_type_1}', '{geom_type_2}')
#             AND tags['ref'] = '{loc_name}'
#             AND tags['{loc_type}'] = 'motorway'
#     ) TO  'locations_geocoded' (FORMAT GDAL, DRIVER GeoJSON, OVERWRITE_OR_IGNORE, PARTITION_BY (loc_type), FILENAME_PATTERN '{loc_name}')
#     ;
# """)#.df()
# #             AND loc_type IN ('{loc_type}')

# #         SELECT id, tags['ref'] AS ref, json_extract(tags, '$.{loc_type}') AS loc_type, tags, geometry, '{FILEPATH_OSM_PBF}' AS source
# #         SELECT id, tags['ref'] AS ref, json_extract(tags, '$.{loc_type}') AS loc_type, tags, geometry, '{FILEPATH_OSM_PBF}' AS source

# #             AND loc_type IN ('{loc_type}')
# #             AND tags['{loc_type}'] = 'motorway'

# duckdb.close() # 

In [ ]:


for doc in df_pred_geolocalized.citation_id.unique():
    

    #FILEPATH_OSM_PBF = "europe-latest.osm.pbf" 
    FILEPATH_OSM_PBF = s.PATH_DATA + 'europe-latest.osm.pbf'

    ## TESTING 
    if doc not in  ["Koks 2022"]:# , "Khazai 2013"]:
        continue

    print(f"\nDocument: {doc}")

    df_single_doc = df_pred_geolocalized[df_pred_geolocalized["citation_id"] == doc]
    

    ## load location names (based on OSM), when existent, and get their geoms
    for loc_name in df_single_doc["location_osm"].dropna().unique():

        print(loc_name)
    
    # for loc_name, loc_type in zip(df_pred_geolocalized["location"], df_pred_geolocalized["addresstype"]):

        # loc_name = #"A 61" # df_pred_geolocalized["location"].iloc[0]

        # if loc_name not in ["Bad Münstereifel", "Sinzig", "Altenahr", "Eschweiler", "Ahrweiler district"]:  
        #     continue 
        try:
            geom_type_1 = 'relation'
            geom_type_2 = 'area'
            name = "name"
            ref_type = 'boundary'
            ref_value = "administrative" 
            # loc_value = xxx  # "Bad Münstereifel", "Eschweiler", "Ahrweiler district"

            # if loc_type in ["city", "town", "village", "region", "state", "county"]:
            #     geom_type = 'relation'         
            
            # extract Ci locations from PBF file
            # NOTE: explicit LOAD osmium and spatial for GDAL format
            r = duckdb.sql(f"""
                COPY(
                    SELECT id, tags['ref'] AS ref, json_value(tags, '$[0]') AS loc_type, tags, geometry, '{FILEPATH_OSM_PBF}' AS source
                    FROM '{FILEPATH_OSM_PBF}'
                    WHERE kind IN ('{geom_type_1}', '{geom_type_2}')
                        AND tags['{name}'] = '{loc_name}'
                        AND tags['{ref_type}'] = '{ref_value}'
                ) TO  'locations_geocoded' (FORMAT GDAL, DRIVER GeoJSON, OVERWRITE_OR_IGNORE, PARTITION_BY (loc_type), FILENAME_PATTERN '{loc_name}')
                ;
            """)# .df()
            duckdb.close()  # NOTE always close the pointer! 
    
        except Exception as e:

            ## try without district prefix/postfix, e.g. "Ahrweiler district" --> "Ahrweiler"
            if "district" in loc_name:
                loc_name = loc_name.replace("district", "").strip()
                print(f"Trying again with modified location name: {loc_name}")
                try:
                    r = duckdb.sql(f"""
                        COPY(
                            SELECT id, tags['ref'] AS ref, json_value(tags, '$[0]') AS loc_type, tags, geometry, '{FILEPATH_OSM_PBF}' AS source
                            FROM '{FILEPATH_OSM_PBF}'
                            WHERE kind IN ('{geom_type_1}', '{geom_type_2}')
                                AND tags['{name}'] = '{loc_name}'
                                AND tags['{ref_type}'] = '{ref_value}'
                        ) TO  'locations_geocoded' (FORMAT GDAL, DRIVER GeoJSON, OVERWRITE_OR_IGNORE, PARTITION_BY (loc_type), FILENAME_PATTERN '{loc_name}')
                        ;
                    """)
                    duckdb.close() 

                    ## update loc_name in DF
                    df_pred_geolocalized.loc[df_pred_geolocalized["location"] == loc_name, "location_osm"] = loc_name

                except Exception as e:
                    print(f"Error occurred while processing location **{loc_name}**: {e}")
                    duckdb.close() 
                    continue
            pass

        try: 
            # highways
            # if ref_type == 'highway' or ref_type == 'river':    
            geom_type_1 = 'way'
            geom_type_2 = 'line'
            name = "ref"
            ref_type = 'highway'
            ref_value = "motorway"
        
            duckdb.sql(f"""
                COPY(
                    SELECT id, tags['ref'] AS ref, json_value(tags, '$[0]') AS loc_type, tags, geometry, '{FILEPATH_OSM_PBF}' AS source
                    FROM '{FILEPATH_OSM_PBF}'
                    WHERE kind IN ('{geom_type_1}', '{geom_type_2}')
                        AND tags['{name}'] = '{loc_name}'
                        AND tags['{ref_type}'] = '{ref_value}'
                ) TO  'locations_geocoded' (FORMAT GDAL, DRIVER GeoJSON, OVERWRITE_OR_IGNORE, PARTITION_BY (loc_type), FILENAME_PATTERN '{loc_name}')
                ;
            """)
            duckdb.close()
                    
        except Exception as e:
            pass

        try:
            # natural
            geom_type_1 = 'way'
            geom_type_2 = 'relation'
            name = "name"
            ref_type_1 = 'natural'
            ref_value_1 = "valley"
            ref_type_2 = "waterway"
            ref_value_2 = "river"
        
            duckdb.sql(f"""
                LOAD osmium; LOAD spatial;
                COPY(
                    SELECT id, tags['ref'] AS ref, json_value(tags, '$[0]') AS loc_type, tags, geometry, '{FILEPATH_OSM_PBF}' AS source
                    FROM '{FILEPATH_OSM_PBF}'
                    WHERE kind IN ('{geom_type_1}', '{geom_type_2}')
                        AND tags['{name}'] = '{loc_name}'
                        AND (tags['{ref_type_1}'] = '{ref_value_1}' 
                        OR tags['{ref_type_2}'] = '{ref_value_2}')

                ) TO  'locations_geocoded' (FORMAT GDAL, DRIVER GeoJSON, OVERWRITE_OR_IGNORE, PARTITION_BY (loc_type), FILENAME_PATTERN '{loc_name}')
                ;
            """)
            duckdb.close()

        except Exception as e:
            print(f"Error occurred while processing location **{loc_name}**: {e}")
            duckdb.close() 
            continue

    # ) TO  'output.csv' (FORMAT CSV, HEADER, DELIMITER ',')

        time.sleep(5)  # to avoid overloading the system, adjust as needed

# SELECT id, tags['ref'] AS ref, tags, geometry, '{FILEPATH_OSM_PBF}' AS source



TypeError: unsupported operand type(s) for +: 'PosixPath' and 'str'

In [ ]:
# ## TEST nominatim API
# ## nominatim (run time: 2min )

# import requests

# url = "https://nominatim.openstreetmap.org/reverse"
# params = {
#     "format": "jsonv2",
#     # "name": "Valencia",
#     # "name": "Valencian Community ",
#     #"addresstype": "city",
#     # test palermo
#     "lat": 38.2608690,
#     "lon": 15.5679169
# }

# response = requests.get(url, params=params, headers={"User-Agent": "my_location_app"})
# data = response.json()

# print(data.get("display_name"))

# # # print(data.get("display_name"))

# # {'place_id': 54077774,
# #  'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright',
# #  'osm_type': 'way',
# #  'osm_id': 524472554,
# #  'lat': '38.2609505',
# #  'lon': '15.5679932',
# #  'category': 'highway',
# #  'type': 'unclassified',
# #  'place_rank': 26,
# #  'importance': 0.053404183260852944,
# #  'addresstype': 'road',
# #  'name': 'Via dei Bianchi',
# #  'display_name': 'Via dei Bianchi, Pace, VI Circoscrizione, Curcuraci, Messina, Sicilia, 98167, Italia',
# #  'address': {'road': 'Via dei Bianchi',
# #   'suburb': 'Pace',
# #   'village': 'Curcuraci',
# #   'city': 'Messina',
# #   'county': 'Messina',
# #   'ISO3166-2-lvl6': 'IT-ME',
# #   'state': 'Sicilia',
# #   'ISO3166-2-lvl4': 'IT-82',
# #   'postcode': '98167',
# #   'country': 'Italia',
# #   'country_code': 'it'},
# #  'boundingbox': ['38.2576647', '38.2610009', '15.5679393', '15.5744505']}
# # data

In [ ]:
import matplotlib.pyplot as plt

# Collect coords into list
coords = []
for element in data['elements']:
  if element['type'] == 'node':
    lon = element['lon']
    lat = element['lat']
    coords.append((lon, lat))
  elif 'center' in element:
    lon = element['center']['lon']
    lat = element['center']['lat']
    coords.append((lon, lat))
# Convert coordinates into numpy array
X = np.array(coords)
plt.plot(X[:, 0], X[:, 1], 'o')
plt.title('Locations from overpass')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.axis('equal')
plt.show()

In [ ]:
df_locs.plot(cmap="viridis", figsize=(10, 10))
#.geometry.line_merge()

In [ ]:
df_locs[df_locs.ref == "A 3"].plot()


In [ ]:
df_locs.iloc[0]#.tags[0]

## Merge prediction entries with potential validation entries (nth:1 pairs)

In [ ]:
import re


print("Match chunk text of each prediction entry with related validation entries (nth:1 pairs)")
print("Align texts from valid and pred set by removing potential whitespaces")
# NOTE Solves issue: of having doubled whitespace or whitespaces due to linebreaks. eg. "Bad Münstereifel" where valid senence differed to pred_chunk due to "- " (instead of "-") in fresh-water

df_pred_valid_all = pd.DataFrame()
threshold = 65  # keep low due to differences in the tranlsation and when linebreaks where used

# find for each prediction entry all validation entries for respective chunk 
# these validation entries are candidates from which the most similar one to the pred. entry is taken to calc. model performance 
# including also entries where pred_info or valid_info is missing (e.g FNs, FPs)
for _, pred_entry in df_pred.iterrows():
    for _, valid_entry in df_valid.iterrows():  # all validation entries of all docs

        if valid_entry.sentence_reference is np.nan:
            continue

        valid_entry.sentence_reference = re.sub(r"([^\s-])\n([^\s-])", r"\1 \2", valid_entry.sentence_reference) # replace linebreak symbols when they occur just once, with whitespace (two linebreaks - probably new subsection)
        valid_entry.sentence_reference = valid_entry.sentence_reference.replace("/\n{2,}/g", "\n")  # remove linebreaks only when they occurred just once, but not for multiple linebreaks (e.g. before subsection)
        valid_entry.sentence_reference = re.sub(r"\s+", " ", valid_entry.sentence_reference)  # replace >1 whitespaces with single whitespace
        valid_entry.sentence_reference = re.sub(r"([^\s-])- ([^\s-])", r"\1-\2", valid_entry.sentence_reference)  # remove hypens in the middle of lines
        
        pred_entry.chunk_text = re.sub(r"([^\s-])\n([^\s-])", r"\1 \2", pred_entry.chunk_text) # replace linebreak symbols when they occur just once, with whitespace (two linebreaks - probably new subsection)
        pred_entry.chunk_text = pred_entry.chunk_text.replace("/\n{2,}/g", "\n")  # remove linebreaks only when they occurred just once, but not for multiple linebreaks (e.g. before subsection)
        pred_entry.chunk_text = re.sub(r"\s+", " ", pred_entry.chunk_text)  # replace >1 whitespaces with single whitespace
        pred_entry.chunk_text = re.sub(r"([^\s-])- ([^\s-])", r"\1-\2", pred_entry.chunk_text)  # remove hypens in the middle of lines

        # Calculate match score by accounting for partial string matches. 
        # In detail, it calculates the similarity ratio using the shortest string (length n, here: "sentence_reference") against all n-length substrings of the larger string and returns the highest score 
        score = fuzz.partial_ratio(valid_entry['sentence_reference'].replace(" ", ""), pred_entry['chunk_text'].replace(" ", ""))



        if score >= threshold:
            entry_pred_valid = {
                "citation_id": pred_entry["citation_id"],
                "ci_pred": pred_entry["infrastructure_type"],
                "ci_group_pred": pred_entry["infrastructure_group"],
                "damage_pred": pred_entry["damage"],
                "location_pred": pred_entry["location"],
                #"coords_pred": pred_entry["coords"],
                "chunk_id_pred": pred_entry["chunk_id"],
                "chunk_text_pred": pred_entry["chunk_text"],
                "ci_valid": valid_entry["ci1_type"],
                "ci_group_valid": valid_entry["ci1_group"],
                "damage_valid": valid_entry["ci1_damage"],
                "location_valid": valid_entry["ci1_location"],
                # "coords_valid": valid_entry["coords"],
                "sentence_text_valid": valid_entry["sentence_reference"],
                "text_similarity": score,
                "id_pred": pred_entry["id_pred"],
                "id_valid": valid_entry["id_valid"]
            }
            df_pred_valid_all = pd.concat([df_pred_valid_all, pd.DataFrame([entry_pred_valid])], ignore_index=True)  # n:1 relationship DF
        
print(len(df_pred_valid_all))

# 85 threshold - 778 entries
# 75 threshold - 778 entries
# 75 threshold + CI subgrou - 513 entries




Match chunk text of each prediction entry with related validation entries (nth:1 pairs)
Align texts from valid and pred set by removing potential whitespaces
192


In [ ]:
df_pred_valid_all.info() # 143 -190 entries  # 656 entries (bei simil 65)

df_pred_valid_all[df_pred_valid_all.text_similarity < 100][["chunk_text_pred", "sentence_text_valid", "text_similarity"]].sort_values(by="text_similarity", ascending=True)[:5]

## --> FPs are more common compared to FNs, especially for predicting locations, 
# as it is easier to get a prep-valid match when pred.info is actually missing due to larger chunk-text (pred set) compared to sentence-text (valid set)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 192 entries, 0 to 191
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   citation_id          192 non-null    object
 1   ci_pred              192 non-null    object
 2   ci_group_pred        192 non-null    object
 3   damage_pred          192 non-null    object
 4   location_pred        192 non-null    object
 5   chunk_id_pred        192 non-null    int64 
 6   chunk_text_pred      192 non-null    object
 7   ci_valid             192 non-null    object
 8   ci_group_valid       192 non-null    object
 9   damage_valid         192 non-null    object
 10  location_valid       192 non-null    object
 11  sentence_text_valid  192 non-null    object
 12  text_similarity      192 non-null    int64 
 13  id_pred              192 non-null    int64 
 14  id_valid             192 non-null    int64 
dtypes: int64(4), object(11)
memory usage: 22.6+ KB


,chunk_text_pred,sentence_text_valid,text_similarity
159,"In Germany, an estimated 180 general-practitioner practices have been affected by the flood event. Impacts range from completely destroyed to unable to operate due to a lack of running water and electricity (Ärzte Zeitung, 2021). After 1.5 months, medical care was guaranteed again in the most affected regions in Rhineland-Palatinate (Hochwasser Ahr, 2021c). In the state of North Rhine-Westphalia, approximately 68 hospitals have been affected, of which several have been affected severely and will take at least 1.5 years to be rebuilt (Fig. 2). Direct damages are estimated to be at least EUR 100 million to repair all medical facilities (Korzilius, 2021). In the town of Eschweiler (Germany), for example, the basement of the hospital was flooded, as well as the outbuildings and the entire outdoor area. The power supply collapsed, the entire building technology was destroyed and some 300 patients had to be evacuated by helicopter. Property damage is expected to be around EUR 50 million.","In the town of Eschweiler (Germany), for ex-ample, the basement of the hospital was ﬂooded, as well as the outbuildings and the entire outdoor area. The power sup-ply collapsed, the entire building technology was destroyed and some 300 patients had to be evacuated by helicopter. Property damage is expected to be around EUR 50 million.",87
145,"In Germany, an estimated 180 general-practitioner practices have been affected by the flood event. Impacts range from completely destroyed to unable to operate due to a lack of running water and electricity (Ärzte Zeitung, 2021). After 1.5 months, medical care was guaranteed again in the most affected regions in Rhineland-Palatinate (Hochwasser Ahr, 2021c). In the state of North Rhine-Westphalia, approximately 68 hospitals have been affected, of which several have been affected severely and will take at least 1.5 years to be rebuilt (Fig. 2). Direct damages are estimated to be at least EUR 100 million to repair all medical facilities (Korzilius, 2021). In the town of Eschweiler (Germany), for example, the basement of the hospital was flooded, as well as the outbuildings and the entire outdoor area. The power supply collapsed, the entire building technology was destroyed and some 300 patients had to be evacuated by helicopter. Property damage is expected to be around EUR 50 million.","In the town of Eschweiler (Germany), for ex-ample, the basement of the hospital was ﬂooded, as well as the outbuildings and the entire outdoor area. The power sup-ply collapsed, the entire building technology was destroyed and some 300 patients had to be evacuated by helicopter. Property damage is expected to be around EUR 50 million.",87
149,"In Germany, an estimated 180 general-practitioner practices have been affected by the flood event. Impacts range from completely destroyed to unable to operate due to a lack of running water and electricity (Ärzte Zeitung, 2021). After 1.5 months, medical care was guaranteed again in the most affected regions in Rhineland-Palatinate (Hochwasser Ahr, 2021c). In the state of North Rhine-Westphalia, approximately 68 hospitals have been affected, of which several have been affected severely and will take at least 1.5 years to be rebuilt (Fig. 2). Direct damages are estimated to be at least EUR 100 million to repair all medical facilities (Korzilius, 2021). In the town of Eschweiler (Germany), for example, the basement of the hospital was flooded, as well as the outbuildings and the entire outdoor area. The power supply collapsed, the entire building technology was destroyed and some 300 patients had to be evacuated by helicopter. Property damage is expected to be around EUR 50 million.","In the town of Eschweiler (Germany), for ex-ample, the basement of the hospital was ﬂooded, as well as the outbuildings and the entire outdoor area. The power sup-ply collapsed, the entire building technology was destroyed and some 300 patients had to b

In [ ]:
## entries with lowest similarity
df_pred_valid_all.text_similarity.describe() # 413  (no regex, step2) , 284 (no c regex, step1)
# df_pred_valid_all.iloc[df_pred_valid_all.text_similarity.sort_values(ascending=True).index] [["sentence_text_valid", "chunk_text_pred","text_similarity"]]

count    192.000000
mean      98.661458
std        3.057045
min       87.000000
25%       99.000000
50%       99.000000
75%      100.000000
max      100.000000
Name: text_similarity, dtype: float64

## Calc similarities 
* TPs (for all cases where text info in pred and valid set exists)
* FNs  (model missed actual cases)
* FPs  (model hallucinated cases)

In [ ]:
df_pred_valid_all = df_pred_valid_all[~df_pred_valid_all.citation_id.isin(["Chamra 2006", "Hladny 2004", "Pescaroli 2017"])]


In [ ]:
list_entity_valid = ["ci_group_valid", "damage_valid", "location_valid"]
list_entity_pred = ["ci_group_pred", "damage_pred", "location_pred"]



#  Set similarity threshold (self-defined) when CI case is valid or not FN/FP
cos_smlrty_thresh = 0.7
norm_pr_smlrty_thresh = 0.7


print(" --- For each unique valid case (unique combi: [ci_valid, damage_valid, location_valid, sentence_text]) calculate similarity ---")
print("Using 100% match for CI types based on subgroups")
print("Using cosine similarity threshold for damages", cos_smlrty_thresh)
print("Using normalized partial ratio similarity threshold for locations", norm_pr_smlrty_thresh)

## AIM of evaluation loop below: 
# remove all cases in df_pred_valid_all where pred_entities were wrongly assigned to a valid_entity
## ie keep only pre-valid pairs with highest similarity per unique valid case

## call embedding model for semantic similarity calculation
embedding_model = u.EmbeddingModel()


# store evaluation results
df_eval_records = pd.DataFrame()


# For each impact case (rows) 
for record_no, impact_record in df_pred_valid_all.iterrows():

    print(f"Record: {record_no } / {len(df_pred_valid_all)}")

    # init dict to store results for each records (row=)
    df_eval = {
        "citation": impact_record.citation_id,
        "chunk_text_pred": impact_record.chunk_text_pred,
        "sentence_text_valid": impact_record.sentence_text_valid,
        "id_pred": impact_record.id_pred,
        "id_valid": impact_record.id_valid
    }
    
    # iterate over the three entity classes (ci, damage, location) to assess LLM performance
    for entity_valid, entity_pred in zip(list_entity_valid, list_entity_pred):

        # Calculate similarities for entries in column pair: entity_pred - entity_valid

        ## calc similarity when both valid_info exist (not NAN) 
        # NOTE df_pred model can put out NAN when case exists but it couldnt find suitable value (e.g. damage_pred="NaN", damage_valid="polluted")
        if impact_record[entity_valid] is not np.nan:
        # if impact_record[entity_pred] and impact_record[entity_valid] is not np.nan:
            pred_impact = impact_record[entity_pred]
            valid_impact = impact_record[entity_valid]

            if entity_pred == "ci_group_pred": # for CI group, only partial ratio similarity is calculated as it is more important to get the correct group than the exact match (e.g. "port infrastructure" <-> "port")
                
                # similarity on idential match 
                if pred_impact == valid_impact:
                    ci_smlrty = 1
                else:
                    ci_smlrty = 0

                # store result for ci entity 
                df_eval["ci_pred"] = pred_impact  # CI subgroup
                df_eval["ci_valid"] = valid_impact # CI subgroup
                df_eval["ci_smlrty"] = ci_smlrty

            if entity_pred == "damage_pred": 

                if isinstance(pred_impact, str) & isinstance(valid_impact, str): # check that pred or valid are not NAN
                    # contextual vectors (transformer-based)
                    embedded_list = embedding_model.vector_calculation(pred_impact, valid_impact)
                    # calculate cosine similarity for each pred-valid damage pair
                    similarity_score_cos = embedding_model.cosine_similarity(embedded_list[0], embedded_list[1])  # 0-1 value, the higher the more similar
                
                # if nan -> then it is either FP or FN
                elif isinstance(pred_impact, float) | isinstance(np.nan, float):
                    similarity_score_cos = 0.0     

                # store result for DAM and LOC entity 
                df_eval["dam_pred"] = pred_impact  
                df_eval["dam_valid"] = valid_impact 
                df_eval["dam_smlrty"] = similarity_score_cos


            if entity_pred == "location_pred": 
                ## Cosine similarity calc.
                # if isinstance(pred_impact, str):
                #     # contextual vectors (transformer-based)
                #     embedded_list = embedding_model.vector_calculation(pred_impact, valid_impact)
                #     # calculate cosine similarity for each pred-valid pair
                #     similarity_score_cos = embedding_model.cosine_similarity(embedded_list[0], embedded_list[1])  # 0-1 value, the higher the more similar
                # elif np.isnan(pred_impact):
                #     similarity_score_cos = 0.0   # NOTE it is FNs
                ## Partial ratio similarity calc. (especially for locations and CI-type  "port infrastructure" <-> "port")
                similarity_score_pr = fuzz.partial_ratio(pred_impact, valid_impact)  

                # store result for DAM and LOC entity 
                df_eval["loc_pred"] = pred_impact  
                df_eval["loc_valid"] = valid_impact 
                # df_eval["loc_smlrty"] = similarity_score_cos
                df_eval["loc_smlrty_norm_pr"] = similarity_score_pr / 100

    # collect all single records (row) with similarity scores
    df_eval_records = pd.concat([df_eval_records, pd.DataFrame([df_eval])], ignore_index=True)


## NOTE Description: How 1:1 pairs for pred-valid are extracted f
## 1. group by single records from df_valid (via id_valid indices), 
##    Column "id_valid": index represents single records from df_valid (when validation_sentence contains 2 cases: -> id-valid:0, id_valid:1,  sentence w 1 case: id-valid:2) 
## 2. then collect from each group the one with highest similarity to predictions
##    --> binary "mask" indicates where we have matches -e.g. correct predictions (true: TP, false: FN or FP)  is our match (1:1 pred-valid pair) - from which TPs can be calculated

## 1. + 2.
# select for each single valid record (ie rows in df_valid) the 1:1 match (pred-valid pair, "head(1)") with highest similarities across all three classes 
# NOTE need to sort based on all three smlrty cols to do correct Tp calc 
#      (if sort_values by on similartiy column would result in too many TPs- as then 1:1 pairs would contain also random matches where randomly CI_red is identical with CI_valid)
df_smltry_selmax = df_eval_records.groupby("id_valid").apply(lambda s: s.sort_values(["ci_smlrty", "dam_smlrty","loc_smlrty_norm_pr"], ascending=False).head(1))
# # OLD  (makes too many 1:1 pairs as described in NOTE)
# mask = df_eval_records.groupby("id_valid").apply(lambda x: x==x["ci_smlrty"].max()).droplevel(0)
# df_smltry_selmax2 = df_eval_records.where(mask.ci_smlrty==mask.ci_smlrty.max()).dropna(how="all") # keep cases only which have highest similarity scores
# df_smltry_selmax2.reset_index(drop=True, inplace=True)


print("for each unique valid record keep only pred-valid pairs of highest similarity")


# iterate over the three entity classes (ci, damage, location) to assess LLM performance
for _, column_pred in zip(list_entity_valid, list_entity_pred):

    if column_pred == "ci_group_pred":

        # remove cases where no CI could be found (for Ci unlikelky, but more common for location or damage)
        df_valid_ci = df_valid[df_valid["ci1_group"].notnull()]
        df_pred_ci = df_pred[df_pred["infrastructure_group"].notnull()]

        # when no similarity could be calculated
        # ## FIXME move outside of loop
        # entries_with_no_similarity = df_eval_records.loc[df_eval_records["impact_sim_identical"].isna()]
        # print(f" --- Pred-valid pairs where no identical similarity score could be calculated: {len(entries_with_no_similarity)} ----")
        # print(entries_with_no_similarity[["impact_valid", "impact_pred", "impact_sim_identical", "impact_sim_cos", "impact_sim_pr", "citation"]])

        
        # TPs 
        tps = df_smltry_selmax.loc[df_smltry_selmax["ci_smlrty"] == 1]
        print(len(tps), len(df_smltry_selmax ), len(df_smltry_selmax[~df_smltry_selmax[["ci_pred", "ci_valid"]].isna().any(axis=1)]))
        
        
        # # FPs
        # --> make mask where records in df-eval record are identical to df_pred.columns (must be 1:1), 
        #     aplly mask on df_pred and substract from output all cases which are in TPs 
        # assert len(output) == fps_len
        
        # FNs
        ## missed docs
        df_valid_ci_pred_missed_docs = df_valid_ci[df_valid_ci["publication_id"].isin(df_pred["citation_id"]) == False]
        ## missed entries
        # extracts all duplicates (except first occurrence eg. id_Pred==537 occurs in df_smltry_selmax_p three times (1st case: TP or FP, 2nd and 3rd are FNs)
        df_valid_cases_missed_by_model = df_smltry_selmax[df_smltry_selmax.duplicated(subset="id_pred", keep="first")]
        ## OLD APPROACH: CI cases in valid set (for docs existing in both sets) - number of corectly predicted CI cases (Tps)
        ## no_valid_cases_missed_by_model  = len(df_valid_pred_same_docs["ci1_group"])  - len(df_smltry_selmax[df_smltry_selmax["impact_sim_identical"]==1])

        # ## TODO FIXME not sure if approach for df_valid_cases_missed_by_model based on df_smltry_selmax is correct
        # ##            as df_smltry_selmax contains only the cases of highest similarity for each case in df_valid (ie unique id_valid)
        # ##            can i then calc the number of missed cases by 
        
        # fns = pd.concat([df_valid_pred_missed_docs, df_valid_cases_missed_by_model], ignore_index=True, axis=0)

        # FIXME WORKAROUND for FP and FN calculation, but not extract respective cases (only numbers of FPs and FNs)
        fps_len = len(df_pred_ci["infrastructure_group"]) - tps.shape[0]
        fns_len = len(df_valid_ci["ci1_group"]) - tps.shape[0]

        print("tps", len(tps), " fps:", fps_len, " fns:", fns_len)
                    
        # performance scores
        recall_score = u.calc_recall(tps_no=len(tps), fns_no=fns_len) 
        precision_score = u.calc_precision(tps_no=len(tps), fps_no=fps_len)
        try:
            f1_score = u.calc_f1(precision=precision_score, recall=recall_score)
        except ZeroDivisionError:
            f1_score = 0.0

        print(f" ---------- Evaluation statistics: {column_pred}-----------")
        print(f"Recall: {recall_score}, Precision: {precision_score}, F1-score: {f1_score}")

        # saving
        dh.DataHandler().save_evaluation_results(column_pred, df_smltry_selmax, recall_score, precision_score, f1_score)
  

    if column_pred == "location_pred":

        # remove cases where no CI could be found (for CI unlikelky, but more common for location or damage)
        df_valid_loc = df_valid[df_valid["ci1_location"].notnull()]
        df_pred_loc = df_pred[df_pred["location"].notnull()]  # when model gave NaN (then actually also corresponding df_valid record would be there NaN)

        # TPs 
        tps = df_smltry_selmax.loc[df_smltry_selmax["loc_smlrty_norm_pr"] >= norm_pr_smlrty_thresh]

        # FIXME WORKAROUND for FP and FN calculation, but not extract respective cases (only numbers of FPs and FNs)
        fps_len = len(df_pred_loc["location"]) - tps.shape[0]
        fns_len = len(df_valid_loc["ci1_location"]) - tps.shape[0]
        
        print("tps", len(tps), " fps:", fps_len, " fns:", fns_len)

        # performance scores
        recall_score = u.calc_recall(tps_no=len(tps),fns_no=fns_len) 
        precision_score = u.calc_precision(tps_no=len(tps), fps_no=fps_len)
        try:
            f1_score = u.calc_f1(precision=precision_score, recall=recall_score)
        except ZeroDivisionError:
            f1_score = 0.0
        
        print(f" ---------- Evaluation statistics: {column_pred}-----------")
        print(f"Recall: {recall_score}, Precision: {precision_score}, F1-score: {f1_score}")

        ## saving
        dh.DataHandler().save_evaluation_results(column_pred, df_smltry_selmax, recall_score, precision_score, f1_score)



    if column_pred == "damage_pred":

        # remove cases where no CI could be found (for CI unlikelky, but more common for location or damage)
        df_valid_loc = df_valid[df_valid["ci1_damage"].notnull()]
        df_pred_loc = df_pred[df_pred["damage"].notnull()]  # when model gave NaN (then actually also corresponding df_valid record would be there NaN)

        # TPs 
        tps = df_smltry_selmax.loc[df_smltry_selmax["dam_smlrty"] >= norm_pr_smlrty_thresh]

    
        # ## OLD APPROACH: CI cases in valid set (for docs existing in both sets) - number of correctly predicted CI cases (Tps)
        # # no_valid_cases_missed_by_model  = len(df_valid_pred_same_docs["ci1_damage"])  - len(df_smltry_selmax["impact_valid"])
        # # no_valid_cases_missed_by_model  = len(df_valid_pred_same_docs["ci1_location"])  - len(df_smltry_selmax["impact_valid"])
        # fns = pd.concat([df_valid_pred_missed_docs, df_valid_cases_missed_by_model], ignore_index=True, axis=0)
        
        # FIXME WORKAROUND for FP and FN calculation, but not extract respective cases (only numbers of FPs and FNs)
        fps_len = len(df_pred_loc["damage"]) - tps.shape[0]
        fns_len = len(df_valid_loc["ci1_damage"]) - tps.shape[0]
        
        print("tps", len(tps), " fps:", fps_len, " fns:", fns_len)

        # performance scores
        recall_score = u.calc_recall(tps_no=len(tps),fns_no=fns_len) 
        precision_score = u.calc_precision(tps_no=len(tps), fps_no=fps_len)
        try:
            f1_score = u.calc_f1(precision=precision_score, recall=recall_score)
        except ZeroDivisionError:
            f1_score = 0.0
        
        print(f" ---------- Evaluation statistics: {column_pred}-----------")
        print(f"Recall: {recall_score}, Precision: {precision_score}, F1-score: {f1_score}")

        ## saving
        dh.DataHandler().save_evaluation_results(column_pred, df_smltry_selmax, recall_score, precision_score, f1_score)


## chain of prompts (step2, verif=true)
## nongeoloc, (only koks)
# 19 19 19
# tps 19  fps: 68  fns: 13
#  ---------- Evaluation statistics: ci_group_pred-----------
# Recall: 0.59375, Precision: 0.21839080459770116, F1-score: 0.31932773109243695
# Saving evaluation statistics, distribution plots, and scores to  ci_group_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
# tps 13  fps: 74  fns: 17
#  ---------- Evaluation statistics: damage_pred-----------
# Recall: 0.43333333333333335, Precision: 0.14942528735632185, F1-score: 0.22222222222222224
# Saving evaluation statistics, distribution plots, and scores to  damage_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
# tps 15  fps: 72  fns: 17
#  ---------- Evaluation statistics: location_pred-----------
# Recall: 0.46875, Precision: 0.1724137931034483, F1-score: 0.25210084033613445
# Saving evaluation statistics, distribution plots, and scores to  location_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]


# only geolocalized + imporved valid set (koks + 2paper)
# Precision: 0.5625, Recall: 0.627906976744186, F1-score: 0.5934065934065934  tps 27  fps: 21  fns: 16
# incl non-geolocalized + imporved valid set (koks + 2paper)
#  Precision: 0.6923076923076923, Recall: 0.627906976744186, F1-score: 0.6585365853658537, tps 27  fps: 12  fns: 16


## only geolocalzed + improved valid set
# 98 117 117
# tps 98  fps: 65  fns: 84
#  ---------- Evaluation statistics: ci_group_pred-----------
# Recall: 0.5384615384615384, Precision: 0.6012269938650306, F1-score: 0.5681159420289855
# Saving evaluation statistics, distribution plots, and scores to  ci_group_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
# tps 34  fps: 119  fns: 136
#  ---------- Evaluation statistics: damage_pred-----------
# Recall: 0.2, Precision: 0.2222222222222222, F1-score: 0.2105263157894737
# Saving evaluation statistics, distribution plots, and scores to  damage_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
# tps 74  fps: 89  fns: 108
#  ---------- Evaluation statistics: location_pred-----------
# Recall: 0.4065934065934066, Precision: 0.4539877300613497, F1-score: 0.42898550724637685
# Saving evaluation statistics, distribution plots, and scores to  location_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]


## only geolocalized --> better in PRECISION when df_valid_pred matching only w. geolocalized cases  comp. non-geolocaiized incl.)
# 70 87 87
# tps 70  fps: 122  fns: 76
#  ---------- Evaluation statistics: ci_group_pred-----------
# Recall: 0.4794520547945205, Precision: 0.3645833333333333, F1-score: 0.41420118343195267
# Saving evaluation statistics, distribution plots, and scores to  ci_group_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
# tps 32  fps: 149  fns: 107
#  ---------- Evaluation statistics: damage_pred-----------
# Recall: 0.2302158273381295, Precision: 0.17679558011049723, F1-score: 0.2
# Saving evaluation statistics, distribution plots, and scores to  damage_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
# tps 60  fps: 132  fns: 86
#  ---------- Evaluation statistics: location_pred-----------
# Recall: 0.410958904109589, Precision: 0.3125, F1-score: 0.35502958579881655
# Saving evaluation statistics, distribution plots, and scores to  location_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]


##  geolocalized + non-gelocalized
# for each unique valid record keep only pred-valid pairs of highest similarity
# 77 94 94
# tps 77  fps: 172  fns: 69
#  ---------- Evaluation statistics: ci_group_pred-----------
# Recall: 0.5273972602739726, Precision: 0.3092369477911647, F1-score: 0.389873417721519
# Saving evaluation statistics, distribution plots, and scores to  ci_group_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
# tps 33  fps: 205  fns: 106
#  ---------- Evaluation statistics: damage_pred-----------
# Recall: 0.23741007194244604, Precision: 0.13865546218487396, F1-score: 0.17506631299734746
# Saving evaluation statistics, distribution plots, and scores to  damage_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
# tps 65  fps: 184  fns: 81
#  ---------- Evaluation statistics: location_pred-----------
# Recall: 0.4452054794520548, Precision: 0.26104417670682734, F1-score: 0.32911392405063294
# Saving evaluation statistics, distribution plots, and scores to  location_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]

### LAMA 3 + NER + GeoLLM (step 2)

# 43 43 22  - 24 docs
# tps 43  fps: 104  fns: 38
#  ---------- Evaluation statistics: ci_group_pred-----------
# Recall: 0.5308641975308642, Precision: 0.2925170068027211, F1-score: 0.37719298245614036
# Saving evaluation statistics, distribution plots, and scores to  ci_group_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
# tps 8  fps: 91  fns: 67
#  ---------- Evaluation statistics: damage_pred-----------
# Recall: 0.10666666666666667, Precision: 0.08080808080808081, F1-score: 0.09195402298850575
# Saving evaluation statistics, distribution plots, and scores to  damage_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
# tps 32  fps: 115  fns: 49
#  ---------- Evaluation statistics: location_pred-----------
# Recall: 0.3950617283950617, Precision: 0.21768707482993196, F1-score: 0.2807017543859649
# Saving evaluation statistics, distribution plots, and scores to  location_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]


### LAMA 3 + NER + GeoLLM (step 1) - no countries (only regions, cities, etc)
## not better than with countries (slight increase in recall+precision for LOC )

### LAMA 3 + NER + GeoLLM (step 1)  - 24 docs
# or each unique valid record keep only pred-valid pairs of highest similarity
# 39 46 46
# tps 39  fps: 186  fns: 49
#  ---------- Evaluation statistics: ci_group_pred-----------
# Recall: 0.4431818181818182, Precision: 0.17333333333333334, F1-score: 0.24920127795527158
# Saving evaluation statistics, distribution plots, and scores to  ci_group_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
# tps 16  fps: 176  fns: 66
#  ---------- Evaluation statistics: damage_pred-----------
# Recall: 0.1951219512195122, Precision: 0.08333333333333333, F1-score: 0.11678832116788321
# Saving evaluation statistics, distribution plots, and scores to  damage_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
# tps 26  fps: 199  fns: 62
# ---------- Evaluation statistics: location_pred-----------
# Recall: 0.29545454545454547, Precision: 0.11555555555555555, F1-score: 0.16613418530351434
# Saving evaluation statistics, distribution plots, and scores to  location_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json] 

### LAMA 3 + NER
# 15 33
# tps 15  fps: 27  fns: 18
#  ---------- Evaluation statistics: ci_group_pred-----------
# Recall: 0.45454545454545453, Precision: 0.35714285714285715, F1-score: 0.4
# Saving evaluation statistics, distribution plots, and scores to  ci_group_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
# tps 11  fps: 31  fns: 19
#  ---------- Evaluation statistics: damage_pred-----------
# Recall: 0.36666666666666664, Precision: 0.2619047619047619, F1-score: 0.3055555555555555
# Saving evaluation statistics, distribution plots, and scores to  damage_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
# tps 10  fps: 32  fns: 23
#  ---------- Evaluation statistics: location_pred-----------
# Recall: 0.30303030303030304, Precision: 0.23809523809523808, F1-score: 0.26666666666666666

## Lama 3 # 1 doc Koks 2022
# 20 33
# tps 20  fps: 30  fns: 13
#  ---------- Evaluation statistics: ci_group_pred-----------
# Recall: 0.6060606060606061, Precision: 0.4, F1-score: 0.4819277108433735
# Saving evaluation statistics, distribution plots, and scores to  ci_group_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
# ps 12  fps: 38  fns: 18
#  ---------- Evaluation statistics: damage_pred-----------
# Recall: 0.4, Precision: 0.24, F1-score: 0.3
# Saving evaluation statistics, distribution plots, and scores to  damage_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
# ps 12  fps: 38  fns: 21
#  ---------- Evaluation statistics: location_pred-----------
# Recall: 0.36363636363636365, Precision: 0.24, F1-score: 0.2891566265060241
# Saving evaluation statistics, distribution plots, and scores to  location_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]



# llm_1_updprompt_dNER.csv
# 45 67
# tps 45  fps: 1115  fns: 22
#  ---------- Evaluation statistics: ci_group_pred-----------
# Recall: 0.6716417910447762, Precision: 0.03879310344827586, F1-score: 0.07334963325183375
# tps 14  fps: 1146  fns: 47
#  ---------- Evaluation statistics: damage_pred-----------
# Recall: 0.22950819672131148, Precision: 0.01206896551724138, F1-score: 0.022932022932022934
# tps 11  fps: 1149  fns: 43
#  ---------- Evaluation statistics: location_pred-----------
# Recall: 0.2037037037037037, Precision: 0.009482758620689655, F1-score: 0.018121911037891267


 --- For each unique valid case (unique combi: [ci_valid, damage_valid, location_valid, sentence_text]) calculate similarity ---
Using 100% match for CI types based on subgroups
Using cosine similarity threshold for damages 0.7
Using normalized partial ratio similarity threshold for locations 0.7


Record: 0 / 192
Record: 1 / 192
Record: 2 / 192
Record: 3 / 192
Record: 4 / 192
Record: 5 / 192
Record: 6 / 192
Record: 7 / 192
Record: 8 / 192
Record: 9 / 192
Record: 10 / 192
Record: 11 / 192
Record: 12 / 192
Record: 13 / 192
Record: 14 / 192
Record: 15 / 192
Record: 16 / 192
Record: 17 / 192
Record: 18 / 192
Record: 19 / 192
Record: 20 / 192
Record: 21 / 192
Record: 22 / 192
Record: 23 / 192
Record: 24 / 192
Record: 25 / 192
Record: 26 / 192
Record: 27 / 192
Record: 28 / 192
Record: 29 / 192
Record: 30 / 192
Record: 31 / 192
Record: 32 / 192
Record: 33 / 192
Record: 34 / 192
Record: 35 / 192
Record: 36 / 192
Record: 37 / 192
Record: 38 / 192
Record: 39 / 192
Record: 40 / 192
Record: 41 / 192
Record: 42 / 192
Record: 43 / 192
Record: 44 / 192
Record: 45 / 192
Record: 46 / 192
Record: 47 / 192
Record: 48 / 192
Record: 49 / 192
Record: 50 / 192
Record: 51 / 192
Record: 52 / 192
Record: 53 / 192
Record: 54 / 192
Record: 55 / 192
Record: 56 / 192
Record: 57 / 192
Record: 58 / 192
Record:

In [ ]:
################## is NER (tool) really improving ?   ####################

## --> for STEP 1 eval (doc-wise, chunk-wise CI-loc pairs) NER tool is not improving 
## --> also STEP 2 is not improving extraction 


## without NER
# for each unique valid record keep only pred-valid pairs of highest similarity
# 112 135 135
# tps 112  fps: 104  fns: 71
#  ---------- Evaluation statistics: ci_group_pred-----------
# Recall: 0.6120218579234973, Precision: 0.5185185185185185, F1-score: 0.5614035087719299
# Saving evaluation statistics, distribution plots, and scores to  ci_group_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
## no DAM
# tps 91  fps: 125  fns: 92
#  ---------- Evaluation statistics: location_pred-----------
# Recall: 0.4972677595628415, Precision: 0.4212962962962963, F1-score: 0.45614035087719296
# Saving evaluation statistics, distribution plots, and scores to  location_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]

## Step 1 excl ["Chamra 2006", "Hladny 2004", "Pescaroli 2017"])  (incl non geolocalized)
## without NER , eval CI-loc pair (chunk-wise)
# Recall: 0.41530054644808745, Precision: 0.35185185185185186, F1-score: 0.380952380952381


## Step 2 excl ["Chamra 2006", "Hladny 2004", "Pescaroli 2017"])  (incl non geolocalized)
## without NER , eval CI-loc pair (chunk-wise)
# Recall: 0.37362637362637363, Precision: 0.35051546391752575, F1-score: 0.3617021276595745




## with NER (tool)

## Step 1 excl ["Chamra 2006", "Hladny 2004", "Pescaroli 2017"])  (incl non geolocalized)
# 120 134 134
# tps 120  fps: 147  fns: 71
#  ---------- Evaluation statistics: ci_group_pred-----------
# Recall: 0.6282722513089005, Precision: 0.449438202247191, F1-score: 0.5240174672489083
# Saving evaluation statistics, distribution plots, and scores to  ci_group_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
# tps 52  fps: 207  fns: 132
#  ---------- Evaluation statistics: damage_pred-----------
# Recall: 0.2826086956521739, Precision: 0.20077220077220076, F1-score: 0.2347629796839729
# Saving evaluation statistics, distribution plots, and scores to  damage_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
# tps 96  fps: 171  fns: 95
#  ---------- Evaluation statistics: location_pred-----------
# Recall: 0.5026178010471204, Precision: 0.3595505617977528, F1-score: 0.4192139737991266

## Step 1 excl ["Chamra 2006", "Hladny 2004", "Pescaroli 2017"])  (incl non geolocalized)
## with NER , eval CI-loc pair (chunk-wise)
# Recall: 0.4607329842931937, Precision: 0.3295880149812734, F1-score: 0.3842794759825327

## Step 2 excl ["Chamra 2006", "Hladny 2004", "Pescaroli 2017"])  (incl non geolocalized)
## with NER , eval CI-loc pair (chunk-wise)
# Recall: 0.37362637362637363, Precision: 0.3238095238095238, F1-score: 0.34693877551020413


## Step 1 incl ["Chamra 2006", "Hladny 2004", "Pescaroli 2017"])  (incl non geolocalized)
# for each unique valid record keep only pred-valid pairs of highest similarity
# 125 140 140
# tps 125  fps: 142  fns: 66
#  ---------- Evaluation statistics: ci_group_pred-----------
# Recall: 0.6544502617801047, Precision: 0.4681647940074906, F1-score: 0.5458515283842794
# Saving evaluation statistics, distribution plots, and scores to  ci_group_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
# tps 55  fps: 204  fns: 129
#  ---------- Evaluation statistics: damage_pred-----------
# Recall: 0.29891304347826086, Precision: 0.21235521235521235, F1-score: 0.24830699774266365
# Saving evaluation statistics, distribution plots, and scores to  damage_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
# tps 101  fps: 166  fns: 90
#  ---------- Evaluation statistics: location_pred-----------
# Recall: 0.5287958115183246, Precision: 0.3782771535580524, F1-score: 0.4410480349344978
# Saving evaluation statistics, distribution plots, and scores to  location_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]



# ## gpt-oss 120,  with all common valid docs
# for each unique valid record keep only pred-valid pairs of highest similarity
# 78 86 86
# tps 78  fps: 152  fns: 68
#  ---------- Evaluation statistics: ci_group_pred-----------
# Recall: 0.5342465753424658, Precision: 0.3391304347826087, F1-score: 0.4148936170212766
# Saving evaluation statistics, distribution plots, and scores to  ci_group_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
# tps 74  fps: 156  fns: 68
#  ---------- Evaluation statistics: damage_pred-----------
# Recall: 0.5211267605633803, Precision: 0.3217391304347826, F1-score: 0.39784946236559143
# Saving evaluation statistics, distribution plots, and scores to  damage_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
# tps 74  fps: 156  fns: 72
#  ---------- Evaluation statistics: location_pred-----------
# Recall: 0.5068493150684932, Precision: 0.3217391304347826, F1-score: 0.3936170212765958
# Saving evaluation statistics, distribution plots, and scores to  location_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]


# for each unique valid record keep only pred-valid pairs of highest similarity
# 19 20 20
# tps 19  fps: 68  fns: 14
#  ---------- Evaluation statistics: ci_group_pred-----------
# Recall: 0.5757575757575758, Precision: 0.21839080459770116, F1-score: 0.3166666666666667
# Saving evaluation statistics, distribution plots, and scores to  ci_group_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
# tps 13  fps: 74  fns: 18
#  ---------- Evaluation statistics: damage_pred-----------
# Recall: 0.41935483870967744, Precision: 0.14942528735632185, F1-score: 0.22033898305084748
# Saving evaluation statistics, distribution plots, and scores to  damage_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
# tps 15  fps: 72  fns: 18
#  ---------- Evaluation statistics: location_pred-----------
# Recall: 0.45454545454545453, Precision: 0.1724137931034483, F1-score: 0.25000000000000006

In [ ]:
# # with FAC entity based incl: 94 241 178

# for each unique valid record keep only pred-valid pairs of highest similarity
# 44 53 53
# tps 44  fps: 134  fns: 50
#  ---------- Evaluation statistics: ci_group_pred-----------
# Recall: 0.46808510638297873, Precision: 0.24719101123595505, F1-score: 0.3235294117647059
# Saving evaluation statistics, distribution plots, and scores to  ci_group_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
# tps 19  fps: 148  fns: 68
#  ---------- Evaluation statistics: damage_pred-----------
# Recall: 0.21839080459770116, Precision: 0.11377245508982035, F1-score: 0.1496062992125984
# Saving evaluation statistics, distribution plots, and scores to  damage_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
# tps 29  fps: 149  fns: 65
#  ---------- Evaluation statistics: location_pred-----------
# Recall: 0.30851063829787234, Precision: 0.16292134831460675, F1-score: 0.21323529411764708
# Saving evaluation statistics, distribution plots, and scores to  location_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]

 

## without FAC-based records
## valid, pred, pred_local: 94 209 151

# for each unique valid record keep only pred-valid pairs of highest similarity
# 44 53 53
# tps 44  fps: 107  fns: 50
#  ---------- Evaluation statistics: ci_group_pred-----------
# Recall: 0.46808510638297873, Precision: 0.2913907284768212, F1-score: 0.3591836734693878
# Saving evaluation statistics, distribution plots, and scores to  ci_group_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
# tps 19  fps: 121  fns: 68
#  ---------- Evaluation statistics: damage_pred-----------
# Recall: 0.21839080459770116, Precision: 0.1357142857142857, F1-score: 0.16740088105726872
# Saving evaluation statistics, distribution plots, and scores to  damage_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]
# tps 25  fps: 126  fns: 69
#  ---------- Evaluation statistics: location_pred-----------
# Recall: 0.26595744680851063, Precision: 0.16556291390728478, F1-score: 0.20408163265306123
# Saving evaluation statistics, distribution plots, and scores to  location_smlrty_llm_1_dNER_fixNER [.parquet, _stats.json]



## Calc similarities for CI-Loc pairs

In [ ]:
list_entity_valid = [["ci_group_valid", "location_valid"]]
list_entity_pred = [["ci_group_pred", "location_pred"]]



#  Set similarity threshold (self-defined) when CI case is valid or not FN/FP
cos_smlrty_thresh = 0.7
norm_pr_smlrty_thresh = 0.7


print(" --- For each unique valid case (unique combi: [ci_valid, damage_valid, location_valid, sentence_text]) calculate similarity ---")
print("Using 100% match for CI types based on subgroups")
print("Using cosine similarity threshold for damages", cos_smlrty_thresh)
print("Using normalized partial ratio similarity threshold for locations", norm_pr_smlrty_thresh)

## AIM of evaluation loop below: 
# remove all cases in df_pred_valid_all where pred_entities were wrongly assigned to a valid_entity
## ie keep only pre-valid pairs with highest similarity per unique valid case

## call embedding model for semantic similarity calculation
# embedding_model = u.EmbeddingModel()


# store evaluation results
df_eval_records = pd.DataFrame()


# For each impact case (rows) 
for record_no, impact_record in df_pred_valid_all.iterrows():

    print(f"Record: {record_no } / {len(df_pred_valid_all)}")

    # init dict to store results for each records (row=)
    df_eval = {
        "citation": impact_record.citation_id,
        "chunk_text_pred": impact_record.chunk_text_pred,
        "sentence_text_valid": impact_record.sentence_text_valid,
        "id_pred": impact_record.id_pred,
        "id_valid": impact_record.id_valid
    }
    
    # iterate over the three entity classes (ci, damage, location) to assess LLM performance
    for ci_loc_valid, ci_loc_pred in zip(list_entity_valid, list_entity_pred):

        # Calculate similarities for entries in column pair: entity_pred - entity_valid

        ## calc similarity when both valid_info exist (not NAN) 
        # NOTE df_pred model can put out NAN when case exists but it couldnt find suitable value (e.g. damage_pred="NaN", damage_valid="polluted")
        if impact_record[ci_loc_valid] is not np.nan:
        # if impact_record[entity_pred] and impact_record[entity_valid] is not np.nan:
            ci_pred_impact = impact_record[ci_loc_pred][0]
            loc_pred_impact = impact_record[ci_loc_pred][1]
            ci_valid_impact = impact_record[ci_loc_valid][0]            
            loc_valid_impact = impact_record[ci_loc_valid][1]

            ## CI 
            # similarity for CI-LOC pairs  
            # NOTE - when CI AND LOC are correct - then this is calculated as a TP
            if (ci_pred_impact == ci_valid_impact) & ((fuzz.partial_ratio(loc_pred_impact, loc_valid_impact)/100) > 0.7):
                ci_loc_smlrty = 1
            else:
                ci_loc_smlrty = 0

            # store result for ci-loc pairs 
            df_eval["ci_pred"] = ci_pred_impact  # CI subgroup
            df_eval["ci_valid"] = ci_valid_impact # CI subgroup
            df_eval["loc_pred"] = loc_pred_impact  
            df_eval["loc_valid"] = loc_valid_impact
            
            df_eval["ci_loc_smlrty"] = ci_loc_smlrty


#             if entity_pred == "damage_pred": 

#                 if isinstance(pred_impact, str) & isinstance(valid_impact, str): # check that pred or valid are not NAN
#                     # contextual vectors (transformer-based)
#                     embedded_list = embedding_model.vector_calculation(pred_impact, valid_impact)
#                     # calculate cosine similarity for each pred-valid damage pair
#                     similarity_score_cos = embedding_model.cosine_similarity(embedded_list[0], embedded_list[1])  # 0-1 value, the higher the more similar
                
#                 # if nan -> then it is either FP or FN
#                 elif isinstance(pred_impact, float) | isinstance(np.nan, float):
#                     similarity_score_cos = 0.0     

#                 # store result for DAM and LOC entity 
#                 df_eval["dam_pred"] = pred_impact  
#                 df_eval["dam_valid"] = valid_impact 
#                 df_eval["dam_smlrty"] = similarity_score_cos


    # collect all single records (row) with similarity scores
    df_eval_records = pd.concat([df_eval_records, pd.DataFrame([df_eval])], ignore_index=True)

df_smltry_selmax = df_eval_records.groupby("id_valid").apply(lambda s: s.sort_values(["ci_loc_smlrty"], ascending=False).head(1))



print("for each unique valid record keep only pred-valid pairs of highest similarity")


# iterate over the three entity classes (ci, damage, location) to assess LLM performance
for _, column_pred in zip(list_entity_valid, list_entity_pred):

    if column_pred == ["ci_group_pred", "location_pred"]:

        # remove cases where no CI could be found (for Ci unlikelky, but more common for location or damage)
        df_valid_ci_loc = df_valid[df_valid[["ci1_group", "ci1_location"]].notnull()]
        df_pred_ci_loc = df_pred[df_pred[["infrastructure_group", "location"]].notnull()]


        # TPs 
        tps = df_smltry_selmax.loc[df_smltry_selmax["ci_loc_smlrty"] == 1]
        print(len(tps), len(df_smltry_selmax ), len(df_smltry_selmax[~df_smltry_selmax[["ci_pred", "ci_valid"]].isna().any(axis=1)]))
        
        
        # # FPs
        # --> make mask where records in df-eval record are identical to df_pred.columns (must be 1:1), 
        #     aplly mask on df_pred and substract from output all cases which are in TPs 
        # assert len(output) == fps_len
        
        # FNs
        ## missed docs
        df_valid_ci_pred_missed_docs = df_valid_ci_loc[df_valid_ci_loc["publication_id"].isin(df_pred["citation_id"]) == False]
        ## missed entries
        # extracts all duplicates (except first occurrence eg. id_Pred==537 occurs in df_smltry_selmax_p three times (1st case: TP or FP, 2nd and 3rd are FNs)
        df_valid_cases_missed_by_model = df_smltry_selmax[df_smltry_selmax.duplicated(subset="id_pred", keep="first")]
     
        # FIXME WORKAROUND for FP and FN calculation, but not extract respective cases (only numbers of FPs and FNs)
        fps_len = len(df_pred_ci_loc[["infrastructure_group", "location"]]) - tps.shape[0]
        fns_len = len(df_valid_ci_loc[["ci1_group", "ci1_location"]]) - tps.shape[0]

        print("tps", len(tps), " fps:", fps_len, " fns:", fns_len)
                    
        # performance scores
        recall_score = u.calc_recall(tps_no=len(tps), fns_no=fns_len) 
        precision_score = u.calc_precision(tps_no=len(tps), fps_no=fps_len)
        try:
            f1_score = u.calc_f1(precision=precision_score, recall=recall_score)
        except ZeroDivisionError:
            f1_score = 0.0

        print(f" ---------- Evaluation statistics: {column_pred}-----------")
        print(f"Recall: {recall_score}, Precision: {precision_score}, F1-score: {f1_score}")

        # saving
        # dh.DataHandler().save_evaluation_results(column_pred, df_smltry_selmax, recall_score, precision_score, f1_score)



## no pairs
# only geolocalized + imporved valid set (koks + 2paper)
# Precision: 0.5625, Recall: 0.627906976744186, F1-score: 0.5934065934065934  tps 27  fps: 21  fns: 16
# incl non-geolocalized + imporved valid set (koks + 2paper)
#  Precision: 0.6923076923076923, Recall: 0.627906976744186, F1-score: 0.6585365853658537, tps 27  fps: 12  fns: 16


 --- For each unique valid case (unique combi: [ci_valid, damage_valid, location_valid, sentence_text]) calculate similarity ---
Using 100% match for CI types based on subgroups
Using cosine similarity threshold for damages 0.7
Using normalized partial ratio similarity threshold for locations 0.7
Record: 0 / 81
Record: 1 / 81
Record: 2 / 81
Record: 3 / 81
Record: 4 / 81
Record: 5 / 81
Record: 6 / 81
Record: 7 / 81
Record: 8 / 81
Record: 9 / 81
Record: 10 / 81
Record: 11 / 81
Record: 12 / 81
Record: 13 / 81
Record: 14 / 81
Record: 15 / 81
Record: 16 / 81
Record: 17 / 81
Record: 18 / 81
Record: 19 / 81
Record: 20 / 81
Record: 21 / 81
Record: 22 / 81
Record: 23 / 81
Record: 24 / 81
Record: 25 / 81
Record: 26 / 81
Record: 27 / 81
Record: 28 / 81
Record: 29 / 81
Record: 30 / 81
Record: 31 / 81
Record: 32 / 81
Record: 33 / 81
Record: 34 / 81
Record: 35 / 81
Record: 36 / 81
Record: 37 / 81
Record: 38 / 81
Record: 39 / 81
Record: 40 / 81
Record: 41 / 81
Record: 42 / 81
Record: 43 / 81
Record: 

,infrastructure_group,location
0,road_others,Castilla-La Mancha
1,road_others,Valencia
2,education_school,Castellón de la Plana
3,rail,Valencia
4,road_others,Valencian Community
...,...,...
262,road_others,Azucarera - Interhorce road
263,road_others,Santa Rosa de Lima
264,road_others,Victoria
265,motorways,MA21


### Benchmark against Ni Li Wiki-impacts - Loc calculation

score for boolean or str : 
if identical = 0
if not identical = 1

score for lists
= 1 - |{schnittmenge von a und r}|  / |{Vereiningunsmenge von a un d r}

-->  no. of common elements ["identicals"] / entire number of elements which occur at least in a or r or in both

```
Schnittmenge (\(\cap \)): Elemente, die in beide Mengen gehören.
Vereinigung (\(\cup \)): Elemente, die in mindestens einer der beiden Mengen liegen (Zeichen: \(A \cup B\), sieht aus wie ein normales „u“).
```

In [ ]:
def common_elements(list1, list2):
    return [element for element in list1 if element in list2]


In [ ]:
from itertools import chain


# calc share of loc=1 and loc=0 (to benchmark against Ni li)


fns_list = []
fps_list = []   
tps_list = []



for publication in df_valid.publication_id.unique():

    print(f"\n\nDocument: {publication}")

    docs_valid = df_valid[df_valid["publication_id"] == publication]
    docs_pred = df_pred[df_pred["citation_id"] == publication]

    docs_valid_pairs = docs_valid[["ci1_location"]].values.tolist()# .drop_duplicates().values.tolist()
    docs_pred_pairs = docs_pred[["location"]].values.tolist()#.drop_duplicates().values.tolist()

    docs_valid_pairs = sorted(list(chain.from_iterable(docs_valid_pairs)))
    docs_pred_pairs = sorted(list(chain.from_iterable(docs_pred_pairs)))
    print(sorted(docs_valid_pairs))
    print(sorted(docs_pred_pairs))

    # print(docs_valid_pairs)

    print(docs_valid_pairs)
    print(docs_pred_pairs)

    # calc similarities
    identical = common_elements(docs_pred_pairs, docs_valid_pairs)# TP
    fns = len(docs_valid_pairs) - len(identical)
    fps = len(docs_pred_pairs) - len(identical)
    print("Identical [score 0]", len(identical))
    print("FNs [score 1]", fns)
    print("FPs [score 1]", fps)
    print("Not Identical [score 1]- Fn + FP sum ", fns + fps)
    print("share according to Ni Li - list",  1 - (len(identical)/ (len(docs_pred_pairs) + len(docs_valid_pairs))))
    print("share according to Ni Li -set ",  1 - (len(set(identical))/ (len(set(docs_pred_pairs)) + len(set(docs_valid_pairs)))))
    #(37/ (docs_pred_pairs + docs_valid_pairs))




Document: Koks 2022
['A1', 'Ahr valley', 'Ahr valley', 'Ahr valley', 'Ahr valley', 'Ahr valley', 'Ahr valley', 'Ahrweiler', 'Altenahr', 'Bad Münstereifel', 'Erft', 'Erft region', 'Eschweiler', 'Liége', 'Maastricht', 'Mayschoss', 'North Rhine-Westphalia', 'North Rhine-Westphalia', 'North Rhine-Westphalia', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Rhineland-Palatinate', 'Sinzig', 'Trier', 'all severely affected areas', 'flood region', 'rivers Ahr']
['A1 motorway', 'A76', 'Ahr', 'Ahr valley', 'Ahr valley', 'Ahr valley', 'Ahr valley', 'Ahr valley', 'Ahr valley', 'Altenahr', 'Altenahr', 'Altenburg', 'Altenburg', 'Altenburg', 'B266', 'Bad Münstereifel', 'Bad Münstereifel', 'Bad Münstereifel', 'Bad Münstereifel', 'Bad Neuenahr-Ahrweiler', 'Chaudfontaine', 'Erft', 'Eschweiler', 'Eschweiler', 'Heimersheim', 'Heimersheim', 'Liége', 'Maastricht'

#### ISSUE: some pred-valid matches are wrongly matched in Text-Simil.based eval
--> (probably bc no better pred_entry for corresponding valid_entry exists) e,g, 
* pred: "in ahr valley some roads were destroyed. the railways in Germany were damaged" - roads,destroyed, ahr valley
* valid: "the railways in Germany were damaged" - railways, damaged, Germany

IDEA. do document-wise comparison or based on spatial agg. --> later maybe better da 

In [ ]:
df_pred.to_csv(f"df_pred_{step}_why_ci_dam_loc_bad.csv")
df_smltry_selmax.to_csv(f"df_smlrty_max_{step}_why_ci_dam_loc_bad.csv")

## EVAL why performance is not good LLAMA 3


In [ ]:
df_smltry_selmax[["citation", "sentence_text_valid", "id_pred", "id_valid", "ci_pred", "ci_valid", "ci_smlrty", "dam_pred", "dam_valid", "dam_smlrty", "loc_pred", "loc_valid", "loc_smlrty", "loc_smlrty_norm_pr"]].head(5)

#### FIXME: find out which cases model predicted existence, but not in valid DS - maybe due that valid DS is incomppete?

In [ ]:
df_pred.info()

In [ ]:
## fix FPs 

## get all pred cases which 
# rows in df_valid where sentence_reference appears as substring in at least one df_pred.chunk_text
chunk_texts = df_pred["chunk_text"].dropna().astype(str)

df_valid_2 = df_valid[
    df_valid["sentence_reference"].fillna("").astype(str).apply(
        lambda s: any(s and s in chunk for chunk in chunk_texts)
    )
]

df_valid_2 # .shape (28, 7)

In [ ]:
# cases where pred-case exist but no fitting valid case could be found based on sentence_reference
df_pred_not_in_valid = df_pred[df_pred['chunk_text'].str.contains('|'.join(df_valid["sentence_reference"]), regex=True)]
print(df_pred_not_in_valid.id_pred.value_counts())
df_pred_not_in_valid.head(10)

# TODO TODO
## documents where model found many CI cases as false-alarms:
# maybe i need to recheck those docs and make df_valid more complete
# print(df_pred_not_in_valid.groupby("citation_id").count())
# citation_id                                                            
# ABC 2024                        4
# Containerlift 2024             24
# European Investment Bank 2025  21
# Ferlita 2023                   22
# Koks 2022                      10


In [ ]:
df_valid

#### FIXME: FPs and FNs

In [ ]:
## TPs + FNs should be == len(df_valid.ci) == 67
        
# TPs 
tps = df_smltry_selmax.loc[df_smltry_selmax["impact_sim_identical"] == 1]

# FNs
df_valid_pred_missed_docs = df_valid[df_valid["publication_id"].isin(df_pred["citation_id"]) == False]
df_valid_cases_missed_by_model = df_smltry_selmax[df_smltry_selmax.duplicated(subset="id_pred", keep="first")]
fns = pd.concat([df_valid_pred_missed_docs, df_valid_cases_missed_by_model], ignore_index=True, axis=0)

print(tps.shape[0], fns.shape[0])
print(tps.shape[0] + fns.shape[0])

# --> 8 cases in FNs are too definitly too much --> fix FN calculation



## FPs should be == len(df_pred.ci) - TPs

## FPs
fps = df_smltry_selmax.loc[df_smltry_selmax["impact_sim_identical"] == 0]

print(len(df_pred.infrastructure_type), tps.shape[0], fps.shape[0])
print(len(df_pred.infrastructure_type) - tps.shape[0])




In [ ]:
# TODO fix FPs
# get records where model predicted presence of impacts but they actually does not exist
# here as definition, that when simi=0 (or below threshold) then model predicted wrongly
df_smltry_not_sim = df_smltry_selmax.loc[df_smltry_selmax["impact_sim_identical"] == 0]

# TODO
# add also as Fps were model_pred case exist but no fitting_vlaid case could be found

# idea: 
# get all df_pred cases where chunk text not occurs in valid.sentece_text

df_pred_not_in_valid = df_pred[df_pred["chunk_text"].isin(df_valid["sentence_reference"])== False]
print(df_pred.shape, df_pred_not_in_valid.shape)
# df_pred_not_in_valid

In [ ]:
# df_valid__pred_no_thresh.id_pred.nunique()
df_pred_valid_no_thresh.id_pred.nunique()

In [ ]:
# df_valid__pred_no_thresh.info()
# df_pred_valid_no_thresh.info()  # 67 valid * 921 pred = 61707
# df_pred_valid_no_thresh.drop("chunk_text_pred", axis=1).sort_values("id_pred").iloc[0:100]
# df_pred_valid_no_thresh.groupby("id_pred").first().sort_values("text_similarity", ascending=False).iloc[0:100]
# df_valid__pred_no_thresh.groupby("id_valid").first().sort_values("text_similarity", ascending=False).iloc[0:100]
#df_valid__pred_no_thresh.sort_values("id_valid", ascending=False).sort_values("text_similarity", ascending=False).iloc[0:100]
#df_valid__pred_no_thresh.groupby("id_valid").first().sort_values("text_similarity", ascending=False).iloc[0:100]
df_valid__pred_no_thresh.groupby("id_valid").apply(lambda x: x.loc[x["text_similarity"].idxmax()])


In [ ]:
# fns

In [ ]:
## FNs 
df_smltry_selmax.loc[df_smltry_selmax.duplicated("id_pred")]
## --> ISSUE: this df (cases of highest sim) should NOT have duplicated cases of predictions -> maybe have to group based on id_pred and not id_valid

## try to fix issue
## --> currently i think this should group based on valid cases to measure were model predicted the same or missed info (ie FNs)
# df_smltry_selmax_p = df_smltry_selmax
# mask of rows with highest similarity score for each set of preds with unique valid case (droplevel(0) remove multiindex)
mask = df_smltry_all.loc[df_smltry_all.id_valid==37].groupby("id_valid").apply(lambda x: x==x["impact_sim_identical"].max()).droplevel(0)
df_smltry_selmax_p = df_smltry_all.where(mask.impact_sim_identical==mask.impact_sim_identical.max()).dropna(how="all") # drop cases which have not highest similairty score
# df_smltry_selmax_p = df_smltry_all.groupby("id_valid").apply(lambda x: x.loc[x["impact_sim_identical"].idxmax()]) # 52 cases
# df_smltry_selmax_p = df_smltry_all.groupby("id_pred").apply(lambda x: x.loc[x["impact_sim_identical"].idxmax()]) # 175 cases
df_smltry_selmax_p.reset_index(drop=True, inplace=True)

## FIXME  df_smltry_all.groupby("id_valid"): should it has duplicated cases of id_pred ? - i dont think so! 
#  bc it means that there model missed cases in valid_set
## --> so all duplicated cases (except one-this is TP or FP) are actual FNs
print(df_smltry_selmax_p.info())
print(df_smltry_selmax_p.id_pred.nunique()  )  # should be len of df
print(df_smltry_selmax_p.duplicated().sum())


# FNs: extracts all duplicates (except first occurrence eg. id_Pred==537 occurs in df_smltry_selmax_p three times (1st case: TP or FP, 2nd and 3rd are FNs)
fns = df_smltry_selmax_p[df_smltry_selmax_p.duplicated(subset="id_pred", keep="first")]

print(fns.info())
fns.id_pred.value_counts()


In [ ]:
# return all cases which has max sim also when max score is shared by multiple rows 
# df_smltry_all.loc[df_smltry_all.groupby("id_valid").transform(lambda x: x==x.max()).astype('bool')].shape
mask = df_smltry_all.loc[df_smltry_all.id_valid==37].groupby("id_valid").apply(lambda x: x==x["impact_sim_identical"].max())
mask = mask.droplevel(0)
#.transform(lambda x: x==x.max())
tt = df_smltry_all.loc[df_smltry_all.id_valid==37]#
tt = tt.where(mask.impact_sim_identical==mask.impact_sim_identical.max()).dropna(how="all") # drop cases which have not highest similairty score

# tt[mask]

# would return only first case of max sim:
#df_smltry_all.loc[df_smltry_all.id_valid==37].groupby("id_valid").apply(lambda x: x.loc[x["impact_sim_identical"].idxmax()]) #


In [ ]:
# FPs. 
print("False alarms (where model predicted ci but no corresponding valid case exists)", 
      len(df_pred["infrastructure_type"])  - len(df_smltry_selmax[df_smltry_selmax["impact_sim_identical"]== 1, "ci_group_pred"])
    )
# Get FPs - cases where model predicted presence of CI (but actually it is absent in valid set)
tt = df_pred.merge(
    # FIXME issue that df_pred_valid_all contains some duplicates where id_pred identical but not valid_entries
    df_smltry_selmax.drop_duplicates(), # safety: make sure that merging is done on 1:1 match
    left_on="id_pred",#["citation_id", "chunk_id","infrastructure_type", "damage", "location"], 
    right_on="id_pred",#["citation_id", "chunk_id_pred", "ci_pred", "damage_pred", "location_pred"],
    how="left",
    indicator=True    # return an extra column indicating which table the row was from.
)
tt = tt.loc[tt["_merge"] == "left_only"].drop(columns=["_merge"])
print("False positives (model predicted CI but no corresponding valid case exists):", len(tt))

In [ ]:
print(df_pred.shape[0])
# print(df_pred_valid_all.info())
print(tt.info())

In [ ]:
df_pred#["infrastructure_type"]

In [ ]:
df_smltry_selmax.info()

In [ ]:
# df_smltry_selmax["impact_sim_identical"] < cos_smlrty_thresh

In [ ]:
# len(df_valid_pred_same_docs["ci1_group"]) 

In [ ]:
# TODO fix FNs
print(df_valid_pred_same_docs.info())
print(df_smltry_selmax[df_smltry_selmax["impact_sim_identical"]==1].info())
# --> FNS should be  23
len(df_valid_pred_same_docs["ci1_group"])  - len(df_smltry_selmax[df_smltry_selmax["impact_sim_identical"]==1]["impact_valid"])

In [ ]:
# #df_valid_pred_same_docs["id_valid"] = df_valid_pred_same_docs.apply(lambda x: f"{x['ci1_group']}_{x['ci1_damage']}_{x['ci1_location']}_{x['sentence_text_valid'][:50]}", axis=1)
# print(df_valid_pred_same_docs["id_valid"].unique().__len__())
# print(df_valid_pred_same_docs.shape[0])
# ## --> check why electricity_others_outages_nan = 3  - (seems correct as sentences_ref are diff). airports_affected_Malaga area=2 are not unique
# df_valid_pred_same_docs[df_valid_pred_same_docs["id_valid"] == "airports_affected_Malaga area"]

# Improve similarity
As all similarity measures - no matter which emebdding model or kind of cosine similarity measure) were not sufficient eg. port ~ power to similar to port~harbor

Thus, it might be better to first group ci impacts into subgroups e.g .based on HARCI-EU categories,as some kind of postprocessing step before applying the similarity measurements



In [ ]:
# s = "dyke" to s2 = "levee", s3 = "dam"
# bge-m3: 0.48  0.54
# all-MIniLM-L6-v2: 0.34 , 0.36  (similar all-mpnet-base-v2)
# gensim word2vec: 0.39 0.40


# s1 = "aviation" s2 = "air traffic"
# word vector spacy: 0.45
# contextual vector spacy: 0.68
# bge-m3: 0.76
# all-MIniLM-L6-v2: xx  (all-mpnet-base-v2: 0.79)
# gensim word2vec: 


# s1 = "power" s2 = "electricity"
# word vector spacy: 0.61
# contextual vector spacy: 0.66
# bge-m3: 
# all-MIniLM-L6-v2: xx   (all-mpnet-base-v2: 0.43)
# gensim word2vec: 0.58


# s1 = "electricity infrastructure" s2 = "electricity"
# word vector spacy: 0.87
# contextual vector spacy: 0.71
# bge-m3: 
# all-MIniLM-L6-v2:   xx  (all-mpnet-base-v2: 0.63)
# gensim word2vec: 


# s1 = "transportation" s2 = "transport infrastructure"
# word vector spacy: 
# contextual vector spacy: 
# bge-m3: 
# all-MIniLM-L6-v2:   xx  (all-mpnet-base-v2: 0.84)
# gensim word2vec: 

# s1 = "port" s2 = "power"  s3= harbour
# bge-m3: 0.58, 0.50
# all-MIniLM-L6-v2: 0.33 , 0.56  (similar all-mpnet-base-v2)
# gensim word2vec: 0.14 0.59

# s1 = "electricity" s2 = "transportation" 
# bge-m3:  0.64
# all-MIniLM-L6-v2:   (all-mpnet-base-v2: 0.47)
# gensim word2vec: 0.33

In [ ]:
# # print(cos_sim(model_scs["transportation"], model_scs["transport infrastructure"]))
# # print(cos_sim(model_scs["electricity infrastructure"], model_scs["electricity"]))
# # print(cos_sim(model_scs["power plant"], model_scs["electricity"]))
# print(cos_sim(model_scs["power"], model_scs["electricity"]))
# print(cos_sim(model_scs["aviation"], model_scs["air traffic"]))
# # identical to model_scs.similarity("port", "power"))


# # similarity_score = 1-distance.cosine(model.encode([s1])[0], model.encode([s2])[0])

### Analyse evaluation results 


In [ ]:
df_smltry_selmax#.info()

In [ ]:
## find out for which docs model performed bad (or good)
## based on this info try to improve model 

df_smltry_selmax.dropna(subset=["impact_sim_cos"]).groupby("citation").apply(lambda x: x.loc[x["impact_sim_cos"].idxmax()]).sort_values(by="impact_sim_cos", ascending=True)
## check EFE, Wilson, European Investment Bank, Containerlift, Lloyds List, Gilbody Dickerson


In [ ]:
## check entries of worst performace docs for damage
df_smltry_selmax.loc[df_smltry_selmax["citation"].isin(["Khazai 2023", "ABC 2024", "Containerlift 2024", "Lloyds List 2024", "Ferlita 2023"])]

In [ ]:
## check entries of worst performance docs for Ci tyes
df_smltry_selmax.loc[df_smltry_selmax["citation"].isin(["EFE 2024", "Containerlift 2024", "Lloyds List 2024", "Wilson 2024", "Gilbody Dickerson 2024", "European Investment Bank 2025"])].head(50)


## For each validation entry, search for all prediction cases of the same chunk 

In [ ]:
## get same impact entries
list_entity_valid = ["ci1_type", "ci1_damage", "ci1_location"]
list_entity_pred = ["infrastructure_type", "damage", "location"]


for entity_valid, entity_pred in zip(list_entity_valid, list_entity_pred):

    print(f" --------- Processing column pair: {entity_valid} - {entity_pred} ------------")
    
    df_valid_pred_all = pd.DataFrame()
    citations_list = []

    ## for each validation record
    for i in range(len(df_valid)):
        
        highest_similarity_score = 0.00
        
        ## needed to traceback info when entry is missing in pred. DS
        # chunk_id_value_valid = df_valid.chunk_id[i]

        # select nth validation record and check that it has value
        df_valid_entry = df_valid.iloc[i]
        if df_valid_entry[entity_valid] is np.nan:
            continue
        
        citation_str = df_valid_entry.publication_id
        citations_list.append(citation_str)


        # get all corresponding prediction records
        df_pred_entries = df_pred[df_pred["citation_id"].isin([citation_str])]

        #  handle on NANs
        df_pred_entries[entity_pred] = np.where(df_pred_entries[entity_pred].isna(), "nan", df_pred_entries[entity_pred])
        # df_pred_entries[entity_pred] = df_pred_entries[entity_pred].astype(str)
        # remove double whitespaces
        # df_pred_doc[entity_pred] = df_pred_doc[entity_pred].replace("  ", " ")
        # df_valid_entries[entity_valid] = df_valid_entries[entity_valid].replace("  ", " ")


        # vector of validiation entry 
        valid_impact = df_valid_entry[entity_valid]
        valid_vec = nlp(valid_impact).vector

        # print(" ------- Searching for citation:", citation_str, " in predictions ------- ")

        # Compute similarity between each validation CI impact case and all potential predicted CI impact cases (cross-product)
        for j in range(len(df_pred_entries[entity_pred])):

            if df_pred_entries[entity_pred].iloc[j] == "nan":
                continue

            pred_impact = df_pred_entries[entity_pred].iloc[j]

            pred_vec = nlp(pred_impact).vector
            similarity_score = embedding_model.cosine_similarity(valid_vec, pred_vec)  # 0-1 value, the higher the more similar
            # print(f"Similarity {i}-{j}: {similarity_score}")

            # print(f"Searching for highest similarity ... ")
            ## get only pair with highest similarity
            if similarity_score > highest_similarity_score:
                
                highest_similarity_score = similarity_score
                
                dict_pair = {
                    "impact_valid": valid_impact, 
                    "impact_pred": pred_impact, 
                    "similarity": highest_similarity_score,
                    "citation": citation_str,
                    "chunk_id_pred": (df_pred.chunk_id[i],  df_pred.chunk_id[j])
                }
            else:
                continue

        df_valid_pred_all = pd.concat([df_valid_pred_all, pd.DataFrame([dict_pair])], ignore_index=True)


    print(f" ---------- Evaluation summary statistics - {entity_pred}: -----------")
    print(df_valid_pred_all.similarity.describe())

    

    SIMILARITY_FILENAME = f'{entity_pred}_{SIMILARITY_LLM_FILENAME}'
    SIMILARITY_FILEPATH = Path(PATH_EVAL_RESULT / SIMILARITY_FILENAME)

    print("Saving evaluation statistics, distribution plots, and scores to ", SIMILARITY_FILEPATH.stem, "[.parquet, _stats.json]")
    with open(SIMILARITY_FILEPATH, 'w') as f:
        # results
        df_valid_pred_all.to_csv(SIMILARITY_FILEPATH.with_suffix('.csv'), index=False)
        df_valid_pred_all_pyarrow = pa.Table.from_pandas(df_valid_pred_all)
        pq.write_table(df_valid_pred_all_pyarrow, SIMILARITY_FILEPATH)   
        #   summary statistics
        df_valid_pred_all_stats = df_valid_pred_all.describe()
        f = SIMILARITY_FILEPATH.parent / f"{SIMILARITY_FILEPATH.stem}_stats.json"  
        df_valid_pred_all_stats.to_json(f, indent=4)
        # distribution plots
        df_valid_pred_all.similarity.hist(bins=100).to_file(SIMILARITY_FILEPATH.parent / f"{SIMILARITY_FILEPATH.stem}_hist.png")



    # cos_smlrty_thresh = 0.75
    # df_valid_pred_all['is_similar'] = df_valid_pred_all['similarity'] <= cos_smlrty_thresh
    # print(f"Number of similar impact cases (similarity >= {cos_smlrty_thresh}): {df_valid_pred_all['is_similar'].sum()} out of {len(df_valid_pred_all)}\n")

    # df_valid_pred_all =  df_valid_pred_all[df_valid_pred_all['similarity'] <= cos_smlrty_thresh]

    # SIMILARITY_FILENAME = f'{entity_pred}_lower75_{SIMILARITY_LLM_FILENAME}'
    # SIMILARITY_FILEPATH = Path(PATH_EVAL_RESULT / SIMILARITY_FILENAME)

    # with open(SIMILARITY_FILEPATH, 'w') as f:
    #     # results
    #     df_valid_pred_all.to_csv(SIMILARITY_FILEPATH.with_suffix('.csv'), index=False)
    #     df_valid_pred_all_pyarrow = pa.Table.from_pandas(df_valid_pred_all)
    #     pq.write_table(df_valid_pred_all_pyarrow, SIMILARITY_FILEPATH)  
    #     #   summary statistics
    #     df_valid_pred_all_stats = df_valid_pred_all.describe()
    #     f = SIMILARITY_FILEPATH.parent / f"{SIMILARITY_FILEPATH.stem}_stats.json"  
    #     df_valid_pred_all_stats.to_json(f, indent=4)


In [ ]:
# df_valid_pred_all[df_valid_pred_all['similarity'] <= 0.75]

# df_valid_pred_all.similarity.hist(bins=100)

In [ ]:
list_entity_pred

In [ ]:
LLM_DATA_FILEPATH

### Load parquet file

In [ ]:

list_entity_pred = ["infrastructure_type", "damage", "location"]

In [ ]:
entity_pred = "infrastructure_type"
SIMILARITY_FILENAME = f'llm1_similarity_{entity_pred}_75.parquet'
SIMILARITY_FILEPATH = Path(PATH_EVAL_RESULT / SIMILARITY_FILENAME)

with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
    df = pd.read_parquet(SIMILARITY_FILEPATH, engine='pyarrow')
    display(df)

## Archive

#### spatial mapping /Nominatim and request

In [ ]:
# #dummy_df = pd.read_
# # ST_ReadOSM
# # cached output

# loc_name = "A 61" # df_pred_geolocalized["location"].iloc[0]
# loc_type = 'highway'
# geom_type = 'line'


# # r = duckdb.sql(f"""
# # COPY(
# #     SELECT id, tags['ref'] AS ref, tags, geometry
# #     FROM 'rheinland-pfalz-260531.osm.pbf'
# #     WHERE kind = '{geom_type}'
# #     AND tags['ref'] = '{loc_name}'
# #     AND tags['{loc_type}'] = 'motorway'
# # ) TO  'output.geojson';
# # ;
# #   """)# .df()
# r = duckdb.sql(f"""
# COPY(
#     SELECT id, tags['ref'] AS ref, tags['{loc_type}']  as loc_type, tags, geometry
#     FROM 'rheinland-pfalz-260531.osm.pbf'
#     WHERE kind = '{geom_type}'
#     AND tags['ref'] = '{loc_name}'
#     AND tags['{loc_type}'] = 'motorway'
# ) TO  'locations_geocoded' (FORMAT GDAL, DRIVER GeoJSON, OVERWRITE_OR_IGNORE, PARTITION_BY (loc_type), FILENAME_PATTERN '{loc_name}');
# ;
#   """)# .df()
#  # ) TO  'output.csv' (FORMAT CSV, HEADER, DELIMITER ',')

# df_locs = pd.DataFrame()

# geolocs = glob("./locations_geocoded/**/*geojson")

# for geoloc in geolocs:
#     # geoloc = geoloc.replace("json ", "")
#     df_geoloc = gpd.read_file(geoloc, driver="GeoJSON", crs="EPSG:4326")
#     df_locs = pd.concat([df_locs, df_geoloc], ignore_index=True)
 

In [ ]:

# ## coords from geollama
# # valencia: 39.4697065', '-0.3763353'

# geolocator = Nominatim(user_agent="your_app_name")
# location = geolocator.geocode("Milan")

# lat = location.latitude
# lng = location.longitude

# print(lat, lng, location)

# # f"""
# #  [out:json];
# #     area["ISO3166-1"="IT"][admin_level=2];
# #     {{geocodeArea:Milan}};
# #     nwr(pivot);
# #     out geom;
# # """

# #        );out;
# #"""

# # display_name: Palermo, Sicily, Italy
# # 

# # if response.status_code == 200:
# #     data = response.json()
# #     print(f"{len(data.elements)} Stolpersteine gefunden!")
# # else:
# #     print("Error", response.text)


# print(r)
# # ILIKE '%mark%'   # case-insentiive "Ahrmark"

# # r = duckdb.sql(f"""

# #     SELECT id, JSON_EXTRACT(tags, '$.highway') AS tags_type, tags, geometry,
# #     CASE
# #       WHEN tags_type == 'motorway' THEN  tags[{loc_type}]==tags['motorway']
# #     END AS dd   
# #     FROM 'rheinland-pfalz-260531.osm.pbf'
# #     WHERE kind = '{geom_type}'
# #     AND tags['ref'] = '{loc_name}'
# # ;
# #   """)#.df()
# # print(r)

# #  WHEN tags[{loc_type}] == tags['highway'] THEN tags[{loc_type}] = tags['motorway']
# #        WHEN tags[{loc_type}] == tags['city'] THEN tags[{loc_type}] = tags['city']

# # TO  'output.csv' (FORMAT CSV, HEADER, DELIMITER ',');
# # ) TO  'output.geojson' WITH (FORMAT GDAL, DRIVER GeoJSON);

# #         AND tags['highway'] = 'motorway'

# # AND tags['place'] = 'city'

# #  type, id, tags, geometry
# # results = duckdb.sql("""
# #     LOAD osmium;                     
# #     SELECT *
# #     FROM 'rheinland-pfalz-260531.osm.pbf'
# #     WHERE tags['name']='Ahr' AND tags['waterway']='river'
# #    ; 
# #   """).df()
# # results.head()
# # duckdb.close()

In [ ]:
# ## test nominatim (2min )
# import requests

# url = "https://nominatim.openstreetmap.org/reverse"
# params = {
#     "format": "jsonv2",
#     # "name": "Valencia",
#     # "name": "Valencian Community ",
#     #"addresstype": "city",
#     # test palermo
#     "lat": 38.2608690,
#     "lon": 15.5679169
# }

# response = requests.get(url, params=params, headers={"User-Agent": "my_location_app"})
# data = response.json()

# print(data.get("display_name"))


In [ ]:
# print(data.get("display_name"))
data

In [ ]:

# import geopandas as gpd 
# from glob import glob

# geolocs = glob("./locations_geocoded/**/*geojson")

# for geoloc in geolocs:
#     # geoloc = geoloc.replace("json ", "")
#     df_geoloc = gpd.read_file(geoloc, driver="GeoJSON", crs="EPSG:4326")
#     df_geoloc
#dd = gpd.read_file("output.csv", crs="EPSG:4326")
# dd['geometry'] = gpd.GeoSeries.from_wkt(dd['geometry'])
# dd = gpd.GeoDataFrame(dd, geometry='geometry')


In [ ]:
# ## test A3 query
# way_query = overpass.WayQuery('["name"="Highway 51"]')
# response = api.get(way_query)


In [ ]:
## Aim 
## for all identical valid entries ie. with same [ci_valid	damage_valid	location_valid	sentence_text_valid]
## get the match to pred_entity with highest similarity

In [ ]:
    # ## calc for each entry with the same chunk_text the similarity between valid_impact and pred_impact
    # ## means we calc also the False Negatives (ie. where valid entry exists but no prediction)


    # # iterate over groups of entities which refer to the same valid case (i.e. which are identical in valid_columns)
    # # TODO iterate over unqiue cases in df_valid (instead of using grouper)
    # grouper = df_pred_valid_all[["ci_valid", "damage_valid", "location_valid", "sentence_text_valid"]].drop_duplicates()
    # for group in grouper.itertuples():
    #     df_pred_valid_group = df_pred_valid_all[df_pred_valid_all[["ci_valid", "damage_valid", "location_valid", "sentence_text_valid"]] == group[["ci_valid", "damage_valid", "location_valid", "sentence_text_valid"]]]

    #     # calc. similarities to pred_entities
    #     for i, entry in df_pred_valid_group.iterrows():

    #         highest_similarity_score = 0 

    #         if entry[entity_pred].iloc[i] == "nan":
    #             continue
            
    #         # calc embeddings
    #         pred_impact = entry[entity_pred].iloc[i]
    #         pred_vec = nlp(pred_impact).vector

    #         valid_impact = entry[entity_valid].iloc[i] # is unique for each group
    #         valid_vec = nlp(valid_impact).vector
    #         print(valid_impact, "valid_impact")
            
    #         similarity_score = u.cosine_similarity(valid_vec, pred_vec)  # 0-1 value, the higher the more similar
    #         # print(f"Similarity {i}-{j}: {similarity_score}")

    #         ## return only pred-valid-pair with highest similarity
    #         if similarity_score > highest_similarity_score:
                
    #             highest_similarity_score = similarity_score
                
    #             entry["impact_similarity"] = highest_similarity_score

    # ## FNs
    # # # calc FN when valid_info exists but not corresponding pred_info
    # ## number of FNs is small due that wrong matching with any chunk-text is more likely due to its text size comapred sentence-level (valid set) 
    # elif entry[entity_pred] is np.nan:
    #     similarity_score = 0
    #     dict_pair = {
    #         "impact_valid": valid_impact, 
    #         "impact_pred": pred_impact, 
    #         "impact_similarity": similarity_score,
    #         "tp_tn_fp_fn": "fn",
    #         "citation": entry.citation_id,
    #         "chunk_text_pred": entry.chunk_text_pred,
    #         "sentence_text_valid": entry.sentence_text_valid,
    #         }
    #     df_smltry_selmax = pd.concat([df_smltry_selmax, pd.DataFrame([dict_pair])], ignore_index=True)

    # ## FPs
    # elif entry[entity_valid] is np.nan:
    #     similarity_score = 0
    #     dict_pair = {
    #         "impact_valid": valid_impact, 
    #         "impact_pred": pred_impact, 
    #         "impact_similarity": similarity_score,
    #         "tp_tn_fp_fn": "fp",
    #         "citation": entry.citation_id,
    #         "chunk_text_pred": entry.chunk_text_pred,
    #         "sentence_text_valid": entry.sentence_text_valid,
    #         }
    #     df_smltry_selmax = pd.concat([df_smltry_selmax, pd.DataFrame([dict_pair])], ignore_index=True)




In [ ]:
# ## get same impact entries
# list_entity_valid = ["ci1_type", "ci1_damage", "ci1_location"]
# list_entity_pred = ["infrastructure_type", "damage", "location"]



## iterate over predictions and search for each prediction reocrds for corresponding valid cases 

# for entity_valid, entity_pred in zip(list_entity_valid, list_entity_pred):

#     print(f" --------- Processing column pair: {entity_valid} - {entity_pred} ------------")
    
#     df_valid_pred_all = pd.DataFrame()
#     citations_list = []

#     ## for each validation record
#     for i in range(len(df_valid)):
        
#         highest_similarity_score = 0.00
        
#         ## needed to traceback info when entry is missing in pred. DS
#         # chunk_id_value_valid = df_valid.chunk_id[i]

#         # select nth validation record
#         df_valid_entry = df_valid.iloc[i]
#         citation_str = df_valid_entry.publication_id
#         citations_list.append(citation_str)
#         print(" ------- Searching for citation:", citation_str, " in predictions ------- ")


#         # get all corresponding prediction records
#         df_pred_entries = df_pred[df_pred["citation_id"].isin([citation_str])]
#         #  handle on NANs
#         df_pred_entries[entity_pred] = np.where(df_pred_entries[entity_pred].isna(), "nan", df_pred_entries[entity_pred])
#         # df_pred_entries[entity_pred] = df_pred_entries[entity_pred].astype(str)
#         # remove double whitespaces
#         # df_pred_doc[entity_pred] = df_pred_doc[entity_pred].replace("  ", " ")
#         # df_valid_entries[entity_valid] = df_valid_entries[entity_valid].replace("  ", " ")

#         # skip when validation entry ha no value
#         if df_valid_entry[entity_valid] is np.nan:
#             continue

#         # vector of validiation entry 
#         valid_impact = df_valid_entry[entity_valid]
#         valid_vec = nlp(valid_impact).vector


#         # Compute similarity between each predicted impact case and all potential validation impact cases (cross-product)
#         # print(f"Searching for highest similarity of`{pred_impact}` in validation set ... ")
#         for j in range(len(df_pred_entries[entity_pred])):

#             if df_pred_entries[entity_pred].iloc[j] == "nan":
#                 continue

#             pred_impact = df_pred_entries[entity_pred].iloc[j]

#             pred_vec = nlp(pred_impact).vector
#             similarity_score = u.cosine_similarity(valid_vec, pred_vec)  # 0-1 value, the higher the more similar
#             # print(f"Similarity {i}-{j}: {similarity_score}")

#             ## get only pair with highest similarity
#             if similarity_score > highest_similarity_score:
                
#                 highest_similarity_score = similarity_score
                
#                 dict_pair = {
#                     "impact_valid": valid_impact, 
#                     "impact_pred": pred_impact, 
#                     "similarity": highest_similarity_score,
#                     "citation": citation_str,
#                     "chunk_id_pred": df_pred.chunk_id[i]
#                 }
#             else:
#                 continue

#         df_valid_pred_all = pd.concat([df_valid_pred_all, pd.DataFrame([dict_pair])], ignore_index=True)


#     print(" ---------- Evaluation summary statistics: -----------")
#     print(df_valid_pred_all.similarity.describe())



#     SIMILARITY_FILENAME = f'{SIMILARITY_LLM_FILENAME}_{entity_pred}.parquet'
#     SIMILARITY_FILEPATH = Path(PATH_EVAL_RESULT / SIMILARITY_FILENAME)

#     print("Saving evaluation statistics and scores to ", SIMILARITY_FILEPATH.stem, "[.parquet, _stats.json]")
#     with open(SIMILARITY_FILEPATH, 'w') as f:
#         # results
#         df_valid_pred_all_pyarrow = pa.Table.from_pandas(df_valid_pred_all)
#         pq.write_table(df_valid_pred_all_pyarrow, SIMILARITY_FILEPATH)   
#         #   summary statistics
#         df_valid_pred_all_stats = df_valid_pred_all.describe()
#         f = SIMILARITY_FILEPATH.parent / f"{SIMILARITY_FILEPATH.stem}_stats.json"  
#         df_valid_pred_all_stats.to_json(f, indent=4)



#     cos_smlrty_thresh = 0.75
#     df_valid_pred_all['is_similar'] = df_valid_pred_all['similarity'] >= cos_smlrty_thresh
#     print(f"Number of similar impact cases (similarity >= {cos_smlrty_thresh}): {df_valid_pred_all['is_similar'].sum()} out of {len(df_valid_pred_all)}")

#     SIMILARITY_FILENAME = f'{SIMILARITY_LLM_FILENAME}_{entity_pred}_75.parquet'
#     SIMILARITY_FILEPATH = Path(PATH_EVAL_RESULT / SIMILARITY_FILENAME)

#     with open(SIMILARITY_FILEPATH, 'w') as f:
#         # results
#         df_valid_pred_all_pyarrow = pa.Table.from_pandas(df_valid_pred_all)
#         pq.write_table(df_valid_pred_all_pyarrow, SIMILARITY_FILEPATH)  
#         #   summary statistics
#         df_valid_pred_all_stats = df_valid_pred_all.describe()
#         f = SIMILARITY_FILEPATH.parent / f"{SIMILARITY_FILEPATH.stem}_stats.json"  
#         df_valid_pred_all_stats.to_json(f, indent=4)


In [ ]:

# #  Define folder for handling and writing outputs
# def write_to_file(data, out_folder, filename):
#     """Convert output to DataFrame and write to file"""
#     df = pd.DataFrame(list(data), columns=['tag', 'sts_score'])
#     #  Sort the DataFrame by similarity (explicitly)
#     df = df.sort_values(by='sts_score', ascending=False)
#     #  Assign integers to ranking
#     df['rank'] = df['sts_score'].rank(method='first', ascending=False).astype(int)
#     #  Only keep the first 20 resulting tags
#     df = df.head(50)
#     #  Save to file
#     df.to_csv(out_folder / f'{filename}_output.csv', index=False)

# #  Fill run metrics to dictionary
# def handle_metrics(metrics, model_name, length, end_time, start_time):
#     print(f'-> Took {end_time - start_time:.2f} seconds. Number of tags: {length}.')
#     metrics.append({
#         'modelname': model_name,
#         'runtime': round(end_time - start_time, 2),
#         'tagcount': length
#     })
#     return metrics

# class CPU_Unpickler(pickle.Unpickler):
#     """Fix for having issues with loading models on CPU"""
#     def find_class(self, module, name):
#         if module == 'torch.storage' and name == '_load_from_bytes':
#             return lambda b: torch.load(io.BytesIO(b), map_location='cpu')
#         else: return super().find_class(module, name)
